# Care Phenotypes as a Complementary Fairness Framework for Healthcare AI
## Severity-Stratified XGBoost Mortality Prediction — MIMIC-IV

This notebook implements the full analysis pipeline for the paper *"Care phenotypes as a complementary fairness framework for healthcare AI"*. Patients are stratified by adherence to two evidence-based ICU nursing interventions — oral mouthcare and patient repositioning (turning) — and mortality prediction models are trained and evaluated across care-frequency quartiles and racial/ethnic groups. The central question is whether care-phenotype stratification exposes model performance disparities that conventional demographic auditing fails to detect.

| Parameter | Specification |
|---|---|
| **Dataset** | MIMIC-IV (Beth Israel Deaconess Medical Center, 2008–2022) |
| **Cohort** | Invasive mechanical ventilation ≥ 24 hours |
| **Outcome** | In-hospital mortality (`hospital_expire_flag`) |
| **Predictive features** | SOFA subscores (LODS proxies) + laboratory values and vital signs (APS III proxies) — 37 features total |
| **Care frequency metric** | `average_item_interval` — mean hours between consecutive documented care events |
| **Q1** | Shortest interval = highest care frequency |
| **Q4** | Longest interval = lowest care frequency |
| **Severity stratification** | SOFA total score: Low (0–6), Medium (7–11), High (≥12)
| **Quartile assignment** | Within each severity stratum independently to decouple care frequency from illness acuity |
| **Model A** | Trained on all quartiles combined (80% per fold); evaluated on each quartile and each racial group |
| **Model B** | Trained on Q1 only (80% per fold); evaluated on all quartiles and each racial group |
| **Cross-validation** | 5-fold stratified CV; folds stratified jointly on quartile and mortality outcome |
| **Quartile CI method** | Mean ± 1.96×SE across 5 CV folds (per-fold AUROC averaged) |
| **Race CI method** | Bootstrap 1,000 resamples on pooled predictions (pool-then-AUROC) |

---

**Hypothesis:** A model trained exclusively on patients receiving the highest frequency of care (Q1) will show declining discriminative performance when evaluated on progressively lower-care quartiles (Q2–Q4), after controlling for illness severity. This degradation would indicate that care frequency shapes the physiology-to-mortality relationship the model learns, carrying mortality risk not captured by clinical severity features alone. The same model is expected to show no analogous degradation when stratified by race and ethnicity.

**Design rationale:** Without severity stratification, critically ill patients naturally receive more intensive nursing care, confounding care frequency with illness acuity. Assigning quartiles within severity strata ensures Q1 and Q4 patients share comparable illness burden, so observed performance differences reflect care quality rather than disease severity.


## 1. Package Installation and Imports

In [ ]:
import subprocess, sys
for pkg in ['xgboost','scikit-learn','pandas','numpy','openpyxl','matplotlib','seaborn','shap']:
    try:
        __import__(pkg.replace('-','_'))
    except ImportError:
        print(f'Installing {pkg}...')
        subprocess.check_call([sys.executable,'-m','pip','install',pkg,'-q'])

import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import shap
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, roc_curve, confusion_matrix,
    precision_score, recall_score, f1_score
)

from sklearn.calibration import calibration_curve
import xgboost as xgb
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

plt.rcParams.update({
    'font.family':'DejaVu Sans','font.size':11,
    'axes.spines.top':False,'axes.spines.right':False,
    'axes.grid':True,'grid.alpha':0.3,'figure.dpi':130
})
print(' All packages ready!')

## 2. File Path Configuration

Set the two directory paths below to point to your local mouthcare and turning data folders. All other configuration is handled automatically by the constants cell below.

In [ ]:
MOUTHCARE_FOLDER = r"\mouthcare"
TURNING_FOLDER   = r"\turningcare"
OUTPUT_FOLDER = r"figures"
OUTPUT_EXCEL  = r"mortality_results.xlsx"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

print(f'Mouthcare : {"" if os.path.exists(MOUTHCARE_FOLDER) else " NOT FOUND"}  {MOUTHCARE_FOLDER}')
print(f'Turning   : {"" if os.path.exists(TURNING_FOLDER)   else " NOT FOUND"}  {TURNING_FOLDER}')

## 3. Analysis Constants and Feature Definitions

In [ ]:
PATIENT_ID   = 'stay_id'
MORTALITY    = 'hospital_expire_flag'
FREQ_COL     = 'average_item_interval'
SOFA_COL     = 'SOFA'    # Used ONLY for severity stratification (Low/Med/High).
                             # SOFA total is NEVER included in FEATURE_COLS (training features).
                             # Only the 6 SOFA subscores (LODS_COLS) are training features.
RACE_COL     = 'race'
N_BOOTSTRAP  = 1000
RANDOM_STATE = 42
QUARTILES    = ['Q1','Q2','Q3','Q4']

SEVERITY_LABELS = ['Low','Medium','High']
SEVERITY_CUTS   = [-1, 6, 11, 999]

# Race mapping: consolidate granular MIMIC categories into 6 groups.
# Unknown/Unable to Obtain/Declined are excluded — not meaningful race categories.
RACE_MAP = {
    'WHITE':                                     'White',
    'WHITE - OTHER EUROPEAN':                    'White',
    'WHITE - RUSSIAN':                           'White',
    'WHITE - BRAZILIAN':                         'White',
    'WHITE - EASTERN EUROPEAN':                  'White',
    'PORTUGUESE':                                'White',
    'BLACK/AFRICAN AMERICAN':                    'Black',
    'BLACK/AFRICAN':                             'Black',
    'BLACK/CAPE VERDEAN':                        'Black',
    'BLACK/CARIBBEAN ISLAND':                    'Black',
    'HISPANIC OR LATINO':                        'Hispanic/Latino',
    'HISPANIC/LATINO - PUERTO RICAN':            'Hispanic/Latino',
    'HISPANIC/LATINO - DOMINICAN':               'Hispanic/Latino',
    'HISPANIC/LATINO - GUATEMALAN':              'Hispanic/Latino',
    'HISPANIC/LATINO - SALVADORAN':              'Hispanic/Latino',
    'HISPANIC/LATINO - HONDURAN':                'Hispanic/Latino',
    'HISPANIC/LATINO - CUBAN':                   'Hispanic/Latino',
    'HISPANIC/LATINO - COLUMBIAN':               'Hispanic/Latino',
    'HISPANIC/LATINO - MEXICAN':                 'Hispanic/Latino',
    'HISPANIC/LATINO - CENTRAL AMERICAN':        'Hispanic/Latino',
    'SOUTH AMERICAN':                            'Hispanic/Latino',
    'ASIAN':                                     'Asian',
    'ASIAN - CHINESE':                           'Asian',
    'ASIAN - SOUTH EAST ASIAN':                  'Asian',
    'ASIAN - ASIAN INDIAN':                      'Asian',
    'ASIAN - KOREAN':                            'Asian',
    'AMERICAN INDIAN/ALASKA NATIVE':             'Other',
    'NATIVE HAWAIIAN OR OTHER PACIFIC ISLANDER': 'Other',
    'MULTIPLE RACE/ETHNICITY':                   'Other',
    'OTHER':                                     'Other',
}
# Exclude from race analysis (uninformative)
RACE_EXCLUDE = {'UNKNOWN','UNABLE TO OBTAIN','PATIENT DECLINED TO ANSWER'}
# All five race groups are included in AUROC reporting and DeLong pairwise tests.
# 'Other' is a heterogeneous catch-all; its wide CI should be interpreted with caution
# given smaller n. The primary permutation JT test excludes 'Other' (as per paper's
# primary analysis); an 'incl Other' sensitivity check is also run (see cell 31).
RACE_LABELS  = ['White','Black','Hispanic/Latino','Asian','Other']

LODS_COLS = ['respiration','cardiovascular','liver','coagulation','renal','cns']
APSIII_COLS = [
    'heart_rate_min','heart_rate_max','heart_rate_mean',
    'mbp_min','mbp_max','mbp_mean',
    'temperature_min','temperature_max','temperature_mean',
    'resp_rate_min','resp_rate_max','resp_rate_mean',
    'hematocrit_min','hematocrit_max',
    'wbc_min','wbc_max',
    'creatinine_min','creatinine_max',
    'bun_min','bun_max',
    'sodium_min','sodium_max',
    'albumin_min','albumin_max',
    'bilirubin_total_min','bilirubin_total_max',
    'glucose_min','glucose_max',
    'bicarbonate_min','bicarbonate_max',
    'gcs_min',
]
FEATURE_COLS = LODS_COLS + APSIII_COLS

SEV_COLOURS  = {'Low':'#2E86C1','Medium':'#E67E22','High':'#C0392B'}
Q_COLOURS    = {'Q1':'#1A5276','Q2':'#2471A3','Q3':'#E67E22','Q4':'#C0392B'}
RACE_COLOURS = {'White':'#2874A6','Black':'#C0392B','Hispanic/Latino':'#E67E22',
                'Asian':'#1E8449','Other':'#7D3C98'}
MODEL_STYLES = {
    'A – All Quartiles': {'color':'#2874A6','marker':'o','ls':'-',
                          'label':'Model A — tested by quartile'},
    'B – Q1 Only':       {'color':'#C0392B','marker':'s','ls':'--',
                          'label':'Model B — Q1-trained, tested by quartile'},
    'A – By Race':       {'color':'#1E8449','marker':'^','ls':':',
                          'label':'Model A — tested by race group'},
    'B – Q1 By Race':    {'color':'#7D3C98','marker':'D','ls':'-.',
                          'label':'Model B — Q1-trained, tested by race group'},
}
print('Constants set!')


## 4. Data Loading and Preprocessing

> **Pre-processing note (paper section 2.1):** The `.pkl` files loaded here have been pre-filtered upstream to include only mechanically ventilated ICU patients (invasive MV ≥ 24 hours) with valid weight values, drawn from MIMIC-IV (Beth Israel Deaconess Medical Center, 2008–2022). All inclusion/exclusion criteria described in the paper are applied prior to this notebook. Race and ethnicity data are mapped to five consolidated groups here; patients with unknown or declined race are excluded from race-stratified analyses only (they remain in model training and quartile evaluation).

In [ ]:
def load_folder(folder):
# PRE-PROCESSING NOTE (paper section 2.1 — applied before this code):
# The .pkl files are pre-filtered upstream to include ONLY:
#   - Mechanically ventilated ICU patients (invasive MV duration >= 24 hours)
#   - Patients with valid, plausible weight values (implausible excluded)
# Source: MIMIC-IV (Beth Israel Deaconess Medical Center, 2008–2022).
# Care intensity = mean interval between consecutive documented care events (hours).
    file_map = {
        'cohort':        os.path.join(folder,'cohort.pkl'),
        'sofa':          os.path.join(folder,'sofa.pkl'),
        'vitals':        os.path.join(folder,'vitals.pkl'),
        'lab':           os.path.join(folder,'lab.pkl'),
        'gcs':           os.path.join(folder,'gcs.pkl'),
        'proxy':         os.path.join(folder,'proxy.pkl'),
        'comorbidities': os.path.join(folder,'comorbidities.pkl'),
    }
    dfs = {}
    for name, path in file_map.items():
        if os.path.exists(path):
            dfs[name] = pd.read_pickle(path)
            print(f'   {name:15s} {dfs[name].shape}')
        else:
            print(f'    {name}.pkl not found')

    proxy_agg = dfs['proxy'].groupby(PATIENT_ID).agg(
        average_item_interval=('average_item_interval','mean'),
        item_volume          =('item_volume','sum'),
        n_caregivers         =('n_caregivers','mean'),
        icu_days             =('day','count'),
    ).reset_index()

    # Include race column from cohort
    cohort_cols = [PATIENT_ID, MORTALITY, 'admission_age', 'los_icu', RACE_COL]
    cohort_cols = [c for c in cohort_cols if c in dfs['cohort'].columns]
    merged = dfs['cohort'][cohort_cols].copy()

    for name in ['sofa','vitals','lab','gcs','comorbidities']:
        if name in dfs:
            df = dfs[name].drop(columns=['subject_id','hadm_id'], errors='ignore')
            merged = merged.merge(df, on=PATIENT_ID, how='left')
    merged = merged.merge(proxy_agg, on=PATIENT_ID, how='left')

    # Map race to consolidated groups
    if RACE_COL in merged.columns:
        merged['race_group'] = merged[RACE_COL].map(RACE_MAP)
        # Patients with unknown/declined race: set to NaN (excluded from race analysis only)
        exclude_mask = merged[RACE_COL].isin(RACE_EXCLUDE)
        merged.loc[exclude_mask, 'race_group'] = np.nan
        race_dist = merged['race_group'].value_counts()
        print(f'  Race group distribution (after mapping):')
        for r,n in race_dist.items():
            print(f'    {r:20s}: {n:,}')
        print(f'    Excluded (unknown/declined): {exclude_mask.sum():,}')

    print(f'  → {len(merged):,} patients | mortality: {merged[MORTALITY].mean()*100:.1f}%')
    return merged

print('Loading MOUTHCARE...')
mc_df = load_folder(MOUTHCARE_FOLDER)
print('\nLoading TURNING...')
turn_df = load_folder(TURNING_FOLDER)
print('\nDone!')


## 5. Severity Stratification and Quartile Assignment

Patients are stratified by SOFA score into Low (0–6), Medium (7–11), and High (≥12) strata. Within each stratum, patients are ranked by mean care interval and divided into four equal-sized quartiles (Q1–Q4) using rank-based assignment to handle ties. This within-severity design ensures that Q1 and Q4 patients share comparable illness burden, so downstream model performance differences can be attributed to care frequency rather than acuity.
> **Key design principle (paper section 2.3):** Quartile labels are assigned on the **full stratum before any cross-validation splitting**, so every fold uses identical Q1–Q4 definitions. This ensures performance comparisons across folds are consistent and that the Q1-trained Model B is always evaluated on the same population definition as Model A.

In [ ]:
def assign_groups(df, care_label):
    """
    1. Assign severity group from SOFA score (Low/Medium/High)
    2. Within each severity group, assign Q1-Q4 by average_item_interval
       Q1 = smallest interval = most frequent care
       Q4 = largest interval  = least frequent care
    3. Keep only relevant columns and drop missing
    """
    avail = [c for c in FEATURE_COLS if c in df.columns]
    keep  = [PATIENT_ID, MORTALITY, FREQ_COL, SOFA_COL, 'race_group'] + avail
    out   = df[[c for c in keep if c in df.columns]].copy()

    # Drop rows missing mortality or frequency
    out = out.dropna(subset=[MORTALITY, FREQ_COL]).copy()
    out[MORTALITY] = out[MORTALITY].astype(int)
    for col in avail + [FREQ_COL]:
        out[col] = pd.to_numeric(out[col], errors='coerce')
    out[SOFA_COL] = pd.to_numeric(out[SOFA_COL], errors='coerce')

    #  Step 1: Severity group from SOFA 
    out['severity'] = pd.cut(
        out[SOFA_COL],
        bins=SEVERITY_CUTS,
        labels=SEVERITY_LABELS,
        right=True
    ).astype(str)
    out.loc[out[SOFA_COL].isna(), 'severity'] = 'Unknown'

    #  Step 2: Quartiles WITHIN each severity group (rank-based for equal sizes)
    # Q1 = smallest average_item_interval = most frequent care
    # rank(method='first') breaks ties by row order so pd.qcut always gets
    # perfectly equal inputs → exactly equal quartile sizes even with many ties
    def _rank_quartile(x):
        ranks = x.rank(method='first')
        return pd.qcut(ranks, q=4, labels=QUARTILES)
    out['quartile'] = out.groupby('severity', group_keys=False)[FREQ_COL].transform(_rank_quartile)

    # Drop patients whose quartile couldn't be assigned (rare edge case)
    n_before = len(out)
    out = out.dropna(subset=['quartile']).copy()
    out = out[out['severity'] != 'Unknown'].copy()
    n_dropped = n_before - len(out)
    if n_dropped > 0:
        print(f'    Dropped {n_dropped} patients (missing SOFA or unassignable quartile)')
    # Re-cast MORTALITY to int after dropna (dropna can silently promote to float64)
    out[MORTALITY] = out[MORTALITY].astype(int)
    # Re-cast quartile to plain string so downstream groupby never sees Categorical NaN
    out['quartile'] = out['quartile'].astype(str)

    #  Step 3: Verification printout 
    print(f'\n{""*60}')
    print(f'  {care_label} — group verification')
    print(f'  Total patients: {len(out):,} | Mortality: {out[MORTALITY].mean()*100:.1f}%')
    print(f'\n  Mean SOFA by severity (should differ clearly):')
    print(' ', out.groupby('severity')[SOFA_COL].mean().round(2).reindex(SEVERITY_LABELS).to_dict())
    print(f'\n  Mean SOFA by quartile within severity (should be SIMILAR — proves balance):')
    sofa_check = out.groupby(['severity','quartile'], observed=True)[SOFA_COL].mean().round(2).unstack()
    print(sofa_check.reindex(SEVERITY_LABELS).to_string())
    print(f'\n  Mean interval by quartile within severity (Q1 should be smallest):')
    interval_check = out.groupby(['severity','quartile'], observed=True)[FREQ_COL].mean().round(3).unstack()
    print(interval_check.reindex(SEVERITY_LABELS).to_string())
    print(f'\n  Mortality by quartile within severity:')
    mort_check = out.groupby(['severity','quartile'], observed=True)[MORTALITY].mean().round(3).unstack()
    print(mort_check.reindex(SEVERITY_LABELS).to_string())
    print(f'  (Hypothesis predicts mortality should increase Q1→Q4)')  # Descriptive only — not a pre-specified hypothesis; primary hypothesis is AUROC degradation in Config 3

    return out, avail

print('Assigning groups for MOUTHCARE...')
mc_data, mc_feats = assign_groups(mc_df, 'Mouthcare')
print('\nAssigning groups for TURNING...')
turn_data, turn_feats = assign_groups(turn_df, 'Turning')
print('\n Group assignment complete!')

## 6. Model Training Utilities, Evaluation Metrics, and Calibration Functions

In [ ]:
from sklearn.metrics import brier_score_loss
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression


# ── Calibration helpers ───────────────────────────────────────────────────────
# scale_pos_weight improves discrimination (AUROC) under class imbalance but
# inflates raw XGBoost predicted probabilities above the true event rate.
# These helpers apply post-hoc calibration to restore accurate probabilities
# without affecting AUROC (calibration is a monotone transform → rank-invariant).

def fit_platt_scaler(y_true, raw_scores):
    """Fit Platt scaler (logistic regression) on raw XGBoost scores.
    Returns fitted LogisticRegression, or None if data insufficient."""
    y_true = np.array(y_true, dtype=float)
    raw    = np.array(raw_scores, dtype=float).reshape(-1, 1)
    if len(np.unique(y_true)) < 2 or len(y_true) < 10:
        return None
    lr = LogisticRegression(C=1.0, solver='lbfgs', max_iter=1000)
    lr.fit(raw, y_true)
    return lr

def fit_isotonic_scaler(y_true, raw_scores):
    """Fit isotonic regression calibrator on raw XGBoost scores.
    Used as sensitivity check alongside Platt scaling."""
    y_true = np.array(y_true, dtype=float)
    raw    = np.array(raw_scores, dtype=float)
    if len(np.unique(y_true)) < 2 or len(y_true) < 10:
        return None
    ir = IsotonicRegression(out_of_bounds='clip')
    ir.fit(raw, y_true)
    return ir

def apply_calibrator(calibrator, raw_scores):
    """Apply a fitted Platt or Isotonic calibrator to raw scores.
    Falls back to raw scores if calibrator is None."""
    if calibrator is None:
        return np.array(raw_scores, dtype=float)
    raw = np.array(raw_scores, dtype=float)
    if isinstance(calibrator, LogisticRegression):
        return calibrator.predict_proba(raw.reshape(-1, 1))[:, 1]
    else:  # IsotonicRegression
        return calibrator.predict(raw)


# ── XGBoost trainer ───────────────────────────────────────────────────────────

def train_xgb(X_train, y_train, X_val=None, y_val=None):
    """
    XGBoost with class imbalance correction and regularisation to prevent overfitting.

    Hyperparameter choices (justified):
      max_depth=3          : shallower trees generalise better (was 4 — caused train 0.999/test 0.65)
      min_child_weight=10  : require >=10 patients per leaf; prevents splits on tiny subgroups
      gamma=1.0            : minimum loss reduction to make a split; prunes unproductive splits
      reg_alpha=0.5        : L1 regularisation; shrinks small feature weights toward zero
      reg_lambda=2.0       : L2 regularisation; smooths weights
      subsample=0.8        : row subsampling per tree (unchanged)
      colsample_bytree=0.8 : feature subsampling per tree (unchanged)
      learning_rate=0.05   : unchanged
      n_estimators=500     : upper bound; early stopping will find true optimum

    NOTE on scale_pos_weight:
      scale_pos_weight=neg/pos corrects class imbalance for DISCRIMINATION (AUROC)
      but deliberately inflates raw predicted probabilities above the true event rate.
      Post-hoc Platt/isotonic calibration (applied in the CV loop) restores accurate
      probability estimates without changing AUROC.

    Early stopping (20 rounds, metric: log-loss): uses 15% internal validation split
    drawn from training data to find optimal n_estimators (the iteration minimising
    validation log-loss, patience=20 rounds), then refits on full training data for that
    many rounds (Appendix A). Prevents overfitting while using all available training data.

    Without regularisation: train AUROC ~0.999, test AUROC ~0.65 (severe overfit).
    With regularisation:    train AUROC ~0.88,  test AUROC ~0.85 (good generalisation).
    """
    neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
    spw = neg / pos if pos > 0 else 1

    base_params = dict(
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=10,
        gamma=1.0,
        reg_alpha=0.5,
        reg_lambda=2.0,
        scale_pos_weight=spw,
        eval_metric='logloss',
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

    if X_val is not None and y_val is not None:
        model = xgb.XGBClassifier(n_estimators=500, early_stopping_rounds=20, **base_params)
        model.fit(X_train, y_train,
                  eval_set=[(X_val, y_val)],
                  verbose=False)
    else:
        from sklearn.model_selection import train_test_split as _tts
        try:
            Xtr, Xv, ytr, yv = _tts(X_train, y_train, test_size=0.15,
                                      stratify=y_train, random_state=RANDOM_STATE)
        except ValueError:
            Xtr, Xv, ytr, yv = _tts(X_train, y_train, test_size=0.15,
                                      random_state=RANDOM_STATE)
        probe = xgb.XGBClassifier(n_estimators=500, early_stopping_rounds=20, **base_params)
        probe.fit(Xtr, ytr, eval_set=[(Xv, yv)], verbose=False)
        best_n = probe.best_iteration + 1

        model = xgb.XGBClassifier(n_estimators=best_n, **base_params)
        model.fit(X_train, y_train)

    return model
print(' Helper functions ready!')


## 7. Main Training and Evaluation Pipeline — Five-Fold Cross-Validation

For each care phenotype (mouthcare, turning) and each severity stratum (Low, Medium, High), two models are trained and evaluated under five-fold stratified cross-validation:

- **Model A**: Trained on 80% of all quartiles combined. Represents a standard all-population clinical predictor. Evaluated on each quartile and each racial group from the held-out 20%.
- **Model B**: Trained on 80% of Q1 patients only. Learns mortality patterns under optimal care conditions. Evaluated on all quartiles and racial groups from the held-out 20%.

Stratification of folds is performed jointly on quartile and mortality outcome to preserve event rates. Race and care frequency are never used as training features in any model. Per-fold Platt scaling is applied to correct for probability inflation from `scale_pos_weight`, with isotonic regression applied in parallel as a sensitivity check (Appendix C). Because both are monotone transformations, AUROC and all rank-based statistics are unaffected.

**CI methods differ by axis:** Quartile AUROC is reported as mean ± 1.96×SE across 5 CV folds. Race AUROC uses pool-then-AUROC (all 5 folds pooled before computing one AUROC) with bootstrap 1,000 resamples CI — because per-fold race-group samples are too small for stable per-fold estimates.

*Runtime: approximately 10–20 minutes.*

In [ ]:

from sklearn.model_selection import StratifiedKFold

all_results           = []
calibration_store     = []
shap_store            = {}
roc_store             = []
true_fold_aurocs_store = {}  # Bootstrap AUROC distributions for directional JT.
                              # Key: (care, sev, model, quartile) → array of 1,000 bootstrap AUROCs.
                              # Populated in run_severity_group after CV loop completes.
                              # Used by jonckheere_trend_test in cell 31 (Configs 2 and 3).

# ── Constants ─────────────────────────────────────────────────────────────────
RACE_TEST_MIN_N         = 30    # minimum pooled patients for a race group to report AUROC
# All 5 groups (incl Other) are shown. Wide CIs on smaller groups (Other, Asian)
# are visible in the figure — intentionally shown so readers can judge reliability.
RACE_TEST_MIN_EVENT_PCT = 0.10  # minimum event rate in pooled race group
N_FOLDS                 = 5     # 5-fold CV => 80% train / 20% test per fold


# ── CV aggregation: quartiles ─────────────────────────────────────────────────
def _aggregate_cv_predictions(fold_preds):
    """
    Aggregate per-fold (y_true, y_prob) pairs for QUARTILE evaluation.

    Each pair comes from a model that never trained on those patients.

    Method:
      - AUROC and Brier computed per fold independently.
      - Point estimate = mean across folds.
      - 95% CI = mean +/- 1.96 * SE (std/sqrt(k)).
        Label in paper as "mean +/- 1.96*SE, 5-fold CV", not "bootstrap CI".
      - calibration_store / roc_store receive pooled predictions.

    Per-fold AUROC averaging is appropriate for quartiles because each fold's
    test set contains enough patients per quartile (~50+) for a stable estimate.
    """
    if not fold_preds:
        return None, None, None

    fold_aurocs, fold_briers = [], []
    fold_sens, fold_spec, fold_ppv, fold_npv, fold_f1 = [], [], [], [], []

    for yt, yp in fold_preds:
        yt = np.array(yt, dtype=float)
        yp = np.array(yp, dtype=float)
        if len(np.unique(yt)) < 2 or len(yt) < 5:
            continue
        try:
            fold_aurocs.append(roc_auc_score(yt, yp))
            fold_briers.append(brier_score_loss(yt, yp))
            fpr_f, tpr_f, thr_f = roc_curve(yt, yp)
            thr = thr_f[np.argmax(tpr_f - fpr_f)]
            yp_bin = (yp >= thr).astype(int)
            try:
                tn, fp, fn, tp = confusion_matrix(yt, yp_bin).ravel()
                fold_sens.append(tp / (tp + fn) if (tp + fn) > 0 else np.nan)
                fold_spec.append(tn / (tn + fp) if (tn + fp) > 0 else np.nan)
                fold_ppv.append( tp / (tp + fp) if (tp + fp) > 0 else np.nan)
                fold_npv.append( tn / (tn + fn) if (tn + fn) > 0 else np.nan)
                fold_f1.append(  2*tp/(2*tp+fp+fn) if (2*tp+fp+fn) > 0 else np.nan)
            except Exception:
                pass
        except Exception:
            continue

    if not fold_aurocs:
        return None, None, None

    def _ci(vals):
        arr = np.array([v for v in vals if not np.isnan(v)])
        if len(arr) == 0:
            return np.nan, np.nan, np.nan
        m  = arr.mean()
        se = arr.std(ddof=1) / np.sqrt(len(arr)) if len(arr) > 1 else 0.0
        return m, max(0.0, m - 1.96*se), min(1.0, m + 1.96*se)

    am, alo, ahi = _ci(fold_aurocs)
    bm, blo, bhi = _ci(fold_briers)
    sm,  _, _ = _ci(fold_sens);  spm, _, _ = _ci(fold_spec)
    pm,  _, _ = _ci(fold_ppv);   nm,  _, _ = _ci(fold_npv)
    fm,  _, _ = _ci(fold_f1)

    yt_all = np.concatenate([f[0] for f in fold_preds])
    yp_all = np.concatenate([f[1] for f in fold_preds])
    k = len(fold_aurocs)

    row = {
        "N":           len(yt_all),
        "Mortality%":  round(float(yt_all.mean()) * 100, 1),
        "AUROC":       round(am,  3) if not np.isnan(am)  else None,
        "AUROC_lo":    round(alo, 3) if not np.isnan(alo) else None,
        "AUROC_hi":    round(ahi, 3) if not np.isnan(ahi) else None,
        "AUROC_CI":    f"{am:.3f} ({alo:.3f}\u2013{ahi:.3f})" if not np.isnan(alo) else None,
        "Brier":       round(bm,  3) if not np.isnan(bm)  else None,
        "Brier_lo":    round(blo, 3) if not np.isnan(blo) else None,
        "Brier_hi":    round(bhi, 3) if not np.isnan(bhi) else None,
        "Brier_CI":    f"{bm:.3f} ({blo:.3f}\u2013{bhi:.3f})" if not np.isnan(blo) else None,
        "Sensitivity": round(sm,  3) if not np.isnan(sm)  else None,
        "Specificity": round(spm, 3) if not np.isnan(spm) else None,
        "PPV":         round(pm,  3) if not np.isnan(pm)  else None,
        "NPV":         round(nm,  3) if not np.isnan(nm)  else None,
        "F1":          round(fm,  3) if not np.isnan(fm)  else None,
        "CV_Folds":    k,
        "AUROC_std":   round(float(np.std(fold_aurocs, ddof=1)), 4) if k > 1 else None,
    }
    return row, yp_all, yt_all


# ── CV aggregation: race (pool-then-AUROC) ────────────────────────────────────
def _aggregate_race_predictions(fold_preds, care_label, sev_label,
                                race_model_name, race, cohort_n,
                                n_boot=1000):
    """
    Aggregate per-fold (y_true, y_prob) pairs for RACE evaluation.

    Design: POOL all fold predictions first, THEN compute one AUROC.

    Why different from quartile aggregation?
      Each fold's test set contains only ~20% of a race group.
      For White that is ~60 patients per fold -- acceptable but noisy.
      For Hispanic/Latino that might be 8-15 patients per fold -- AUROC
      from 8 patients is unreliable and averages of unreliable numbers
      are still unreliable.

      By pooling all 5 folds first we recover the full race-group cohort
      (e.g. White=300, Hispanic=60) and compute one stable AUROC on it.
      Every patient in the pool was predicted by a model that never trained
      on them -- the unseen-data guarantee is fully preserved.

    CI: bootstrap (1000 resamples) on the pooled predictions.
        This is appropriate because we have one estimate (not k fold estimates)
        and bootstrap directly quantifies uncertainty in that single AUROC.

    Returns: row_dict or None if group is too small / single-class.
    """
    if not fold_preds:
        return None

    yt_all = np.concatenate([f[0] for f in fold_preds])
    yp_all = np.concatenate([f[1] for f in fold_preds])

    n = len(yt_all)
    if n < RACE_TEST_MIN_N:
        print(f"      {race_model_name} \u2014 {race}: excluded "
              f"(pooled n={n} < minimum {RACE_TEST_MIN_N})")
        return None
    ev_rate = float(yt_all.mean())
    if ev_rate < RACE_TEST_MIN_EVENT_PCT:
        print(f"      {race_model_name} \u2014 {race}: excluded "
              f"(event rate {ev_rate*100:.1f}%)")
        return None
    if len(np.unique(yt_all)) < 2:
        print(f"      {race_model_name} \u2014 {race}: excluded (single class)")
        return None

    auc   = float(roc_auc_score(yt_all, yp_all))
    brier = float(brier_score_loss(yt_all, yp_all))

    rng = np.random.default_rng(RANDOM_STATE)
    idx = np.arange(n)
    boot_aucs, boot_briers = [], []
    for _ in range(n_boot):
        b = rng.choice(idx, size=n, replace=True)
        if len(np.unique(yt_all[b])) < 2:
            continue
        boot_aucs.append(roc_auc_score(yt_all[b], yp_all[b]))
        boot_briers.append(brier_score_loss(yt_all[b], yp_all[b]))

    ci_lo = float(np.percentile(boot_aucs,   2.5)) if boot_aucs else np.nan
    ci_hi = float(np.percentile(boot_aucs,  97.5)) if boot_aucs else np.nan
    blo   = float(np.percentile(boot_briers, 2.5)) if boot_briers else np.nan
    bhi   = float(np.percentile(boot_briers,97.5)) if boot_briers else np.nan

    row = {
        "Care Type":  care_label,
        "Severity":   sev_label,
        "Model":      race_model_name,
        "Test Set":   f"Race: {race}",
        "N":          n,
        "Mortality%": round(ev_rate * 100, 1),
        "AUROC":      round(auc,   3),
        "AUROC_lo":   round(ci_lo, 3) if not np.isnan(ci_lo) else None,
        "AUROC_hi":   round(ci_hi, 3) if not np.isnan(ci_hi) else None,
        "AUROC_CI":   f"{auc:.3f} ({ci_lo:.3f}\u2013{ci_hi:.3f})" if not np.isnan(ci_lo) else None,
        "Brier":      round(brier, 3),
        "Brier_lo":   round(blo,   3) if not np.isnan(blo) else None,
        "Brier_hi":   round(bhi,   3) if not np.isnan(bhi) else None,
        "Brier_CI":   f"{brier:.3f} ({blo:.3f}\u2013{bhi:.3f})" if not np.isnan(blo) else None,
    }
    return row, yt_all, yp_all


def run_severity_group(data, feats, care_label, sev_label):
    """
    QUARTILE EVALUATION: 5-fold CV (80% train / 20% test per fold).
    ---------------------------------------------------------------
    Quartile labels are assigned on the full stratum before CV so that
    every fold uses identical Q1-Q4 definitions.

    Model A: trained on ALL quartiles in the 80% training fold.
    Model B: trained on Q1-ONLY patients in the 80% training fold.

    Both models tested on the 20% held-out fold, sliced by quartile.
    Per-fold AUROC averaged across 5 folds (+/- 1.96*SE) per quartile.
    Each patient is tested exactly once, by a model that never saw them.

    RACE EVALUATION: CV pool-then-AUROC.
    -------------------------------------
    Inside each fold, both models predict on the held-out 20% patients
    sliced by race. These per-fold race predictions are accumulated.
    After all 5 folds, predictions for each race group are POOLED across
    folds (covering the full cohort, every patient unseen), then ONE AUROC
    is computed on the pool. Bootstrap CI on the pooled predictions.

    This design preserves the unseen-data guarantee (every race prediction
    comes from a model that never trained on that patient) while ensuring
    large enough pools for stable AUROC (e.g. White: ~300 pooled vs ~60
    per fold; Hispanic/Latino: ~60 pooled vs ~12 per fold).

    Race and care frequency are never training features.
    """
    results  = []
    tag      = f"{care_label} | {sev_label}"
    cohort_n = len(data)
    has_race = "race_group" in data.columns

    if data["quartile"].nunique() < 2:
        print(f"    [{tag}] Not enough quartile variation, skipping.")
        return results

    data = data.copy()
    data[MORTALITY] = data[MORTALITY].fillna(0).astype(int)
    for _fc in feats:
        if _fc in data.columns:
            data[_fc] = data[_fc].astype(float)

    # Stratification key: quartile x mortality (fall back to mortality if sparse)
    strat_key = data["quartile"].astype(str) + "_" + data[MORTALITY].astype(str)
    if strat_key.value_counts().min() < N_FOLDS:
        strat_key = data[MORTALITY].astype(str)

    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

    # Accumulators — primary (Platt-calibrated) and isotonic sensitivity check
    fold_preds_A     = {q: [] for q in QUARTILES}   # Platt-calibrated predictions
    fold_preds_B     = {q: [] for q in QUARTILES}
    fold_preds_A_iso = {q: [] for q in QUARTILES}   # isotonic sensitivity
    fold_preds_B_iso = {q: [] for q in QUARTILES}
    # Race: accumulate raw (y_true, y_prob) tuples from each fold's held-out slice
    # Pool across all folds after the loop, then compute one AUROC per race group.
    pool_A_race    = {r: ([], []) for r in RACE_LABELS}  # Platt-calibrated (y_true, y_prob) per race
    pool_B_race    = {r: ([], []) for r in RACE_LABELS}  # Platt-calibrated (y_true, y_prob) per race

    model_A_last = None
    model_B_last = None
    Xtr_A_last   = None
    Xtr_B_last   = None

    print(f"    5-fold CV (80/20) | cohort n={cohort_n:,} | "
          f"mortality={data[MORTALITY].mean()*100:.1f}%")

    data_arr = data.reset_index(drop=True)

    # ── CV fold loop ──────────────────────────────────────────────────────────
    for fold_i, (tr_idx, te_idx) in enumerate(skf.split(data_arr, strat_key.values)):

        train_fold = data_arr.iloc[tr_idx]
        test_fold  = data_arr.iloc[te_idx]
        train_med  = train_fold[feats].median()

        # ── Model A: train on all-Q training fold ─────────────────────────────
        Xtr_A = train_fold[feats].fillna(train_med)
        ytr_A = train_fold[MORTALITY]
        try:
            mA = train_xgb(Xtr_A, ytr_A)
        except Exception as e:
            print(f"    [{tag}] fold {fold_i} Model A failed: {e}")
            continue

        # ── Calibration for Model A (Platt primary, isotonic sensitivity) ─────
        # Fit on 15% held-out slice of the training fold.
        # scale_pos_weight inflates raw scores; calibration restores accurate
        # probabilities without changing AUROC (monotone transform, rank-invariant).
        from sklearn.model_selection import train_test_split as _cal_tts
        cal_A_platt, cal_A_iso = None, None
        try:
            _, Xcal_A, _, ycal_A = _cal_tts(
                Xtr_A, ytr_A, test_size=0.15,
                stratify=ytr_A, random_state=RANDOM_STATE)
            raw_cal_A   = mA.predict_proba(Xcal_A)[:, 1]
            cal_A_platt = fit_platt_scaler(ycal_A, raw_cal_A)
            cal_A_iso   = fit_isotonic_scaler(ycal_A, raw_cal_A)
        except Exception:
            pass  # falls back to raw scores via apply_calibrator(None, ...)

        # Test A by quartile — Platt primary, isotonic sensitivity
        for q in QUARTILES:
            te_q = test_fold[test_fold["quartile"] == q]
            if len(te_q) < 5 or te_q[MORTALITY].nunique() < 2:
                continue
            raw_yp = mA.predict_proba(te_q[feats].fillna(train_med))[:, 1]
            yp     = apply_calibrator(cal_A_platt, raw_yp)
            yp_iso = apply_calibrator(cal_A_iso,   raw_yp)
            fold_preds_A[q].append((te_q[MORTALITY].values, yp))
            fold_preds_A_iso[q].append((te_q[MORTALITY].values, yp_iso))

        # Collect Model A race predictions (Platt-calibrated)
        if has_race:
            for race in RACE_LABELS:
                te_r = test_fold[test_fold["race_group"] == race]
                if len(te_r) < 2 or te_r[MORTALITY].nunique() < 2:
                    continue
                raw_a = mA.predict_proba(te_r[feats].fillna(train_med))[:, 1]
                yp_a  = apply_calibrator(cal_A_platt, raw_a)
                pool_A_race[race][0].extend(te_r[MORTALITY].values.tolist())
                pool_A_race[race][1].extend(yp_a.tolist())

        model_A_last = mA
        Xtr_A_last   = Xtr_A

        # ── Model B: train on Q1-only portion of training fold ────────────────
        q1_tr = train_fold[train_fold["quartile"] == "Q1"]
        if len(q1_tr) < 20:
            print(f"    [{tag}] fold {fold_i} Q1 train too small ({len(q1_tr)}), skipping B.")
            continue

        q1_med = q1_tr[feats].median()   # Q1-only medians for all imputation
        Xtr_B  = q1_tr[feats].fillna(q1_med)
        ytr_B  = q1_tr[MORTALITY]
        try:
            mB = train_xgb(Xtr_B, ytr_B)
        except Exception as e:
            print(f"    [{tag}] fold {fold_i} Model B failed: {e}")
            continue

        # ── Calibration for Model B (Platt primary, isotonic sensitivity) ─────
        cal_B_platt, cal_B_iso = None, None
        try:
            _, Xcal_B, _, ycal_B = _cal_tts(
                Xtr_B, ytr_B, test_size=0.15,
                stratify=ytr_B, random_state=RANDOM_STATE)
            raw_cal_B   = mB.predict_proba(Xcal_B)[:, 1]
            cal_B_platt = fit_platt_scaler(ycal_B, raw_cal_B)
            cal_B_iso   = fit_isotonic_scaler(ycal_B, raw_cal_B)
        except Exception:
            pass

        # Test B by quartile — Platt primary, isotonic sensitivity
        for q in QUARTILES:
            te_q = test_fold[test_fold["quartile"] == q]
            if len(te_q) < 5 or te_q[MORTALITY].nunique() < 2:
                continue
            raw_yp = mB.predict_proba(te_q[feats].fillna(q1_med))[:, 1]
            yp     = apply_calibrator(cal_B_platt, raw_yp)
            yp_iso = apply_calibrator(cal_B_iso,   raw_yp)
            fold_preds_B[q].append((te_q[MORTALITY].values, yp))
            fold_preds_B_iso[q].append((te_q[MORTALITY].values, yp_iso))

        # Collect Model B race predictions (Platt-calibrated)
        if has_race:
            for race in RACE_LABELS:
                te_r = test_fold[test_fold["race_group"] == race]
                if len(te_r) < 2 or te_r[MORTALITY].nunique() < 2:
                    continue
                raw_b = mB.predict_proba(te_r[feats].fillna(q1_med))[:, 1]
                yp_b  = apply_calibrator(cal_B_platt, raw_b)
                pool_B_race[race][0].extend(te_r[MORTALITY].values.tolist())
                pool_B_race[race][1].extend(yp_b.tolist())

        model_B_last = mB
        Xtr_B_last   = Xtr_B

    # ── Aggregate: Model A by quartile (mean of per-fold AUROCs) ─────────────
    for q in QUARTILES:
        if not fold_preds_A[q]:
            continue
        row, yp_all, yt_all = _aggregate_cv_predictions(fold_preds_A[q])
        if row is None:
            continue
        row.update({"Care Type": care_label, "Severity": sev_label,
                    "Model": "A \u2013 All Quartiles",
                    "Test Set": f"CV {q} ({N_FOLDS}-fold)"})
        results.append(row)
        calibration_store.append(
            (care_label, sev_label, "A \u2013 All Quartiles", "quartile", q, yt_all, yp_all))
        fpr_v, tpr_v, _ = roc_curve(yt_all, yp_all)
        roc_store.append((care_label, sev_label, "A \u2013 All Quartiles",
                          "quartile", q, fpr_v, tpr_v, row["AUROC"]))
        print(f"    Model A | {q}: AUROC={row['AUROC']} ({row['AUROC_CI']}) n={row['N']}")

    # ── Aggregate: Model B by quartile (mean of per-fold AUROCs) ─────────────
    for q in QUARTILES:
        if not fold_preds_B[q]:
            continue
        row, yp_all, yt_all = _aggregate_cv_predictions(fold_preds_B[q])
        if row is None:
            continue
        row.update({"Care Type": care_label, "Severity": sev_label,
                    "Model": "B \u2013 Q1 Only",
                    "Test Set": f"CV {q} ({N_FOLDS}-fold)"})
        results.append(row)
        calibration_store.append(
            (care_label, sev_label, "B \u2013 Q1 Only", "quartile", q, yt_all, yp_all))
        fpr_v, tpr_v, _ = roc_curve(yt_all, yp_all)
        roc_store.append((care_label, sev_label, "B \u2013 Q1 Only",
                          "quartile", q, fpr_v, tpr_v, row["AUROC"]))
        print(f"    Model B | {q}: AUROC={row['AUROC']} ({row['AUROC_CI']}) n={row['N']}")

    # ── Bootstrap AUROC distributions for directional JT (Configs 2 and 3) ────
    # For jonckheere_trend_test we need multiple AUROC values per quartile group
    # to compare distributions across Q1→Q4.
    #
    # Bootstrap approach (more rigorous than 5 fold AUROCs):
    #   - Draws 1,000 bootstrap resamples from each quartile's pooled predictions
    #   - Same predictions used by DeLong -> internally consistent
    #   - Between-group independence holds (patient-disjoint quartiles)
    #   - Within-group correlation causes conservative bias in normal approx
    #     (p-values if anything too large, not too small -> safe direction)
    #   - n=1,000 per group vs n=5 -> vastly more power to detect the trend
    #
    # bootstrap_jt_store: (care, sev, model, quartile) -> array of 1000 bootstrap AUROCs
    _boot_n    = 1000
    _boot_seed = RANDOM_STATE
    _rng_jt    = np.random.default_rng(_boot_seed)

    for model_name, model_key in [('A – All Quartiles', 'A – All Quartiles'),
                                   ('B – Q1 Only',       'B – Q1 Only')]:
        for q in QUARTILES:
            # Pull pooled predictions for this (care, sev, model, quartile)
            entry = next(
                ((yt, yp) for (cl, sl, mn, gt, g, yt, yp) in calibration_store
                 if cl == care_label and sl == sev_label
                 and mn == model_name and gt == 'quartile' and g == q
                 and len(np.unique(yt)) > 1),
                None
            )
            if entry is None:
                continue
            yt_pool = np.array(entry[0], dtype=float)
            yp_pool = np.array(entry[1], dtype=float)
            n_pool  = len(yt_pool)
            idx_all = np.arange(n_pool)
            boot_aucs = []
            for _ in range(_boot_n):
                b = _rng_jt.choice(idx_all, size=n_pool, replace=True)
                if len(np.unique(yt_pool[b])) < 2:
                    continue
                boot_aucs.append(float(roc_auc_score(yt_pool[b], yp_pool[b])))
            if len(boot_aucs) >= 10:
                true_fold_aurocs_store[(care_label, sev_label, model_key, q)] = np.array(boot_aucs)

    # ── Isotonic sensitivity: store in calibration_store only ────────────────
    # Does not affect results_df or any AUROC/stat results.
    # Used only by calibration plot cells as a sensitivity comparison.
    for q in QUARTILES:
        if fold_preds_A_iso.get(q):
            r_iso, yp_iso_all, yt_iso_all = _aggregate_cv_predictions(fold_preds_A_iso[q])
            if r_iso is not None:
                calibration_store.append(
                    (care_label, sev_label, "A – All Q (Isotonic)", "quartile", q,
                     yt_iso_all, yp_iso_all))
        if fold_preds_B_iso.get(q):
            r_iso, yp_iso_all, yt_iso_all = _aggregate_cv_predictions(fold_preds_B_iso[q])
            if r_iso is not None:
                calibration_store.append(
                    (care_label, sev_label, "B – Q1 (Isotonic)", "quartile", q,
                     yt_iso_all, yp_iso_all))

    # ── SHAP: last-fold models ────────────────────────────────────────────────
    if model_A_last is not None:
        shap_store[(care_label, sev_label, "A \u2013 All Quartiles")] = (
            model_A_last, Xtr_A_last, feats)
    if model_B_last is not None:
        shap_store[(care_label, sev_label, "B \u2013 Q1 Only")] = (
            model_B_last, Xtr_B_last, feats)

    # ── Aggregate: race (pool-then-AUROC, bootstrap CI) ──────────────────────
    # pool_A_race[race] and pool_B_race[race] contain predictions from every
    # fold's held-out test set. Pooled together they cover the full cohort.
    # Every prediction was made by a model that never trained on that patient.
    # One AUROC is computed on the full pool; CI via 1000-iter bootstrap.
    if has_race:
        print(f"\n    Race evaluation (CV pool-then-AUROC, bootstrap CI):")
        for pool_race, race_model_name in [
            (pool_A_race, "A \u2013 By Race"),
            (pool_B_race, "B \u2013 Q1 By Race"),
        ]:
            n_included = 0
            for race in RACE_LABELS:
                yt_list, yp_list = pool_race[race]
                if not yt_list:
                    print(f"      {race_model_name} \u2014 {race}: no predictions collected")
                    continue
                yt_pool = np.array(yt_list, dtype=float)
                yp_pool = np.array(yp_list, dtype=float)

                # Pool-level eligibility checks
                if len(yt_pool) < RACE_TEST_MIN_N:
                    print(f"      {race_model_name} \u2014 {race}: excluded "
                          f"(pooled n={len(yt_pool)} < {RACE_TEST_MIN_N})")
                    continue
                ev_rate = float(yt_pool.mean())
                if ev_rate < RACE_TEST_MIN_EVENT_PCT:
                    print(f"      {race_model_name} \u2014 {race}: excluded "
                          f"(event rate {ev_rate*100:.1f}%)")
                    continue
                if len(np.unique(yt_pool)) < 2:
                    print(f"      {race_model_name} \u2014 {race}: excluded (single class)")
                    continue

                # One AUROC on the full pool
                auc   = float(roc_auc_score(yt_pool, yp_pool))
                brier = float(brier_score_loss(yt_pool, yp_pool))

                # Bootstrap CI on the pool
                rng  = np.random.default_rng(RANDOM_STATE)
                idx  = np.arange(len(yt_pool))
                boot_aucs, boot_briers = [], []
                for _ in range(1000):
                    b = rng.choice(idx, size=len(idx), replace=True)
                    if len(np.unique(yt_pool[b])) < 2:
                        continue
                    boot_aucs.append(roc_auc_score(yt_pool[b], yp_pool[b]))
                    boot_briers.append(brier_score_loss(yt_pool[b], yp_pool[b]))

                ci_lo = float(np.percentile(boot_aucs,   2.5)) if boot_aucs else np.nan
                ci_hi = float(np.percentile(boot_aucs,  97.5)) if boot_aucs else np.nan
                blo   = float(np.percentile(boot_briers, 2.5)) if boot_briers else np.nan
                bhi   = float(np.percentile(boot_briers,97.5)) if boot_briers else np.nan

                n_included += 1
                row = {
                    "Care Type":  care_label,
                    "Severity":   sev_label,
                    "Model":      race_model_name,
                    "Test Set":   f"Race: {race}",
                    "N":          len(yt_pool),
                    "Mortality%": round(ev_rate * 100, 1),
                    "AUROC":      round(auc,   3),
                    "AUROC_lo":   round(ci_lo, 3) if not np.isnan(ci_lo) else None,
                    "AUROC_hi":   round(ci_hi, 3) if not np.isnan(ci_hi) else None,
                    "AUROC_CI":   f"{auc:.3f} ({ci_lo:.3f}\u2013{ci_hi:.3f})" if not np.isnan(ci_lo) else None,
                    "Brier":      round(brier, 3),
                    "Brier_lo":   round(blo,   3) if not np.isnan(blo) else None,
                    "Brier_hi":   round(bhi,   3) if not np.isnan(bhi) else None,
                    "Brier_CI":   f"{brier:.3f} ({blo:.3f}\u2013{bhi:.3f})" if not np.isnan(blo) else None,
                }
                results.append(row)
                calibration_store.append(
                    (care_label, sev_label, race_model_name, "race", race, yt_pool, yp_pool))
                fpr_v, tpr_v, _ = roc_curve(yt_pool, yp_pool)
                roc_store.append((care_label, sev_label, race_model_name,
                                  "race", race, fpr_v, tpr_v, round(auc, 3)))
                print(f"      {race_model_name} | {race}: AUROC={auc:.3f} "
                      f"({ci_lo:.3f}\u2013{ci_hi:.3f}) n={len(yt_pool)}")
            print(f"      {race_model_name}: {n_included}/{len(RACE_LABELS)} race groups included.")

    return results


def run_pipeline(data, feats, care_label):
    sep = "="*60
    print(f"\n{sep}\n  {care_label}\n{sep}")
    res = []
    for sev in SEVERITY_LABELS:
        sub = data[data["severity"] == sev].copy()
        print(f"\n  {sev} severity (n={len(sub):,}, "
              f"mortality={sub[MORTALITY].mean()*100:.1f}%)")
        if len(sub) < 80:
            print("    Insufficient patients, skipping.")
            continue
        res += run_severity_group(sub, feats, care_label, sev)
    return res


all_results  = run_pipeline(mc_data,   mc_feats,   "Mouthcare")
all_results += run_pipeline(turn_data, turn_feats, "Turning")

COL_ORDER = ["Care Type","Severity","Model","Test Set","N","Mortality%",
             "AUROC","AUROC_lo","AUROC_hi","AUROC_CI",
             "Brier","Brier_lo","Brier_hi","Brier_CI",
             "Sensitivity","Specificity","PPV","NPV","F1"]
results_df = pd.DataFrame(all_results)
results_df = results_df[[c for c in COL_ORDER if c in results_df.columns]]
print(f"\nCV complete. Result rows: {len(results_df)}")
print(results_df[["Care Type","Severity","Model","Test Set",
                  "N","Mortality%","AUROC","AUROC_CI"]].to_string(index=False))


## 8. Descriptive Analysis — Mortality by Severity Stratum and Care Frequency Quartile

Mortality rates and cohort characteristics tabulated by severity stratum and quartile. A systematic increase in mortality from Q1 to Q4 within strata — after controlling for illness acuity — provides initial descriptive evidence that care frequency is associated with patient outcomes.

In [ ]:
def mortality_table(data, care_label):
    rows = []
    for sev in SEVERITY_LABELS:
        sub = data[data['severity'] == sev]
        for q in QUARTILES:
            qdf = sub[sub['quartile'] == q]
            if len(qdf) == 0:
                continue
            rows.append({
                'Care Type':          care_label,
                'Severity':           sev,
                'Quartile':           q,
                'N':                  len(qdf),
                'Deaths':             int(qdf[MORTALITY].fillna(0).astype(int).sum()),
                'Mortality%':         round(qdf[MORTALITY].mean()*100, 1),
                'Mean Interval (hrs)':round(qdf[FREQ_COL].mean(), 3),
                'Mean SOFA':          round(qdf[SOFA_COL].mean(), 2),
            })
    return pd.DataFrame(rows)

mc_mort   = mortality_table(mc_data,   'Mouthcare')
turn_mort = mortality_table(turn_data, 'Turning')
mort_tbl  = pd.concat([mc_mort, turn_mort], ignore_index=True)

print('=== MOUTHCARE — Mortality by Severity × Quartile ===')
print(mc_mort.to_string(index=False))
print('\n=== TURNING — Mortality by Severity × Quartile ===')
print(turn_mort.to_string(index=False))
print('\n(Q1=most frequent care, Q4=least frequent care)')
print('Key check: Mean SOFA should be similar across Q1-Q4 within each severity row')

## 9. AUROC Summary — Diagnostic Pivot Table

*Diagnostic cell only — produces a quick heatmap of AUROC values across severity × quartile. Not a paper figure. Run for sanity-checking results_df before proceeding to cell 41 (main figures).*

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  DIAGNOSTIC ONLY — not a paper figure                           ║
# ║  Produces fig1_auc_heatmap.png — AUROC pivot heatmap.           ║
# ║  Useful for quick visual QC of the results_df table.            ║
# ║  The paper's main AUROC figures are in cell 41 (enhanced 2×2).  ║
# ╚══════════════════════════════════════════════════════════════════╝
def auc_pivot(results_df, care_label, model_name):
    sub = results_df[
        (results_df['Care Type'] == care_label) &
        (results_df['Model'] == model_name)
    ].copy()
    sub['Q'] = sub['Test Set'].str.extract(r'(Q\d)')
    pivot = sub.pivot_table(index='Severity', columns='Q', values='AUROC', aggfunc='first')
    existing_q = [q for q in QUARTILES if q in pivot.columns]
    pivot = pivot[existing_q]
    pivot.index = pd.CategoricalIndex(pivot.index, categories=SEVERITY_LABELS, ordered=True)
    return pivot.sort_index()

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
titles = [
    ('Mouthcare','A – All Quartiles','Mouthcare — Model A\n(trained on all Q within severity)'),
    ('Mouthcare','B – Q1 Only',      'Mouthcare — Model B\n(trained on Q1 only within severity)'),
    ('Turning',  'A – All Quartiles','Turning — Model A\n(trained on all Q within severity)'),
    ('Turning',  'B – Q1 Only',      'Turning — Model B\n(trained on Q1 only within severity)'),
]
for ax, (care, model, title) in zip(axes.flat, titles):
    try:
        piv = auc_pivot(results_df, care, model)
        sns.heatmap(piv.astype(float), annot=True, fmt='.3f',
                    cmap='RdYlGn', vmin=0.50, vmax=0.85,
                    ax=ax, linewidths=0.5, linecolor='white',
                    cbar_kws={'label':'AUROC'})
        ax.set_title(title, fontweight='bold', fontsize=10)
        ax.set_xlabel('Mouthcare/Turning Quartile\n(Q1=most frequent, Q4=least frequent)', fontsize=9)
        ax.set_ylabel('Severity Group (SOFA)', fontsize=9)
    except Exception as e:
        ax.text(0.5,0.5,f'No data\n{e}',ha='center',va='center',transform=ax.transAxes)

plt.suptitle('AUROC Heatmap — Severity-Stratified Models\n'
             'Hypothesis (Model B): AUROC should DECREASE left→right within each severity row',
             fontsize=12, fontweight='bold')
plt.tight_layout()
# [unused figure removed] plt.savefig(...fig1_auc_heatmap.png...)
# [unused figure removed] plt.show()
# [unused figure removed] print('Saved: fig1_auc_heatmap.png')

## 10. Discriminative Performance — AUROC Lines (Superseded)

*Function definitions only — figure generation is disabled (superseded by cell 41). The enhanced 2×2 panel in cell 41 produces the paper figures with full statistical annotations.*

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  SUPERSEDED BY CELL 41 (enhanced 2×2 with stat annotations)     ║
# ║  Produces fig2a/fig2b_auroc_*.png — basic AUROC lines only.     ║
# ║  Cell 41 generates the same content with DeLong brackets,       ║
# ║  Spearman ρ, and JT annotations. Run cell 41 instead.           ║
# ║  Kept here because plot_auroc_lines() is not called elsewhere.  ║
# ╚══════════════════════════════════════════════════════════════════╝
def plot_auroc_lines(results_df, care_label, save_path):
    sevs = SEVERITY_LABELS
    fig, axes = plt.subplots(1, len(sevs), figsize=(5*len(sevs), 5), sharey=True)

    for ax, sev in zip(axes, sevs):
        sub = results_df[
            (results_df['Care Type'] == care_label) &
            (results_df['Severity'] == sev)
        ]
        for mname, style in MODEL_STYLES.items():
            mdf = sub[sub['Model'] == mname].copy()
            mdf['Q'] = mdf['Test Set'].str.extract(r'(Q\d)')
            mdf = mdf.dropna(subset=['Q','AUROC']).sort_values('Q')
            if len(mdf) == 0:
                continue
            x    = np.arange(len(mdf))
            auc  = mdf['AUROC'].values.astype(float)
            lo   = mdf['AUROC_lo'].values.astype(float)
            hi   = mdf['AUROC_hi'].values.astype(float)
            yerr = np.array([auc-lo, hi-auc])
            ax.errorbar(x, auc, yerr=yerr,
                        color=style['color'], marker=style['marker'],
                        linestyle=style['ls'], linewidth=2, markersize=8,
                        capsize=5, capthick=2, label=style['label'],
                        ecolor=style['color'], alpha=0.9)
            for xi, ai in enumerate(auc):
                ax.annotate(f'{ai:.3f}', (xi,ai),
                            textcoords='offset points', xytext=(0,10),
                            ha='center', fontsize=8, color=style['color'], fontweight='bold')

        ax.axhline(0.5, color='grey', ls=':', lw=1, alpha=0.6, label='Chance (0.5)')
        ax.axhline(0.7, color='#E67E22', ls=':', lw=1, alpha=0.5, label='Good (0.7)')
        ax.set_xticks(range(len(QUARTILES)))
        ax.set_xticklabels(['Q1\n(most\nfrequent)','Q2','Q3','Q4\n(least\nfrequent)'], fontsize=9)
        ax.set_xlabel('Test Quartile', fontsize=10)
        ax.set_ylim(0.35, 1.02)
        ax.set_title(f'SOFA {sev}\n(SOFA {"0–6" if sev=="Low" else "7–11" if sev=="Medium" else "≥12"})',
                     fontweight='bold', fontsize=11)
        ax.set_ylabel('AUROC (95% CI)', fontsize=10)
        ax.legend(fontsize=8, loc='lower left')

    fig.suptitle(f'{care_label} — AUROC by Severity Group and Quartile\n'
                 f'Model B hypothesis: green dashed line should slope DOWNWARD Q1→Q4',
                 fontsize=12, fontweight='bold', y=1.02)
    plt.tight_layout()
    # [figure removed] plt.savefig(save_path, bbox_inches='tight', dpi=150)
    # [figure removed] plt.show()
    print(f'  Saved: {save_path}')

# [unused figure removed] plot_auroc_lines(results_df, 'Mouthcare', ...)
# [unused figure removed] plot_auroc_lines(results_df, 'Turning', ...)
# Superseded by cell 41 enhanced 2x2 panels with statistical annotations.

<!-- superseded — see statistical framework in cells 29–39 -->

In [ ]:
# This cell is superseded by the shared stat helpers in cell 30.
# Left blank to preserve cell numbering.
print("Old stat cell — superseded. See stat framework cells below.")


## 10c. AUROC Comparison — Care vs Race Axis (Superseded)

*Function definition only — figure generation disabled (superseded by cell 41). The enhanced 2×2 panel in cell 41 shows this comparison with statistical annotations.*

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  SUPERSEDED BY CELL 26 (combo figures) and CELL 41              ║
# ║  Produces fig_comparison_*.png — Model B quartile (top) vs      ║
# ║  Model A race (bottom). Cell 26 combo1/combo5 cover this        ║
# ║  comparison with more configurations and better layout.         ║
# ║  Kept because plot_quartile_vs_race_comparison() has no         ║
# ║  downstream callers and is safe to skip.                        ║
# ╚══════════════════════════════════════════════════════════════════╝
def plot_quartile_vs_race_comparison(results_df, care_label, save_path):
    """
    Side-by-side AUROC comparison:
    Left panels  — Model B tested by quartile (Q1..Q4) per severity
    Right panels — Model A tested by race group per severity
    Allows visual comparison of generalisation gaps.
    """
    sevs = SEVERITY_LABELS
    fig, axes = plt.subplots(2, len(sevs), figsize=(5*len(sevs), 9))

    df = results_df[results_df['Care Type'] == care_label]

    for col_i, sev in enumerate(sevs):
        sub = df[df['Severity'] == sev]

        # ── Top row: Model B by quartile ──────────────────────────────────────
        ax_top = axes[0, col_i]
        mdf = sub[sub['Model'] == 'B – Q1 Only'].copy()
        mdf['q_label'] = mdf['Test Set'].str.extract(r'(Q\d)')
        mdf = mdf.dropna(subset=['q_label','AUROC']).sort_values('q_label')

        if len(mdf) > 0:
            x    = np.arange(len(mdf))
            auc  = mdf['AUROC'].values.astype(float)
            lo   = mdf['AUROC_lo'].values.astype(float)
            hi   = mdf['AUROC_hi'].values.astype(float)
            yerr = np.array([auc-lo, hi-auc])
            ax_top.errorbar(x, auc, yerr=yerr,
                            color='#C0392B', marker='s', linestyle='--',
                            linewidth=2, markersize=8, capsize=5, capthick=2,
                            ecolor='#C0392B', alpha=0.9,
                            label='Model B — Q1 trained')
            for xi, ai in enumerate(auc):
                ax_top.annotate(f'{ai:.3f}', (xi, ai),
                                textcoords='offset points', xytext=(0,8),
                                ha='center', fontsize=8, color='#C0392B', fontweight='bold')
            ax_top.set_xticks(x)
            ax_top.set_xticklabels(mdf['q_label'].values, fontsize=9)
            # Shade to show degradation
            if len(auc) >= 2:
                ax_top.fill_between(x, auc, auc[0], alpha=0.08, color='#C0392B')

        ax_top.axhline(0.5, color='grey', ls=':', lw=1, alpha=0.5)
        ax_top.set_ylim(0.4, 1.0)
        ax_top.set_title(f'{sev}\nModel B — by Quartile', fontweight='bold', fontsize=10)
        ax_top.set_xlabel('Test Quartile', fontsize=9)
        if col_i == 0:
            ax_top.set_ylabel('AUROC (95% CI)', fontsize=10)
        ax_top.legend(fontsize=8)

        # ── Bottom row: Model A by race ───────────────────────────────────────
        ax_bot = axes[1, col_i]
        rdf = sub[sub['Model'] == 'A – By Race'].copy()
        # Extract race label from 'Race: White' format
        rdf['race_label'] = rdf['Test Set'].str.replace('Race: ','', regex=False)
        rdf = rdf.dropna(subset=['race_label','AUROC'])
        # Order by RACE_LABELS
        rdf['race_order'] = rdf['race_label'].map({r:i for i,r in enumerate(RACE_LABELS)})
        rdf = rdf.sort_values('race_order')

        if len(rdf) > 0:
            x    = np.arange(len(rdf))
            auc  = rdf['AUROC'].values.astype(float)
            lo   = rdf['AUROC_lo'].values.astype(float)
            hi   = rdf['AUROC_hi'].values.astype(float)
            yerr = np.array([auc-lo, hi-auc])
            colors_r = [RACE_COLOURS.get(r,'grey') for r in rdf['race_label']]
            for xi, (ai, ci_col, loi, hii) in enumerate(zip(auc, colors_r, lo, hi)):
                ax_bot.errorbar(xi, ai, yerr=[[ai-loi],[hii-ai]],
                                color=ci_col, marker='o', linestyle='none',
                                markersize=9, capsize=5, capthick=2, ecolor=ci_col)
                ax_bot.annotate(f'{ai:.3f}', (xi, ai),
                                textcoords='offset points', xytext=(0,8),
                                ha='center', fontsize=8, color=ci_col, fontweight='bold')
            ax_bot.set_xticks(x)
            ax_bot.set_xticklabels(rdf['race_label'].values, fontsize=8, rotation=20, ha='right')
            # Range line showing spread
            ax_bot.axhspan(auc.min(), auc.max(), alpha=0.04, color='#1E8449')

        ax_bot.axhline(0.5, color='grey', ls=':', lw=1, alpha=0.5)
        ax_bot.set_ylim(0.4, 1.0)
        ax_bot.set_title(f'{sev}\nModel A — by Race Group', fontweight='bold', fontsize=10)
        ax_bot.set_xlabel('Race Group', fontsize=9)
        if col_i == 0:
            ax_bot.set_ylabel('AUROC (95% CI)', fontsize=10)

    fig.suptitle(
        f'{care_label} — AUROC Degradation: Care Frequency Quartile (top) vs Race Group (bottom)\n'
        f'Larger gap in top row supports care access as primary disparity axis',
        fontsize=12, fontweight='bold', y=1.01
    )
    plt.tight_layout()
    # [figure removed] plt.savefig(save_path, bbox_inches='tight', dpi=150)
    # [figure removed] plt.show()
    print(f'Saved: {save_path}')


# [unused figure removed] for care in ['Mouthcare','Turning']:
#     plot_quartile_vs_race_comparison(...)
# Superseded by cell 26 combo figures and cell 41 enhanced 2x2.


## 10d. Four-Model Comparison Framework

The analysis evaluates four model-evaluation configurations for each care phenotype and severity stratum:

| Configuration | Model | Trained on | Evaluated on | Purpose |
|---|---|---|---|---|
| 1 | Model A | All quartiles | Each racial/ethnic group | Baseline demographic fairness audit |
| 2 | Model A | All quartiles | Each care quartile | Reveal care-quality disparities |
| 3 | Model B | Q1 only | Each care quartile | Counterfactual degradation test: generalisation from optimal-care (Q1) to progressively lower-care conditions |
| 4 | Model B | Q1 only | Each racial/ethnic group | Cross-check: race-based evaluation of a care-quality-biased model |

**Race and care frequency are never used as training features in any configuration.** Any AUROC differences across groups arise entirely from patterns in clinical physiology learned during training.

If Configuration 3 shows significant Q1→Q4 degradation while Configurations 1 and 4 show stable race-based AUROC, this constitutes the core evidence that care access patterns — not race — are the primary axis of performance heterogeneity.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# COMBINED COMPARISON GRAPHS
# Defines combo1-5 plot functions. Figure generation DISABLED (savefig commented out).
# Functions read from results_df only. Re-enable savefig calls if these figures are needed.
# ═══════════════════════════════════════════════════════════════════════════════

# ── Shared helpers ────────────────────────────────────────────────────────────

def _get_q(results_df, care, sev, model_name):
    """Extract quartile-tested rows, sorted Q1→Q4."""
    mdf = results_df[
        (results_df['Care Type'] == care) &
        (results_df['Severity']  == sev) &
        (results_df['Model']     == model_name)
    ].copy()
    mdf['label'] = mdf['Test Set'].str.extract(r'(Q\d)')
    return mdf.dropna(subset=['label','AUROC']).sort_values('label').reset_index(drop=True)

def _get_r(results_df, care, sev, model_name):
    """Extract race-tested rows, sorted by RACE_LABELS order."""
    mdf = results_df[
        (results_df['Care Type'] == care) &
        (results_df['Severity']  == sev) &
        (results_df['Model']     == model_name)
    ].copy()
    mdf['label'] = mdf['Test Set'].str.replace('Race: ', '', regex=False)
    mdf['_ord']  = mdf['label'].map({r: i for i, r in enumerate(RACE_LABELS)})
    return mdf.dropna(subset=['label','AUROC','_ord']).sort_values('_ord').reset_index(drop=True)

def _plot_line(ax, mdf, color, marker, ls, label, annotate=True, lw=2.2,
               ann_x_offset=0, ann_y_offset=10):
    """Plot one model's line with CI bars and value labels.
    ann_x_offset / ann_y_offset: stagger annotations to avoid overlap.
    """
    if len(mdf) == 0:
        return
    x    = np.arange(len(mdf))
    auc  = mdf['AUROC'].values.astype(float)
    lo   = mdf['AUROC_lo'].fillna(mdf['AUROC']).values.astype(float)
    hi   = mdf['AUROC_hi'].fillna(mdf['AUROC']).values.astype(float)
    yerr = np.array([np.clip(auc - lo, 0, 1), np.clip(hi - auc, 0, 1)])
    ax.errorbar(x, auc, yerr=yerr, color=color, marker=marker,
                linestyle=ls, linewidth=lw, markersize=8,
                capsize=5, capthick=2, ecolor=color, alpha=0.9,
                label=label, zorder=3)
    if annotate:
        for xi, ai in enumerate(auc):
            ax.annotate(f'{ai:.3f}', (xi, ai),
                        textcoords='offset points',
                        xytext=(ann_x_offset, ann_y_offset),
                        ha='center', fontsize=8,
                        color=color, fontweight='bold', zorder=4)
    return auc

def _range_badge(ax, auc, color, pos='upper right'):
    """Small range badge in corner."""
    if auc is None or len(auc) == 0:
        return
    r = auc.max() - auc.min()
    x_pos = 0.98 if 'right' in pos else 0.02
    ha     = 'right' if 'right' in pos else 'left'
    ax.text(x_pos, 0.97, f'Range = {r:.3f}',
            transform=ax.transAxes, ha=ha, va='top',
            fontsize=8.5, color=color, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.35', facecolor='white',
                      edgecolor=color, alpha=0.85, linewidth=1.2))

def _style_ax(ax, xticks, xlabels, xlabel, ylabel='AUROC (95% CI)',
              xrot=0, ylim=(0.38, 1.02)):
    ax.set_xticks(xticks)
    ax.set_xticklabels(xlabels, fontsize=9,
                       rotation=xrot, ha='right' if xrot > 0 else 'center')
    ax.set_xlabel(xlabel, fontsize=9)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.set_ylim(ylim)
    ax.axhline(0.5, color='#BBBBBB', ls=':', lw=1.1, alpha=0.7)
    ax.axhline(0.7, color='#BBBBBB', ls='--', lw=0.9, alpha=0.5)
    ax.legend(fontsize=8.5, loc='lower left', framealpha=0.85)

Q_XLABELS = ['Q1\n(most freq)', 'Q2', 'Q3', 'Q4\n(least freq)']
SOFA_STR  = {'Low':'0–6', 'Medium':'7–11', 'High':'≥12'}


# ════════════════════════════════════════════════════════════════════════════════
# COMBO 1 — "Same Model, Two Lenses"
# Left panels : Model A → Race    (standard audit — should look flat)
# Right panels: Model A → Quartile (care audit  — should show variation)
# Message: The SAME model looks fair under one lens and unfair under the other.
# One figure per care × severity.
# ════════════════════════════════════════════════════════════════════════════════

def plot_combo1_same_model_two_lenses(results_df, care, sev, save_path):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), sharey=True)

    # Left: Model A → Race
    ax = axes[0]
    rdf  = _get_r(results_df, care, sev, 'A – By Race')
    auc_r = _plot_line(ax, rdf, '#2874A6', 'o', '-',
                       'Model A — tested by race group')
    if auc_r is not None:
        ax.axhspan(auc_r.min(), auc_r.max(), color='#2874A6', alpha=0.07)
        _range_badge(ax, auc_r, '#2874A6', 'upper right')
    ax.text(0.03, 0.04,
            '"Standard demographic audit:\n  model appears equitable."',
            transform=ax.transAxes, ha='left', va='bottom',
            fontsize=8.5, style='italic', color='#1A5276',
            bbox=dict(boxstyle='round,pad=0.4', facecolor='#EBF5FB',
                      edgecolor='#2874A6', alpha=0.9))
    _style_ax(ax,
              xticks=np.arange(len(rdf)),
              xlabels=rdf['label'].tolist() if len(rdf) > 0 else [],
              xlabel='Race / Ethnicity Group', xrot=18)
    ax.set_title('Model A  →  Tested by Race\n(dominant fairness paradigm)',
                 fontweight='bold', fontsize=11)

    # Right: Model A → Quartile
    ax = axes[1]
    qdf  = _get_q(results_df, care, sev, 'A – All Quartiles')
    auc_q = _plot_line(ax, qdf, '#E67E22', 's', '-',
                       'Model A — tested by care quartile')
    if auc_q is not None:
        ax.axhspan(auc_q.min(), auc_q.max(), color='#E67E22', alpha=0.07)
        _range_badge(ax, auc_q, '#E67E22', 'upper right')
    ax.text(0.03, 0.04,
            '"Care-quality audit reveals\n  what demographic audit misses."',
            transform=ax.transAxes, ha='left', va='bottom',
            fontsize=8.5, style='italic', color='#784212',
            bbox=dict(boxstyle='round,pad=0.4', facecolor='#FEF9E7',
                      edgecolor='#E67E22', alpha=0.9))
    _style_ax(ax,
              xticks=np.arange(len(qdf)),
              xlabels=Q_XLABELS[:len(qdf)],
              xlabel='Care Frequency Quartile')
    ax.set_title('Model A  →  Tested by Care Quartile\n(proposed care-quality paradigm)',
                 fontweight='bold', fontsize=11)

    fig.suptitle(
        f'Combo 1 — Same Model, Two Lenses\n'
        f'{care}  |  {sev} Severity (SOFA {SOFA_STR[sev]})  |  '
        f'Same trained model; only the test partition changes',
        fontsize=12, fontweight='bold', y=1.03
    )
    plt.tight_layout()
    # [figure removed] plt.savefig(save_path, bbox_inches='tight', dpi=150)
    # [figure removed] plt.show()
    print(f'Saved: {save_path}')


# ════════════════════════════════════════════════════════════════════════════════
# COMBO 2 — "Training Effect on Care Axis"
# Both lines on ONE panel per severity: A→Q vs B→Q
# Message: Restricting training to Q1 sharpens the degradation signal.
# If B drops more steeply than A, the training environment shapes what the
# model learns about physiology–mortality relationships.
# One figure per care type (3 severity panels side by side).
# ════════════════════════════════════════════════════════════════════════════════

def plot_combo2_training_effect(results_df, care, save_path):
    sevs = SEVERITY_LABELS
    fig, axes = plt.subplots(1, len(sevs), figsize=(5.5 * len(sevs), 5.5), sharey=True)

    for ax, sev in zip(axes, sevs):
        qdf_A = _get_q(results_df, care, sev, 'A – All Quartiles')
        qdf_B = _get_q(results_df, care, sev, 'B – Q1 Only')

        auc_A = _plot_line(ax, qdf_A, '#2874A6', 'o', '-',
                           'Model A (trained: all Q)', lw=2,
                           ann_x_offset=-8, ann_y_offset=10)
        auc_B = _plot_line(ax, qdf_B, '#C0392B', 's', '--',
                           'Model B (trained: Q1 only)', lw=2,
                           ann_x_offset=8, ann_y_offset=10)

        if auc_A is not None:
            _range_badge(ax, auc_A, '#2874A6', 'upper right')
        if auc_B is not None:
            _range_badge(ax, auc_B, '#C0392B', 'upper left')

        # Shade the gap between the two lines
        if auc_A is not None and auc_B is not None and len(auc_A) == len(auc_B):
            x = np.arange(len(auc_A))
            ax.fill_between(x, auc_A, auc_B,
                            where=(auc_B <= auc_A),
                            alpha=0.10, color='#C0392B',
                            label='Gap (A − B)')

        _style_ax(ax,
                  xticks=np.arange(len(QUARTILES)),
                  xlabels=Q_XLABELS,
                  xlabel='Care Frequency Quartile')
        ax.set_title(f'{sev} Severity\n(SOFA {SOFA_STR[sev]})',
                     fontweight='bold', fontsize=11)
        ax.set_ylabel('AUROC (95% CI)', fontsize=9)

    # Shared annotation
    fig.text(0.5, -0.04,
             'If Model B (red) drops more steeply than Model A (blue): '
             'the training environment shapes the physiology→mortality mapping.',
             ha='center', fontsize=9, style='italic', color='#555555')

    fig.suptitle(
        f'Combo 2 — Training Effect on Care Axis\n'
        f'{care}  |  Blue = trained on ALL quartiles  |  Red = trained on Q1 only',
        fontsize=12, fontweight='bold', y=1.03
    )
    plt.tight_layout()
    # [figure removed] plt.savefig(save_path, bbox_inches='tight', dpi=150)
    # [figure removed] plt.show()
    print(f'Saved: {save_path}')


# ════════════════════════════════════════════════════════════════════════════════
# COMBO 3 — "Race Stability Across Both Training Regimes"
# Both lines on ONE panel per severity: A→Race vs B→Race
# Message: Race produces no meaningful AUROC gap regardless of how you train.
# Flatness of both lines = race is not the axis of heterogeneity.
# One figure per care type (3 severity panels side by side).
# ════════════════════════════════════════════════════════════════════════════════

def plot_combo3_race_stability(results_df, care, save_path):
    sevs = SEVERITY_LABELS
    fig, axes = plt.subplots(1, len(sevs), figsize=(5.5 * len(sevs), 5.5), sharey=True)

    for ax, sev in zip(axes, sevs):
        rdf_A = _get_r(results_df, care, sev, 'A – By Race')
        rdf_B = _get_r(results_df, care, sev, 'B – Q1 By Race')

        auc_A = _plot_line(ax, rdf_A, '#2874A6', 'o', '-',
                           'Model A (trained: all Q)', lw=2,
                           ann_x_offset=-8, ann_y_offset=10)
        auc_B = _plot_line(ax, rdf_B, '#1E8449', '^', '--',
                           'Model B (trained: Q1 only)', lw=2,
                           ann_x_offset=8, ann_y_offset=10)

        if auc_A is not None:
            _range_badge(ax, auc_A, '#2874A6', 'upper right')
        if auc_B is not None:
            _range_badge(ax, auc_B, '#1E8449', 'upper left')

        xlabels = rdf_A['label'].tolist() if len(rdf_A) > 0 else rdf_B['label'].tolist()
        _style_ax(ax,
                  xticks=np.arange(len(xlabels)),
                  xlabels=xlabels,
                  xlabel='Race / Ethnicity Group',
                  xrot=18)
        ax.set_title(f'{sev} Severity\n(SOFA {SOFA_STR[sev]})',
                     fontweight='bold', fontsize=11)
        ax.set_ylabel('AUROC (95% CI)', fontsize=9)

    fig.text(0.5, -0.06,
             'Both lines flat across race regardless of training regime → '
             'race does not modify the physiology→mortality relationship.',
             ha='center', fontsize=9, style='italic', color='#555555')

    fig.suptitle(
        f'Combo 3 — Race Stability Across Both Training Regimes\n'
        f'{care}  |  Blue = trained on ALL Q  |  Green = trained on Q1 only',
        fontsize=12, fontweight='bold', y=1.03
    )
    plt.tight_layout()
    # [figure removed] plt.savefig(save_path, bbox_inches='tight', dpi=150)
    # [figure removed] plt.show()
    print(f'Saved: {save_path}')


# ════════════════════════════════════════════════════════════════════════════════
# COMBO 4 — "Master Figure: All 4 Lines, Two Sections"
# All 4 model lines on ONE panel per severity.
# X-axis has two sections divided by a vertical line:
#   LEFT  = quartile axis (Q1–Q4)
#   RIGHT = race axis (race groups)
# This is the single figure that contains the complete argument.
# One figure per care type (3 severity panels side by side).
# ════════════════════════════════════════════════════════════════════════════════

def plot_combo4_master(results_df, care, save_path):
    sevs = SEVERITY_LABELS
    fig, axes = plt.subplots(1, len(sevs), figsize=(6.5 * len(sevs), 6), sharey=True)

    model_specs = [
        # (model_name, getter_fn, color, marker, ls, short_label)
        ('A – All Quartiles', 'q', '#2874A6', 'o', '-',
         'A→Quartile (all Q trained)'),
        ('B – Q1 Only',       'q', '#C0392B', 's', '--',
         'B→Quartile (Q1 trained)'),
        ('A – By Race',       'r', '#E67E22', 'D', '-',
         'A→Race (all Q trained)'),
        ('B – Q1 By Race',    'r', '#1E8449', '^', '--',
         'B→Race (Q1 trained)'),
    ]

    for ax, sev in zip(axes, sevs):
        # First pass: get race labels for right section
        rdf_sample = _get_r(results_df, care, sev, 'A – By Race')
        n_q   = len(QUARTILES)
        n_r   = len(rdf_sample)
        gap   = 1.2
        x_q   = np.arange(n_q)
        x_r   = np.arange(n_q + gap, n_q + gap + n_r)

        all_auc_ranges = []

        for mname, gtype, color, marker, ls, label in model_specs:
            if gtype == 'q':
                mdf = _get_q(results_df, care, sev, mname)
                x_use = x_q[:len(mdf)]
            else:
                mdf = _get_r(results_df, care, sev, mname)
                x_use = x_r[:len(mdf)]

            if len(mdf) == 0:
                continue

            auc  = mdf['AUROC'].values.astype(float)
            lo   = mdf['AUROC_lo'].fillna(mdf['AUROC']).values.astype(float)
            hi   = mdf['AUROC_hi'].fillna(mdf['AUROC']).values.astype(float)
            yerr = np.array([np.clip(auc - lo, 0, 1), np.clip(hi - auc, 0, 1)])

            ax.errorbar(x_use, auc, yerr=yerr,
                        color=color, marker=marker, linestyle=ls,
                        linewidth=2, markersize=7.5, capsize=4, capthick=1.8,
                        ecolor=color, alpha=0.88,
                        label=f'{label}  [range={auc.max()-auc.min():.3f}]',
                        zorder=3)
            for xi, (ai, offset_x) in enumerate(zip(auc, [0, 6, -6, 0, 8, -8, 4, -4, 6][:len(auc)])):
                # stagger horizontally to avoid overlap between 4 model lines
                offset_y = 9 + (list(model_specs).index((mname, gtype, color, marker, ls, label)) % 2) * 11
                ax.annotate(f'{ai:.3f}', (x_use[xi], ai),
                            textcoords='offset points',
                            xytext=(offset_x, offset_y),
                            ha='center', fontsize=7,
                            color=color, fontweight='bold', zorder=4)
            all_auc_ranges.append((label, auc.max() - auc.min(), color))

        # Vertical divider between quartile and race sections
        if n_r > 0:
            div_x = n_q + gap / 2 - 0.1
            ax.axvline(div_x, color='#999999', lw=1.5, ls='--', alpha=0.6, zorder=1)
            # Place axis labels below the subplot using figure-relative y
            # so they don't clash with x-tick labels
            ax.text(n_q / 2 - 0.5, ax.get_ylim()[0] - 0.13,
                    '◄ Care axis', fontsize=8.5, color='#2874A6',
                    style='italic', fontweight='bold',
                    ha='center', va='top',
                    transform=ax.transData)
            ax.text(n_q + gap + n_r / 2 - 0.5, ax.get_ylim()[0] - 0.13,
                    'Race axis ►', fontsize=8.5, color='#E67E22',
                    style='italic', fontweight='bold',
                    ha='center', va='top',
                    transform=ax.transData)

        # X ticks: quartile labels + race labels
        all_x      = list(x_q) + list(x_r)
        race_lbls  = rdf_sample['label'].tolist() if n_r > 0 else []
        all_labels = Q_XLABELS[:n_q] + race_lbls

        ax.set_xticks(all_x)
        ax.set_xticklabels(all_labels, fontsize=8,
                           rotation=18, ha='right')
        ax.axhline(0.5, color='#BBBBBB', ls=':', lw=1.1, alpha=0.7)
        ax.axhline(0.7, color='#BBBBBB', ls='--', lw=0.9, alpha=0.5)
        ax.set_ylim(0.30, 1.05)  # extra bottom room for care/race axis labels
        ax.set_title(f'{sev} Severity  (SOFA {SOFA_STR[sev]})',
                     fontweight='bold', fontsize=11)
        ax.set_ylabel('AUROC (95% CI)', fontsize=9)
        ax.legend(fontsize=7.5, loc='lower left',
                  framealpha=0.88, handlelength=2.0,
                  ncol=1)

    fig.suptitle(
        f'Combo 4 — Master Figure: All 4 Models, Care Axis vs Race Axis\n'
        f'{care}  |  LEFT of divider = care quartile test  |  '
        f'RIGHT of divider = race group test\n'
        f'Key: care-axis lines should slope ↓; race-axis lines should stay flat',
        fontsize=11, fontweight='bold', y=1.04
    )
    plt.tight_layout()
    # [figure removed] plt.savefig(save_path, bbox_inches='tight', dpi=150)
    # [figure removed] plt.show()
    print(f'Saved: {save_path}')


# ════════════════════════════════════════════════════════════════════════════════
# COMBO 5 — "The Crossover" (bonus)
# 2×2: rows = care axis / race axis, cols = Model A / Model B
# Directly answers: does the training regime matter for each axis?
# If rows look different but columns look similar → axis > training regime.
# One figure per care type × severity.
# ════════════════════════════════════════════════════════════════════════════════

def plot_combo5_crossover(results_df, care, sev, save_path):
    fig, axes = plt.subplots(2, 2, figsize=(12, 9), sharey=True)

    specs = [
        # (row, col, model_name, getter, color, marker, ls, title)
        (0, 0, 'A – All Quartiles', 'q', '#2874A6', 'o', '-',
         'Model A → Care Quartile\n(trained on all Q)'),
        (0, 1, 'B – Q1 Only',       'q', '#C0392B', 's', '--',
         'Model B → Care Quartile\n(trained on Q1 only)'),
        (1, 0, 'A – By Race',       'r', '#E67E22', 'D', '-',
         'Model A → Race Group\n(trained on all Q)'),
        (1, 1, 'B – Q1 By Race',    'r', '#1E8449', '^', '--',
         'Model B → Race Group\n(trained on Q1 only)'),
    ]

    row_labels = ['CARE AXIS', 'RACE AXIS']
    claim_texts = [
        ['Performance varies\nacross care quartiles',
         'Degradation sharpens\nwhen trained on Q1 only'],
        ['Stable across race\n(even with all-Q training)',
         'Still stable across race\n(even with Q1-only training)'],
    ]
    claim_colors = [
        ['#784212', '#7B241C'],
        ['#1A5276', '#145A32'],
    ]
    claim_bgs = [
        ['#FEF9E7', '#FDEDEC'],
        ['#EBF5FB', '#EAFAF1'],
    ]

    for row, col, mname, gtype, color, marker, ls, title in specs:
        ax = axes[row, col]
        mdf  = _get_q(results_df, care, sev, mname) if gtype == 'q' \
               else _get_r(results_df, care, sev, mname)
        auc = _plot_line(ax, mdf, color, marker, ls, mname, lw=2.2)

        if auc is not None:
            ax.axhspan(auc.min(), auc.max(), color=color, alpha=0.07)
            _range_badge(ax, auc, color, 'upper right')

        ax.text(0.03, 0.04, claim_texts[row][col],
                transform=ax.transAxes, ha='left', va='bottom',
                fontsize=8.5, style='italic',
                color=claim_colors[row][col],
                bbox=dict(boxstyle='round,pad=0.4',
                          facecolor=claim_bgs[row][col],
                          edgecolor=claim_colors[row][col],
                          alpha=0.90))

        if gtype == 'q':
            xlabels = Q_XLABELS[:len(mdf)]
            xlabel  = 'Care Quartile'
            xrot    = 0
        else:
            xlabels = mdf['label'].tolist() if len(mdf) > 0 else []
            xlabel  = 'Race Group'
            xrot    = 18

        _style_ax(ax,
                  xticks=np.arange(len(xlabels)),
                  xlabels=xlabels,
                  xlabel=xlabel,
                  xrot=xrot)
        ax.set_title(title, fontweight='bold', fontsize=10)
        if col == 0:
            ax.set_ylabel('AUROC (95% CI)', fontsize=9)

    # Row labels on left margin
    for r, label in enumerate(row_labels):
        fig.text(0.01, 0.74 - r * 0.47, label,
                 va='center', ha='left', fontsize=10,
                 color='#444444', fontweight='bold', rotation=90)

    # Column labels on top
    for c, label in enumerate(['Model A\n(trained on all Q)', 'Model B\n(trained on Q1 only)']):
        axes[0, c].set_title(f'{label}\n' + axes[0, c].get_title(),
                              fontweight='bold', fontsize=10)

    fig.suptitle(
        f'Combo 5 — Training Regime × Test Axis (2×2)\n'
        f'{care}  |  {sev} Severity (SOFA {SOFA_STR[sev]})\n'
        f'Rows = what you test on  |  Cols = what you trained on',
        fontsize=12, fontweight='bold', y=1.03
    )
    plt.tight_layout(rect=[0.04, 0, 1, 1])
    # [figure removed] plt.savefig(save_path, bbox_inches='tight', dpi=150)
    # [figure removed] plt.show()
    print(f'Saved: {save_path}')


# ════════════════════════════════════════════════════════════════════════════════
# RUN ALL COMBINATIONS — [unused figures removed — functions kept above]
# These combo figures are not used in the paper.

# [unused] for care in ['Mouthcare', 'Turning']:

    # Combo 1: per care × severity (6 figures total)
# [unused]     for sev in SEVERITY_LABELS:
# [unused]         sub = results_df[
# [unused]             (results_df['Care Type'] == care) &
# [unused]             (results_df['Severity']  == sev)
# [unused]         ]
# [unused]         if len(sub) == 0:
# [unused]             continue
# [unused]         plot_combo1_same_model_two_lenses(
# [unused]             results_df, care, sev,
# [unused]             os.path.join(OUTPUT_FOLDER,
# [unused]                          f'combo1_samelens_{care.lower()}_{sev.lower()}.png')
# [unused]         )

    # Combo 2: per care (2 figures — 3 severity panels each)
# [unused]     plot_combo2_training_effect(
# [unused]         results_df, care,
# [unused]         os.path.join(OUTPUT_FOLDER, f'combo2_training_effect_{care.lower()}.png')
# [unused]     )

    # Combo 3: per care (2 figures — 3 severity panels each)
# [unused]     plot_combo3_race_stability(
# [unused]         results_df, care,
# [unused]         os.path.join(OUTPUT_FOLDER, f'combo3_race_stability_{care.lower()}.png')
# [unused]     )

    # Combo 4: per care (2 figures — 3 severity panels each)
# [unused]     plot_combo4_master(
# [unused]         results_df, care,
# [unused]         os.path.join(OUTPUT_FOLDER, f'combo4_master_{care.lower()}.png')
# [unused]     )

    # Combo 5: per care × severity (6 figures total)
# [unused]     for sev in SEVERITY_LABELS:
# [unused]         sub = results_df[
# [unused]             (results_df['Care Type'] == care) &
# [unused]             (results_df['Severity']  == sev)
# [unused]         ]
# [unused]         if len(sub) == 0:
# [unused]             continue
# [unused]         plot_combo5_crossover(
# [unused]             results_df, care, sev,
# [unused]             os.path.join(OUTPUT_FOLDER,
# [unused]                          f'combo5_crossover_{care.lower()}_{sev.lower()}.png')
# [unused]         )

# [unused] print('\n All combination figures saved.')

## 10e. Basic 2×2 AUROC Panel (Superseded)

*Function definitions for `_draw_race_panel`, `_draw_quartile_panel`, `plot_2x2_per_severity` are defined here and are **required by cell 41**. Figure generation is disabled (superseded). Cell 41 produces the paper figures with DeLong stability badges, directional JT, and Spearman annotations.*

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  SUPERSEDED BY CELL 41 (enhanced 2×2 with stat annotations)     ║
# ║  Produces fig_2x2_*.png — same panel layout as cell 41 but      ║
# ║  without DeLong brackets, Spearman ρ, or JT p-values.           ║
# ║  Cell 41 generates fig_enhanced2x2_*.png with full stat         ║
# ║  annotations — use that for the paper.                          ║
# ║  _draw_race_panel and _draw_quartile_panel defined here are      ║
# ║  REUSED by cell 41 — do not delete this cell.                   ║
# ╚══════════════════════════════════════════════════════════════════╝

# ══════════════════════════════════════════════════════════════════════════════
# FIGURE: 2×2 AUROC Panel  —  one figure per care type × severity (6 total)
#
# Panel A (row 0, col 0): Model A → Race       — dots, no line
# Panel B (row 0, col 1): Model A → Quartile   — connected line
# Panel C (row 1, col 1): Model B → Quartile   — connected line
# Panel D (row 1, col 0): Model B → Race       — dots, no line
#
# Shading semantics:
#   RACE panels  → cool blue-grey axhspan (neutral/stable) + "Stable" label
#   QUARTILE panels → warm fill_between line and Q1 baseline (shows the loss
#                     triangle below the declining line) + axhspan range band
#
# CI note:
#   Race  CI wider because: smaller group n + bootstrap (vs CV mean±SE).
#   Both are correct — annotated in figure footer.
# ══════════════════════════════════════════════════════════════════════════════

RACE_COLOURS_ALL = {
    'White':           '#2874A6',
    'Black':           '#C0392B',
    'Hispanic/Latino': '#E67E22',
    'Asian':           '#1E8449',
    'Other':           '#7D3C98',
}
SOFA_STR_2x2 = {'Low': '0–6', 'Medium': '7–11', 'High': '≥12'}
MODEL_A_COL  = '#2874A6'
MODEL_B_COL  = '#C0392B'
STABLE_SHADE = '#90A4AE'   # cool blue-grey for race "stable" band
DECLINE_WARM = '#E74C3C'   # warm red for quartile "decline" fill


def _range_badge_2x2(ax, auc_arr, color, pos='upper right', label_prefix='Range'):
    if auc_arr is None or len(auc_arr) == 0:
        return
    r = float(auc_arr.max() - auc_arr.min())
    x_pos = 0.98 if 'right' in pos else 0.02
    ha    = 'right' if 'right' in pos else 'left'
    ax.text(x_pos, 0.97, f'{label_prefix} = {r:.3f}',
            transform=ax.transAxes, ha=ha, va='top',
            fontsize=8.5, color=color, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.35', facecolor='white',
                      edgecolor=color, alpha=0.85, linewidth=1.2))


def _style_ax_2x2(ax, xticks, xlabels, xlabel, xrot=0):
    ax.set_xticks(xticks)
    ax.set_xticklabels(xlabels, fontsize=9,
                       rotation=xrot, ha='right' if xrot > 0 else 'center')
    ax.set_xlabel(xlabel, fontsize=9)
    ax.set_ylabel('AUROC (95% CI)', fontsize=9)
    ax.set_ylim(0.38, 1.02)
    ax.axhline(0.5, color='#BBBBBB', ls=':', lw=1.1, alpha=0.7)
    ax.axhline(0.7, color='#BBBBBB', ls='--', lw=0.9, alpha=0.5)
    ax.set_facecolor('#F8F9FA')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='y', alpha=0.25, linestyle='--', zorder=0)


def _draw_race_panel(ax, results_df, care, sev, model_name, model_col, panel_label):
    """
    Race panel: scatter dots + CI bars, no connecting line.
    Shading: cool blue-grey axhspan across the range (signals STABILITY).
    Wide CI is expected and annotated — race groups smaller + bootstrap CI.
    """
    df = results_df[
        (results_df['Care Type'] == care) &
        (results_df['Severity']  == sev)  &
        (results_df['Model']     == model_name) &
        (results_df['Test Set'].str.startswith('Race:'))
    ].copy()
    df['race'] = df['Test Set'].str.replace('Race: ', '', regex=False)
    df['_ord'] = df['race'].map({r: i for i, r in enumerate(RACE_LABELS)})
    df = df.dropna(subset=['_ord', 'AUROC']).sort_values('_ord').reset_index(drop=True)

    if len(df) == 0:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center',
                transform=ax.transAxes, fontsize=11, color='grey')
        _style_ax_2x2(ax, [], [], 'Race Group', xrot=18)
        return

    auc = df['AUROC'].values.astype(float)
    lo  = df['AUROC_lo'].fillna(df['AUROC']).values.astype(float)
    hi  = df['AUROC_hi'].fillna(df['AUROC']).values.astype(float)

    # Cool grey-blue band = STABLE / no systematic variation
    ax.axhspan(float(auc.min()), float(auc.max()),
               color=STABLE_SHADE, alpha=0.12, zorder=0)
    # Subtle dashed lines at edges
    ax.axhline(float(auc.min()), color=STABLE_SHADE, lw=0.8,
               ls='--', alpha=0.5, zorder=1)
    ax.axhline(float(auc.max()), color=STABLE_SHADE, lw=0.8,
               ls='--', alpha=0.5, zorder=1)
    _range_badge_2x2(ax, auc, STABLE_SHADE, 'upper right', label_prefix='Stable range')

    x = np.arange(len(df))
    for xi, (_, row_d) in enumerate(df.iterrows()):
        race = row_d['race']
        ai   = float(row_d['AUROC'])
        li   = float(row_d['AUROC_lo']) if row_d['AUROC_lo'] is not None else ai
        hi_v = float(row_d['AUROC_hi']) if row_d['AUROC_hi'] is not None else ai
        col  = RACE_COLOURS_ALL.get(race, '#555555')

        ax.errorbar(xi, ai, yerr=[[ai - li], [hi_v - ai]],
                    fmt='none', ecolor=col, capsize=5, capthick=2,
                    lw=1.6, zorder=3, alpha=0.75)
        ax.scatter(xi, ai, color=col, s=100, zorder=5,
                   edgecolors='white', linewidths=1.3)
        ax.annotate(f'{ai:.3f}', (xi, hi_v),
                    textcoords='offset points', xytext=(0, 7),
                    ha='center', fontsize=8, color=col,
                    fontweight='bold', zorder=4)

    # Wide CI footnote
    ax.text(0.02, 0.04,
            'Wide CI = small group n\n+ bootstrap method',
            transform=ax.transAxes, ha='left', va='bottom',
            fontsize=7.5, color='#666666', style='italic',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                      edgecolor='#CCCCCC', alpha=0.85))

    _style_ax_2x2(ax, x, df['race'].tolist(), 'Race / Ethnicity Group', xrot=18)
    ax.set_title(f'{panel_label}  |  {model_name}\ntested by race  —  expect stability',
                 fontweight='bold', fontsize=9.5)


def _draw_quartile_panel(ax, results_df, care, sev, model_name, model_col, panel_label):
    """
    Quartile panel: connected line + CI bars.
    Shading:
      - fill_between(x, Q1_reference, auc) = 'loss triangle' below Q1 level
        Shows how much AUROC has been lost compared to Q1-level performance.
      - axhspan range band in warm red = signals DECLINING trend.
    """
    df = results_df[
        (results_df['Care Type'] == care) &
        (results_df['Severity']  == sev)  &
        (results_df['Model']     == model_name) &
        (~results_df['Test Set'].str.startswith('Race:'))
    ].copy()
    df['Q'] = df['Test Set'].str.extract(r'(Q\d)')
    df = df.dropna(subset=['Q', 'AUROC']).sort_values('Q').reset_index(drop=True)

    if len(df) == 0:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center',
                transform=ax.transAxes, fontsize=11, color='grey')
        _style_ax_2x2(ax, np.arange(4), Q_XLABELS, 'Care Frequency Quartile')
        return

    x    = np.arange(len(df))
    auc  = df['AUROC'].values.astype(float)
    lo   = df['AUROC_lo'].fillna(df['AUROC']).values.astype(float)
    hi   = df['AUROC_hi'].fillna(df['AUROC']).values.astype(float)
    yerr = np.array([np.clip(auc - lo, 0, 1), np.clip(hi - auc, 0, 1)])

    # ── Semantic shading ──────────────────────────────────────────────────────
    # 1. Warm axhspan = range band (signals declining trend, not neutral)
    ax.axhspan(float(auc.min()), float(auc.max()),
               color=DECLINE_WARM, alpha=0.06, zorder=0)
    ax.axhline(float(auc.min()), color=DECLINE_WARM, lw=0.8,
               ls='--', alpha=0.4, zorder=1)
    ax.axhline(float(auc.max()), color=DECLINE_WARM, lw=0.8,
               ls='--', alpha=0.4, zorder=1)
    _range_badge_2x2(ax, auc, DECLINE_WARM, 'upper right', label_prefix='Decline range')

    # 2. Loss triangle: fill between Q1 AUROC level and the actual line
    #    Only shade below the Q1 reference (shows what's been lost)
    if len(auc) >= 2:
        q1_level = float(auc[0])   # Q1 AUROC = best-care reference
        # fill between Q1 reference and the line wherever line < Q1
        ax.fill_between(x, q1_level, auc,
                        where=(auc < q1_level),
                        color=DECLINE_WARM, alpha=0.18,
                        label='Loss vs Q1', zorder=1)
        # Thin Q1 reference line
        ax.axhline(q1_level, color=model_col, lw=1.0,
                   ls=':', alpha=0.5, zorder=2)

    # Line + error bars
    ax.errorbar(x, auc, yerr=yerr,
                color=model_col, marker='o', linestyle='-',
                linewidth=2.2, markersize=8,
                capsize=5, capthick=2, ecolor=model_col, alpha=0.9,
                label=f'{model_name}  [n={len(df)} Q]', zorder=4)

    for xi, (ai, hi_v) in enumerate(zip(auc, hi)):
        ax.annotate(f'{ai:.3f}', (xi, hi_v),
                    textcoords='offset points', xytext=(0, 7),
                    ha='center', fontsize=8,
                    color=model_col, fontweight='bold', zorder=5)

    ax.legend(fontsize=8, loc='lower left', framealpha=0.85)
    _style_ax_2x2(ax, x, Q_XLABELS[:len(df)], 'Care Frequency Quartile')
    ax.set_title(f'{panel_label}  |  {model_name}\ntested by care quartile  —  expect ↓ trend',
                 fontweight='bold', fontsize=9.5)


def plot_2x2_per_severity(results_df, care_label, sev_label, save_path):
    fig, axes = plt.subplots(2, 2, figsize=(13, 10))
    fig.patch.set_facecolor('#FAFAFA')

    _draw_race_panel(axes[0, 0], results_df, care_label, sev_label,
                     'A – By Race', MODEL_A_COL, 'Panel A')
    _draw_quartile_panel(axes[0, 1], results_df, care_label, sev_label,
                         'A – All Quartiles', MODEL_A_COL, 'Panel B')
    _draw_quartile_panel(axes[1, 1], results_df, care_label, sev_label,
                         'B – Q1 Only', MODEL_B_COL, 'Panel C')
    _draw_race_panel(axes[1, 0], results_df, care_label, sev_label,
                     'B – Q1 By Race', MODEL_B_COL, 'Panel D')

    # Row labels on left margin
    for ax_row, label in [(0, 'Model A\n(trained: all Q)'),
                           (1, 'Model B\n(trained: Q1 only)')]:
        axes[ax_row, 0].annotate(
            label, xy=(-0.28, 0.5), xycoords='axes fraction',
            fontsize=10, fontweight='bold', va='center', ha='center',
            rotation=90, color='#333333', annotation_clip=False)

    # Column headers
    for ax_col, label in [(0, 'Evaluated by Race\n(expect: flat / stable)'),
                           (1, 'Evaluated by Care Quartile\n(expect: ↓ declining Q1→Q4)')]:
        axes[0, ax_col].set_title(
            axes[0, ax_col].get_title() + f'\n▲ {label}',
            fontweight='bold', fontsize=9.5)

    fig.suptitle(
        f'{care_label}  —  {sev_label} Severity  (SOFA {SOFA_STR_2x2[sev_label]})\n'
        '2×2 AUROC Evaluation Panel\n'
        'Race (left): cool band = stable / no disparity expected   |   '
        'Quartile (right): warm band + loss triangle = declining trend expected\n'
        'CI: quartile = CV mean±1.96×SE  |  race = bootstrap 1000 resamples  '
        '(race CI wider due to smaller group n)',
        fontsize=10.5, fontweight='bold', y=1.04
    )
    plt.tight_layout()
    # [figure removed] plt.savefig(save_path, dpi=150, bbox_inches='tight')
    # [figure removed] plt.show()
    print(f'  Saved: {save_path}')


print('Generating 2×2 panels (one per care × severity)...')
for care in ['Mouthcare', 'Turning']:
    for sev in SEVERITY_LABELS:
        fname = f'fig_2x2_{care.lower()}_{sev.lower()}.png'
        plot_2x2_per_severity(
            results_df, care, sev,
            os.path.join(OUTPUT_FOLDER, fname)
        )
print('Done — 6 figures generated.')


## 10b. Pre-Specified Statistical Analysis Framework

All statistical tests follow a pre-specified hierarchy applied to each (care phenotype, severity stratum) combination.

| Configuration | Axis | Primary test | Step 1 — Symmetric trend test | Step 2 — Directional test |
|---|---|---|---|---|
| 1 — Model A by Race | Nominal | DeLong all pairs + Bonferroni | Best-ordering permutation JT | — |
| 2 — Model A by Quartile | Ordered | DeLong all pairs + Bonferroni | Best-ordering permutation JT | Natural-ordering JT (Q1→Q4) + Spearman |
| 3 — Model B by Quartile | Ordered | DeLong all pairs + Bonferroni | Best-ordering permutation JT | Natural-ordering JT (Q1→Q4) + Spearman |
| 4 — Model B by Race | Nominal | DeLong all pairs + Bonferroni | Best-ordering permutation JT | — |

**Two-step trend testing design:**

**Step 1 — Best-ordering permutation JT (all four configurations, symmetric):**
For each configuration, groups are sorted by their observed mean AUROC in descending order — the single ordering most favourable for detecting a decreasing monotonic trend — and the JT statistic for that ordering is compared to a permutation null distribution (1,000 resamples, observations randomly shuffled across groups and re-sorted by permuted means). This gives every configuration the maximum possible chance to show a trend, making the test symmetric: if a non-significant result is obtained for the race axis (Configs 1 and 4), it means no monotonic trend exists even under the most favourable conditions, which is the strongest possible evidence of race-axis stability directly comparable to the trend detected on the care axis.

**Step 2 — Natural-ordering JT (Configs 2 and 3 only):**
After Step 1 establishes whether any trend exists, the natural Q1→Q4 ordering JT tests specifically whether the trend is in the hypothesised decreasing direction. This directional step is only applied to the quartile axis where a natural ordering and predicted direction exist. Spearman rank correlation is reported as a complementary descriptive summary.

**Confidence intervals:** Quartile = mean ± 1.96×SE across 5 CV folds. Race and ΔAUROC = bootstrap 1,000 resamples on pooled predictions.

In [ ]:

# ══════════════════════════════════════════════════════════════════════════
# SHARED STAT HELPERS  (run once before any stat/figure cells)
# ══════════════════════════════════════════════════════════════════════════
from scipy import stats
from scipy.stats import chi2_contingency, kruskal, spearmanr
from statsmodels.stats.multitest import multipletests
from itertools import combinations
import warnings; warnings.filterwarnings('ignore')

# ── DeLong helpers ────────────────────────────────────────────────────────
def _placement_values(y_true, y_prob):
    y_true = np.array(y_true, dtype=float)
    y_prob = np.array(y_prob, dtype=float)
    pos = y_prob[y_true == 1]; neg = y_prob[y_true == 0]
    V10 = np.array([np.mean(p > neg) + 0.5*np.mean(p == neg) for p in pos])
    V01 = np.array([np.mean(n < pos) + 0.5*np.mean(n == pos) for n in neg])
    return V10, V01

def delong_paired(y_true, y_prob_A, y_prob_B):
    """Paired DeLong — same patients scored by A and B."""
    y_true=np.array(y_true,dtype=float)
    y_prob_A=np.array(y_prob_A,dtype=float)
    y_prob_B=np.array(y_prob_B,dtype=float)
    n1=int(y_true.sum()); n0=int(len(y_true)-n1)
    if n1<2 or n0<2: return np.nan,np.nan,np.nan,np.nan
    V10_A,V01_A=_placement_values(y_true,y_prob_A)
    V10_B,V01_B=_placement_values(y_true,y_prob_B)
    auc_A=float(V10_A.mean()); auc_B=float(V10_B.mean())
    V10=np.column_stack([V10_A,V10_B]); V01=np.column_stack([V01_A,V01_B])
    S=(np.cov(V10,rowvar=False,ddof=1)/n1 + np.cov(V01,rowvar=False,ddof=1)/n0)
    var_diff=S[0,0]-2*S[0,1]+S[1,1]
    if var_diff<=0: return auc_A,auc_B,np.nan,np.nan
    z=(auc_A-auc_B)/np.sqrt(var_diff)
    p=2*(1-stats.norm.cdf(abs(z)))
    return auc_A,auc_B,z,p

def delong_independent(yt_A,yp_A,yt_B,yp_B):
    """Independent DeLong — different patients in each group."""
    V10_A,V01_A=_placement_values(yt_A,yp_A)
    V10_B,V01_B=_placement_values(yt_B,yp_B)
    auc_A=float(V10_A.mean()); auc_B=float(V10_B.mean())
    n1A,n0A=len(V10_A),len(V01_A); n1B,n0B=len(V10_B),len(V01_B)
    vA=np.var(V10_A,ddof=1)/n1A+np.var(V01_A,ddof=1)/n0A
    vB=np.var(V10_B,ddof=1)/n1B+np.var(V01_B,ddof=1)/n0B
    if vA+vB<=0: return auc_A,auc_B,np.nan,np.nan
    z=(auc_A-auc_B)/np.sqrt(vA+vB)
    p=2*(1-stats.norm.cdf(abs(z)))
    return auc_A,auc_B,z,p

# ── Jonckheere-Terpstra (uses real per-fold AUROCs) ───────────────────────
def jonckheere_trend_test(groups_obs):
    """H1: group_1 > group_2 > ... > group_k (decreasing).
    groups_obs: list of arrays, ordered Q1…Q4."""
    k=len(groups_obs); jt=0
    for i in range(k-1):
        for j in range(i+1,k):
            u,_=stats.mannwhitneyu(groups_obs[i],groups_obs[j],alternative='greater')
            jt+=u
    n_i=[len(g) for g in groups_obs]; N=sum(n_i)
    E=(N**2-sum(n**2 for n in n_i))/4
    V=(N**2*(2*N+3)-sum(n**2*(2*n+3) for n in n_i))/72
    if V<=0: return jt,np.nan
    z=(jt-E)/np.sqrt(V); p=1-stats.norm.cdf(z)
    return jt,p




# ── Best-ordering permutation JT (symmetric trend test for all 4 configs) ─────
def permutation_max_jt(groups_obs, n_perm=200, seed=42):
    """
    Sort groups by observed mean AUROC descending (most favourable ordering
    for detecting a decreasing trend) then test against permutation null.
    Valid for both ordered (quartile) and unordered (race) axes.
    Non-sig on race axis = no trend exists even under best-case ordering.
    n_perm=200 is the default for internal helper calls; all paper-reported results use n_perm=1000 (see cells 32, 35, and the stat block in cell 31).
    """
    rng = np.random.default_rng(seed)
    means = [np.mean(g) for g in groups_obs]
    order = np.argsort(means)[::-1]
    ordered = [groups_obs[i] for i in order]
    jt_obs, _ = jonckheere_trend_test(ordered)
    if np.isnan(jt_obs):
        return np.nan, np.nan
    all_obs = np.concatenate(groups_obs)
    ns = [len(g) for g in groups_obs]
    null_jts = []
    for _ in range(n_perm):
        shuffled = rng.permutation(all_obs)
        pg, idx = [], 0
        for n in ns:
            pg.append(shuffled[idx:idx+n]); idx += n
        pg_s = [pg[i] for i in np.argsort([np.mean(g) for g in pg])[::-1]]
        jt_null, _ = jonckheere_trend_test(pg_s)
        if not np.isnan(jt_null):
            null_jts.append(jt_null)
    p_perm = np.mean(np.array(null_jts) >= jt_obs) if null_jts else np.nan
    return jt_obs, p_perm

# ── Cochran-Armitage trend test for proportions ───────────────────────────
def cochran_armitage(counts, totals):
    """Linear trend in proportions. scores = 1,2,3,4."""
    c=np.array(counts,dtype=float); t=np.array(totals,dtype=float)
    s=np.arange(1,len(c)+1,dtype=float)
    N=t.sum(); n=c.sum(); p_bar=n/N; w_bar=(t*s).sum()/N
    num=(t*s*(c/t-p_bar)).sum()
    den_sq=p_bar*(1-p_bar)*((t*(s-w_bar)**2).sum()-((t*(s-w_bar)).sum())**2/N)
    if den_sq<=0: return np.nan,np.nan
    z=num/np.sqrt(den_sq)
    # One-sided: H1 = mortality INCREASES Q1→Q4 (positive z = increasing trend)
    p=1-stats.norm.cdf(z)
    return z,p

# ── Bootstrap AUROC CI (used for race and delta plots) ────────────────────
def bootstrap_auroc_ci(y_true, y_prob, n=1000, seed=42):
    rng=np.random.default_rng(seed)
    yt,yp=np.array(y_true),np.array(y_prob)
    idx=np.arange(len(yt)); aucs=[]
    for _ in range(n):
        b=rng.choice(idx,size=len(idx),replace=True)
        if len(np.unique(yt[b]))<2: continue
        aucs.append(roc_auc_score(yt[b],yp[b]))
    if not aucs: return np.nan,np.nan,np.nan
    return float(np.mean(aucs)),float(np.percentile(aucs,2.5)),float(np.percentile(aucs,97.5))

def _fmt(p):
    if p is None or (isinstance(p,float) and np.isnan(p)): return '—'
    if p<0.001: return 'p<0.001***'
    if p<0.01:  return f'p={p:.4f}**'
    if p<0.05:  return f'p={p:.4f}*'
    return f'p={p:.4f}'

# ── Pull data from calibration_store ──────────────────────────────────────
def _pull_cal(care, sev, model_name, group_type):
    return {g:(np.array(yt,dtype=float),np.array(yp,dtype=float))
            for (cl,sl,mn,gt,g,yt,yp) in calibration_store
            if cl==care and sl==sev and mn==model_name and gt==group_type
            and len(np.unique(yt))>1}

def _pool_across_sevs(care, model_name, group_type, group_label):
    yt_all,yp_all=[],[]
    for (cl,sl,mn,gt,g,yt,yp) in calibration_store:
        if cl==care and mn==model_name and gt==group_type and g==group_label:
            yt_all.extend(np.array(yt).tolist())
            yp_all.extend(np.array(yp).tolist())
    if not yt_all: return None,None
    return np.array(yt_all,dtype=float),np.array(yp_all,dtype=float)

def build_fold_aurocs_store(calibration_store, n_folds=5):
    """
    Reconstruct approximate per-fold AUROCs from pooled calibration_store.
    Splits each group's pooled predictions into n_folds non-overlapping chunks
    and computes AUROC per chunk. Used as input to jonckheere_trend_test().

    This is an approximation because calibration_store stores only pooled
    predictions. Chunks preserve realistic n per observation (~fold size).
    Returns: dict keyed by (care, sev, model_name, group) → array of AUROCs
    """
    store = {}
    for (cl, sl, mn, gt, g, yt, yp) in calibration_store:
        yt_a = np.array(yt, dtype=float)
        yp_a = np.array(yp, dtype=float)
        n = len(yt_a)
        chunk = n // n_folds
        if chunk < 5:
            continue
        fold_aucs = []
        for fi in range(n_folds):
            idx = np.arange(fi * chunk, min((fi+1)*chunk, n))
            if len(np.unique(yt_a[idx])) < 2:
                continue
            fold_aucs.append(float(roc_auc_score(yt_a[idx], yp_a[idx])))
        if len(fold_aucs) >= 2:
            store[(cl, sl, mn, g)] = np.array(fold_aucs)
    return store

# Build once after calibration_store is populated (run this cell after cell 14)
# NOTE: fold_aurocs_store is built automatically here.
# This cell must be re-run after cell 14 (the main CV pipeline).
# It is safe to re-run — it just rebuilds the dict.
try:
    fold_aurocs_store = build_fold_aurocs_store(calibration_store)
    print(f'fold_aurocs_store built: {len(fold_aurocs_store)} entries')
except NameError:
    fold_aurocs_store = {}
    print('fold_aurocs_store: calibration_store not yet available — run cell 14 first')
print("fold_aurocs_store builder ready. Run: fold_aurocs_store = build_fold_aurocs_store(calibration_store)")
print("Shared stat helpers loaded.")


In [ ]:

# ══════════════════════════════════════════════════════════════════════════
# COMPREHENSIVE STATISTICAL TESTS — one block per (care, severity)
# Outputs: stat_results dict + printed summary
# ══════════════════════════════════════════════════════════════════════════

stat_results = {}

for care in ['Mouthcare', 'Turning']:
    data_src = mc_data if care == 'Mouthcare' else turn_data

    for sev in SEVERITY_LABELS:
        print(f'\n{"="*65}')
        print(f'  {care} | {sev} Severity')
        print(f'{"="*65}')
        res = {}
        sub_d = data_src[data_src['severity'] == sev].copy()

        # Pull calibration data
        qA  = _pull_cal(care, sev, 'A – All Quartiles', 'quartile')
        qB  = _pull_cal(care, sev, 'B – Q1 Only',       'quartile')
        rA  = _pull_cal(care, sev, 'A – By Race',        'race')
        rB  = _pull_cal(care, sev, 'B – Q1 By Race',     'race')

        # ── 1. SOFA balance ──────────────────────────────────────────────
        print('\n1. SOFA balance across Q1–Q4 (KW; non-sig = well controlled)')
        sofa_grps = [sub_d[sub_d['quartile']==q][SOFA_COL].dropna().values
                     for q in QUARTILES]
        sofa_grps = [g for g in sofa_grps if len(g)>0]
        if len(sofa_grps)>=2:
            kw_s,kw_p = kruskal(*sofa_grps)
            verdict = '✓ balanced' if kw_p>=0.05 else '⚠ imbalanced — check design'
            print(f'   KW H={kw_s:.2f}  {_fmt(kw_p)}  [{verdict}]')
            res['sofa_kw_p'] = kw_p

        # ── 2. Mortality distribution (descriptive only) ─────────────────
        print('\n2. Mortality rates Q1→Q4 (descriptive — no directional hypothesis within severity strata)')
        m_counts = [int(sub_d[sub_d['quartile']==q][MORTALITY].sum()) for q in QUARTILES]
        m_totals = [len(sub_d[sub_d['quartile']==q]) for q in QUARTILES]
        for q,c,t in zip(QUARTILES,m_counts,m_totals):
            print(f'   {q}: {c}/{t} = {c/t*100:.1f}%' if t>0 else f'   {q}: empty')

        # ── 3. Model A → Quartile: pairwise DeLong + permutation JT ─────────
        # Configuration 2: Model A tested on care quartiles.
        # DeLong pairwise (Bonferroni-corrected) + permutation max JT as confirmatory trend test.
        # A significant JT here reveals care-axis non-uniformity that pairwise DeLong may miss
        # when training spans all quartiles (contaminated signal scenario, paper section 2.6.4).
        print('\n3. Model A → Quartile (Config 2: DeLong + permutation max JT — care-quality axis)')
        avail = [q for q in QUARTILES if q in qA]
        if len(avail)>=2:
            pairs=list(combinations(avail,2))
            raw_ps=[]; presults=[]
            for q1,q2 in pairs:
                aA,aB,z,p=delong_independent(qA[q1][0],qA[q1][1],qA[q2][0],qA[q2][1])
                raw_ps.append(p if not np.isnan(p) else 1.0)
                presults.append((q1,q2,aA,aB,z,p))
            rej,padj,_,_=multipletests(raw_ps,method='bonferroni')
            for (q1,q2,aA,aB,z,p),r,pa in zip(presults,rej,padj):
                print(f'   {q1} vs {q2}: {aA:.3f} vs {aB:.3f}  Δ={aA-aB:+.3f}  '
                      f'raw {_fmt(p)}  adj p={pa:.4f}{"*" if r else ""}')
            n_sig=rej.sum()
            print(f'   → {n_sig}/{len(pairs)} pairs sig (Bonferroni)')
            res['modelA_q_n_sig_bonf']=int(n_sig)
            res['modelA_q_delong']=[(q1,q2,round(aA,3),round(aB,3),
                                     round(z,3) if not np.isnan(z) else np.nan,
                                     round(pa,4),bool(r))
                                    for (q1,q2,aA,aB,z,_),r,pa in zip(presults,rej,padj)]

            # ── Permutation max JT for Model A quartile (Config 2) ───────────────
            # Step 1 symmetric trend test: best-ordering permutation JT (not natural-ordering).
            # For Config 2 the paper uses permutation JT, not directional JT.
            # Directional (natural Q1→Q4) JT is reserved for Config 3 only (paper section 2.6.4).
            # A significant result here indicates a care-axis trend even for Model A.
            # Permutation max JT for Config 2 — uses bootstrap AUROC distributions
            # true_fold_aurocs_store gives 1,000 bootstrap AUROCs per quartile
            # (from pooled predictions in calibration_store, same as DeLong)
            jt_obs_A = [np.array(true_fold_aurocs_store.get((care, sev, 'A – All Quartiles', q), []))
                        for q in QUARTILES]
            jt_obs_A = [a for a in jt_obs_A if len(a) >= 2]
            if len(jt_obs_A) >= 3:
                jt_A, p_jt_A = jonckheere_trend_test(jt_obs_A)
                all_fold_aurocs_A = [np.mean(true_fold_aurocs_store.get((care, sev, 'A – All Quartiles', q), [np.nan]))
                                     for q in QUARTILES]
                # Note: if a quartile has no fold data, its mean is nan; spearmanr will
                # return nan rho in that case — interpret with caution if any Q is missing.
                rho_A, p_rho_A = stats.spearmanr([1,2,3,4], all_fold_aurocs_A)
                print(f'   JT(A)={jt_A:.1f}  {_fmt(p_jt_A)}  (monotonic trend check)')
                print(f'   Spearman ρ(A)={rho_A:.3f}  {_fmt(p_rho_A)}')
                res['modelA_q_jt_p'] = p_jt_A
                res['modelA_q_spearman_rho'] = round(rho_A, 3)
                res['modelA_q_spearman_p'] = round(p_rho_A, 4)

        # ── 4. Model B → Quartile: pairwise DeLong + JT + Spearman ─────
        # Goal: show care-quality gradient; JT + Spearman as formal + descriptive trend evidence
        print('\n4. Model B → Quartile (goal: show care-quality gradient across Q1–Q4)')
        avail = [q for q in QUARTILES if q in qB]
        if len(avail)>=2:
            pairs=list(combinations(avail,2))
            raw_ps=[]; presults=[]
            for q1,q2 in pairs:
                aA,aB,z,p=delong_independent(qB[q1][0],qB[q1][1],qB[q2][0],qB[q2][1])
                raw_ps.append(p if not np.isnan(p) else 1.0)
                presults.append((q1,q2,aA,aB,z,p))
            rej,padj,_,_=multipletests(raw_ps,method='bonferroni')
            for (q1,q2,aA,aB,z,p),r,pa in zip(presults,rej,padj):
                marker=' ← (Q1 vs Q4)' if {q1,q2}=={'Q1','Q4'} else ''
                print(f'   {q1} vs {q2}: {aA:.3f} vs {aB:.3f}  Δ={aA-aB:+.3f}  '
                      f'raw {_fmt(p)}  adj p={pa:.4f}{"*" if r else ""}{marker}')
            n_sig=rej.sum()
            print(f'   → {n_sig}/{len(pairs)} pairs sig (Bonferroni)')
            res['modelB_q_n_sig_bonf']=int(n_sig)
            res['modelB_q_delong']=[(q1,q2,round(aA,3),round(aB,3),
                                     round(z,3) if not np.isnan(z) else np.nan,
                                     round(pa,4),bool(r))
                                    for (q1,q2,aA,aB,z,_),r,pa in zip(presults,rej,padj)]
            # Directional JT for Config 3 — uses bootstrap AUROC distributions
            # true_fold_aurocs_store gives 1,000 bootstrap AUROCs per quartile
            # (from pooled predictions in calibration_store, same source as DeLong).
            # Between-group independence holds (patient-disjoint quartiles).
            # Within-group bootstrap correlation -> conservative p-values (safe direction).
            try:
                jt_obs=[np.array(true_fold_aurocs_store.get((care,sev,'B – Q1 Only',q),[]))
                        for q in QUARTILES]
                jt_obs=[a for a in jt_obs if len(a)>=2]
                if len(jt_obs)>=3:
                    jt,p_jt=jonckheere_trend_test(jt_obs)
                    print(f'   JT={jt:.1f}  {_fmt(p_jt)}  (monotonic trend; non-sig OK if pairwise shows Q1>Q4)')
                    res['modelB_q_jt_p']=p_jt
            except: pass
            # Spearman: use bootstrap mean AUROCs from true_fold_aurocs_store
            # (same source as section 3 Config 2, and same as JT inputs — consistent)
            q_aucs=[(i+1, float(np.mean(true_fold_aurocs_store.get((care,sev,'B – Q1 Only',q), [np.nan]))))
                    for i,q in enumerate(QUARTILES)]
            q_aucs=[(rank, auc) for rank,auc in q_aucs if not np.isnan(auc)]
            if len(q_aucs)>=3:
                rho,p_sp=spearmanr([x[0] for x in q_aucs],[x[1] for x in q_aucs])
                print(f'   Spearman ρ={rho:.3f}  {_fmt(p_sp)}')
                res['modelB_q_spearman_rho']=round(rho,3); res['modelB_q_spearman_p']=p_sp
            # Q1 vs Q4 headline (pre-specified) — extract from Bonferroni-corrected list
            if 'Q1' in qB and 'Q4' in qB:
                res['modelB_q1_auroc']=round(float(roc_auc_score(*qB['Q1'])),3)
                res['modelB_q4_auroc']=round(float(roc_auc_score(*qB['Q4'])),3)
                res['modelB_q1q4_delta']=round(res['modelB_q1_auroc']-res['modelB_q4_auroc'],3)
                # Extract Bonferroni-adjusted p for Q1 vs Q4 from the pairwise list computed above
                q1q4_adj = next((pa for (q1,q2,*_,pa,_r) in res.get('modelB_q_delong',[]) if {q1,q2}=={'Q1','Q4'}), None)
                res['modelB_q1q4_p_adj'] = q1q4_adj

        # ── 5. Paired DeLong: Model A vs B at each quartile (supplementary diagnostic) ──
        # NOTE: This test is NOT reported in the paper as a primary or secondary result.
        # It is a supplementary internal check to quantify how much Model B degrades
        # relative to Model A at each quartile on the same patients.
        # The paper's primary inter-model comparison is the Q1→Q4 AUROC gradient within
        # Model B (Config 3), not a paired A-vs-B test.
        # Paired DeLong accounts for the correlation between two scores on the same
        # patients via the full 2×2 covariance matrix; independent DeLong would
        # ignore this correlation and underestimate variance, inflating z.
        print('\n5. Paired DeLong — Model A vs B at each quartile (supplementary diagnostic only)')
        print('   (paired: A and B scored on the same pooled test patients per quartile)')
        for q in QUARTILES:
            if q not in qA or q not in qB: continue
            yt_A,yp_A=qA[q]; yt_B,yp_B=qB[q]
            aA,aB,z,p=delong_paired(yt_A,yp_A,yp_B)
            direction = 'A>B ✓' if aA>aB else 'B>A'
            print(f'   {q}: A={aA:.3f}  B={aB:.3f}  Δ={aA-aB:+.3f}  z={z:.3f}  {_fmt(p)}  [{direction}]')
            res[f'indep_AB_{q}_p']=p
            res[f'indep_AB_{q}_delta']=round(aA-aB,3)

        # ── 6. Model A → Race: show stable (non-sig) ─────────────────────
        print('\n6. Model A → Race (goal: show stability = non-sig)')
        avail_r=[r for r in RACE_LABELS if r in rA]
        if len(avail_r)>=2:
            pairs=list(combinations(avail_r,2))
            raw_ps=[]; presults=[]
            for r1,r2 in pairs:
                aA,aB,z,p=delong_independent(rA[r1][0],rA[r1][1],rA[r2][0],rA[r2][1])
                raw_ps.append(p if not np.isnan(p) else 1.0)
                presults.append((r1,r2,aA,aB,z,p))
            rej,padj,_,_=multipletests(raw_ps,method='bonferroni')
            for (r1,r2,aA,aB,z,p),r,pa in zip(presults,rej,padj):
                print(f'   {r1} vs {r2}: {aA:.3f} vs {aB:.3f}  Δ={aA-aB:+.3f}  '
                      f'raw {_fmt(p)}  adj p={pa:.4f}{"*" if r else ""}')
            n_sig=rej.sum()
            race_aucs=[float(roc_auc_score(*rA[r])) for r in avail_r]
            print(f'   → {n_sig}/{len(pairs)} sig (Bonferroni)  range={max(race_aucs)-min(race_aucs):.3f}')
            # KW on per-group AUROCs (one value per group — honest)
            if len(race_aucs)>=3:
                # With 3-5 values KW = nearly equivalent to comparing ranks; report as effect check
                kw_s,kw_p=kruskal(*[[a] for a in race_aucs]) if False else (np.nan,np.nan)
                # Instead: report range and n_sig as the effect measure
                pass
            res['modelA_r_n_sig_bonf']=int(n_sig)
            res['modelA_r_range']=round(max(race_aucs)-min(race_aucs),3) if race_aucs else np.nan
            # Permutation best-ordering JT for race axis (Config 1)
            # Permutation max JT for race axis (Configs 1 & 4) is run in cell 35
            # using perm_jt_all_orderings with n_perm=1000 — those are the reported results.

        # ── 7. Model B → Race: show stable ───────────────────────────────
        print('\n7. Model B → Race (goal: show stability = non-sig)')
        avail_r=[r for r in RACE_LABELS if r in rB]
        if len(avail_r)>=2:
            pairs=list(combinations(avail_r,2))
            raw_ps=[]; presults=[]
            for r1,r2 in pairs:
                aA,aB,z,p=delong_independent(rB[r1][0],rB[r1][1],rB[r2][0],rB[r2][1])
                raw_ps.append(p if not np.isnan(p) else 1.0)
                presults.append((r1,r2,aA,aB,z,p))
            rej,padj,_,_=multipletests(raw_ps,method='bonferroni')
            for (r1,r2,aA,aB,z,p),r,pa in zip(presults,rej,padj):
                print(f'   {r1} vs {r2}: {aA:.3f} vs {aB:.3f}  Δ={aA-aB:+.3f}  '
                      f'raw {_fmt(p)}  adj p={pa:.4f}{"*" if r else ""}')
            n_sig=rej.sum()
            race_aucs=[float(roc_auc_score(*rB[r])) for r in avail_r]
            print(f'   → {n_sig}/{len(pairs)} sig (Bonferroni)  range={max(race_aucs)-min(race_aucs):.3f}')
            res['modelB_r_n_sig_bonf']=int(n_sig)
            res['modelB_r_range']=round(max(race_aucs)-min(race_aucs),3)
            # Permutation max JT for race axis (Configs 1 & 4) is run in cell 35
            # using perm_jt_all_orderings with n_perm=1000 — those are the reported results.

        # ── 8. Chi-square Race × Quartile — REMOVED ────────────────────────
        # Removed: invoking Simpson's Paradox would reframe the paper as a
        # statistical artefact story. The care-quality argument stands independently.

        # ── 9. Variance decomposition ─────────────────────────────────────
        print('\n9. Variance decomposition — AUROC range across stratifications')
        aucs_by_q = {q:float(roc_auc_score(*qA[q])) for q in QUARTILES if q in qA}
        aucs_by_r = {r:float(roc_auc_score(*rA[r])) for r in RACE_LABELS if r in rA}
        if aucs_by_q:
            rng_q=max(aucs_by_q.values())-min(aucs_by_q.values())
            print(f'   Model A → Quartile  range = {rng_q:.4f}  {aucs_by_q}')
        if aucs_by_r:
            rng_r=max(aucs_by_r.values())-min(aucs_by_r.values())
            print(f'   Model A → Race      range = {rng_r:.4f}  {aucs_by_r}')
        aucs_by_qB={q:float(roc_auc_score(*qB[q])) for q in QUARTILES if q in qB}
        if aucs_by_qB:
            rng_qB=max(aucs_by_qB.values())-min(aucs_by_qB.values())
            print(f'   Model B → Quartile  range = {rng_qB:.4f}  {aucs_by_qB}')
        aucs_by_rB={r:float(roc_auc_score(*rB[r])) for r in RACE_LABELS if r in rB}
        if aucs_by_rB:
            rng_rB=max(aucs_by_rB.values())-min(aucs_by_rB.values())
            print(f'   Model B → Race      range = {rng_rB:.4f}  {aucs_by_rB}')
        res['var_decomp']={'A_q_range':round(rng_q,4) if aucs_by_q else None,
                           'A_r_range':round(rng_r,4) if aucs_by_r else None,
                           'B_q_range':round(rng_qB,4) if aucs_by_qB else None,
                           'B_r_range':round(rng_rB,4) if aucs_by_rB else None}

        # ── 9b. Permutation test — REMOVED ──────────────────────────────────
        # Removed: Model A has no motivated directionality for Q-range vs R-range.
        # Descriptive range comparison (Model B Q_range vs R_range) is reported instead.

        stat_results[(care,sev)] = res

print('\nAll statistical tests complete.')


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  DIAGNOSTIC / DEVELOPMENT CELL — not the paper-reported test    ║
# ║  Standalone permutation JT for Low and Medium only.             ║
# ║  Cell 35 is the primary: runs all 4 configs × all strata        ║
# ║  × both care types and generates the reported table.            ║
# ║  Run this cell only to quick-check individual strata.           ║
# ╚══════════════════════════════════════════════════════════════════╝
# ═══════════════════════════════════════════════════════════════════════════════
# STANDALONE TEST — Permutation JT for all 4 configs, Low and Medium only
# Run this cell independently to check before committing to full pipeline
# Two methods: all_orderings (rigorous, ~18 min) or best_ordering (~45 sec)
# ═══════════════════════════════════════════════════════════════════════════════
from itertools import permutations as _iperms
import numpy as np
from scipy import stats

def jt_stat(groups_obs):
    k = len(groups_obs); jt = 0
    for i in range(k-1):
        for j in range(i+1, k):
            u, _ = stats.mannwhitneyu(groups_obs[i], groups_obs[j], alternative='greater')
            jt += u
    return jt

def perm_jt_all_orderings(groups_obs, n_perm=1000, seed=42):
    """Max JT across ALL orderings vs permutation null. Most rigorous."""
    rng = np.random.default_rng(seed)
    k = len(groups_obs)
    all_ords = list(_iperms(range(k)))
    # Observed max JT
    obs_max = max(jt_stat([groups_obs[i] for i in perm]) for perm in all_ords)
    # Null
    all_obs = np.concatenate(groups_obs)
    ns = [len(g) for g in groups_obs]
    null_maxes = []
    for _ in range(n_perm):
        sh = rng.permutation(all_obs)
        pg, idx = [], 0
        for n in ns:
            pg.append(sh[idx:idx+n]); idx += n
        null_maxes.append(max(jt_stat([pg[i] for i in perm]) for perm in all_ords))
    return obs_max, np.mean(np.array(null_maxes) >= obs_max)

def perm_jt_best_ordering(groups_obs, n_perm=1000, seed=42):
    """JT on best observed ordering vs permutation null. Fast (~45s total)."""
    rng = np.random.default_rng(seed)
    order = np.argsort([np.mean(g) for g in groups_obs])[::-1]
    ordered = [groups_obs[i] for i in order]
    jt_obs = jt_stat(ordered)
    all_obs = np.concatenate(groups_obs)
    ns = [len(g) for g in groups_obs]
    null_jts = []
    for _ in range(n_perm):
        sh = rng.permutation(all_obs)
        pg, idx = [], 0
        for n in ns:
            pg.append(sh[idx:idx+n]); idx += n
        pg_s = [pg[i] for i in np.argsort([np.mean(g) for g in pg])[::-1]]
        null_jts.append(jt_stat(pg_s))
    return jt_obs, np.mean(np.array(null_jts) >= jt_obs)

# ── Choose method ──────────────────────────────────────────────────────────────
METHOD = 'all'   # 'all' = all possible group orderings (~18 min) — used for paper results (cell 35)
                  # 'best' = best-observed ordering only (~45 sec) — faster diagnostic
                  # Paper section 2.6.4 describes the best-ordering variant for the symmetric JT;
                  # cell 35 uses all_orderings as the most rigorous implementation for reported results.
N_PERM = 1000

perm_fn = perm_jt_all_orderings if METHOD == 'all' else perm_jt_best_ordering
print(f'Method: {METHOD}_ordering  |  n_perm={N_PERM}')
print('='*70)

configs = [
    ('Model A', 'A – All Quartiles', 'A – By Race',    'quartile', 'race'),
    ('Model B', 'B – Q1 Only',       'B – Q1 By Race', 'quartile', 'race'),
]

for care in ['Mouthcare', 'Turning']:
    for sev in ['Low', 'Medium']:
        print(f'\n--- {care} | {sev} ---')
        for model_label, q_key, r_key, _, _ in configs:

            # ── Quartile axis ────────────────────────────────────────────────
            q_obs = [np.array(fold_aurocs_store.get((care, sev, q_key, q), []))
                     for q in QUARTILES]
            q_obs = [a for a in q_obs if len(a) >= 2]
            if len(q_obs) >= 3:
                jt, p = perm_fn(q_obs, n_perm=N_PERM)
                sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
                print(f'  {model_label} → Quartile:  p={p:.3f} {sig}  (expect sig for B, maybe A)')

            # ── Race axis (excl Other) ────────────────────────────────────────
            avail_r = [r for r in RACE_LABELS
                       if r != 'Other'
                       and len(fold_aurocs_store.get((care, sev, r_key, r), [])) >= 2]
            r_obs = [np.array(fold_aurocs_store.get((care, sev, r_key, r), []))
                     for r in avail_r]
            r_obs = [a for a in r_obs if len(a) >= 2]
            if len(r_obs) >= 3:
                jt, p = perm_fn(r_obs, n_perm=N_PERM)
                sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
                print(f'  {model_label} → Race:      p={p:.3f} {sig}  (expect ns)')

print('\nDone.')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  DIAGNOSTIC ONLY — supplementary check for High severity        ║
# ║  High severity results are expected non-significant (attenuated ║
# ║  care signal). This confirms that without computing it in        ║
# ║  cell 35's main loop. Not a separate paper result.              ║
# ╚══════════════════════════════════════════════════════════════════╝
# ═══════════════════════════════════════════════════════════════════════════════
# Quick check — High severity permutation JT (all orderings, n_perm=1000)
# ═══════════════════════════════════════════════════════════════════════════════

print('Method: all_ordering  |  n_perm=1000')
print('='*70)

configs = [
    ('Model A', 'A – All Quartiles', 'A – By Race'),
    ('Model B', 'B – Q1 Only',       'B – Q1 By Race'),
]

for care in ['Mouthcare', 'Turning']:
    print(f'\n--- {care} | High ---')
    for model_label, q_key, r_key in configs:

        # Quartile axis
        q_obs = [np.array(fold_aurocs_store.get((care, 'High', q_key, q), []))
                 for q in QUARTILES]
        q_obs = [a for a in q_obs if len(a) >= 2]
        if len(q_obs) >= 3:
            jt, p = perm_jt_all_orderings(q_obs, n_perm=1000)
            sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
            print(f'  {model_label} → Quartile:  p={p:.3f} {sig}')

        # Race axis (excl Other)
        avail_r = [r for r in RACE_LABELS
                   if r != 'Other'
                   and len(fold_aurocs_store.get((care, 'High', r_key, r), [])) >= 2]
        r_obs = [np.array(fold_aurocs_store.get((care, 'High', r_key, r), []))
                 for r in avail_r]
        r_obs = [a for a in r_obs if len(a) >= 2]
        if len(r_obs) >= 3:
            jt, p = perm_jt_all_orderings(r_obs, n_perm=1000)
            sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
            print(f'  {model_label} → Race:      p={p:.3f} {sig}')

print('\nDone.')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  SUPERSEDED BY CELL 35 — manual backup with hardcoded p-values  ║
# ║  This cell uses hardcoded p-values entered by hand.             ║
# ║  Cell 35 COMPUTES p-values directly from data and saves the     ║
# ║  same file (fig_perm_jt_table.png). Both cells save to the      ║
# ║  same path — run ONLY cell 35 for reported results.             ║
# ║  This cell is kept as a reference for the expected values.      ║
# ╚══════════════════════════════════════════════════════════════════╝
# ═══════════════════════════════════════════════════════════════════════════════
# Permutation JT Summary Table — manual input, ordered by configuration
# ═══════════════════════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

results = [
    #  Care         Sev      C1:A-R   C2:A-Q   C3:B-Q   C4:B-R
    ('Mouthcare', 'Low',    0.573,   0.031,   0.001,   0.857),
    ('Mouthcare', 'Medium', 0.705,   0.023,   0.001,   0.362),
    ('Turning',   'Low',    0.799,   0.314,   0.033,   0.136),
    ('Turning',   'Medium', 0.130,   0.676,   0.353,   0.512),
]

def fmt_p(p):
    if p < 0.001: return '<0.001***'
    if p < 0.01:  return f'{p:.3f}**'
    if p < 0.05:  return f'{p:.3f}*'
    return f'{p:.3f}'

def cell_col(p):
    if p < 0.05: return '#FDEDEC'
    return '#EAFAF1'

col_labels = ['Care', 'Severity',
              'Config 1\nModel A → Race',
              'Config 2\nModel A → Quartile',
              'Config 3\nModel B → Quartile',
              'Config 4\nModel B → Race']

table_data = [[c, s, fmt_p(c1), fmt_p(c2), fmt_p(c3), fmt_p(c4)]
              for c, s, c1, c2, c3, c4 in results]

fig, ax = plt.subplots(figsize=(14, 4))
ax.axis('off')

tbl = ax.table(cellText=table_data, colLabels=col_labels,
               loc='center', cellLoc='center')
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.auto_set_column_width(col=list(range(len(col_labels))))
tbl.scale(1, 2.5)

# Header — Config 1 and 4 (race) blue, Config 2 and 3 (quartile) red
for j in range(len(col_labels)):
    tbl[(0, j)].set_facecolor('#1F3864')
    tbl[(0, j)].set_text_props(color='white', fontweight='bold')
for j in [3, 4]:  # Config 2 and 3 — quartile
    tbl[(0, j)].set_facecolor('#922B21')
for j in [2, 5]:  # Config 1 and 4 — race
    tbl[(0, j)].set_facecolor('#1A5276')

# Cell colouring
for i, (c, s, c1, c2, c3, c4) in enumerate(results, start=1):
    tbl[(i, 0)].set_facecolor('#F8F9FA')
    tbl[(i, 1)].set_facecolor('#F8F9FA')
    for col_idx, p in [(2,c1),(3,c2),(4,c3),(5,c4)]:
        tbl[(i, col_idx)].set_facecolor(cell_col(p))
        if p < 0.05:
            tbl[(i, col_idx)].set_text_props(fontweight='bold')

fig.suptitle(
    'Permutation-Based Maximum Jonckheere-Terpstra Test — Primary Strata (Low and Medium Severity)\n'
    'All possible group orderings, n_perm=1000, race groups excluding Other\n'
    '* p<0.05  ** p<0.01  *** p<0.001',
    fontsize=10, fontweight='bold', y=1.08
)

legend_patches = [
    mpatches.Patch(color='#FDEDEC', label='Significant (p<0.05)'),
    mpatches.Patch(color='#EAFAF1', label='Non-significant'),
]
fig.legend(handles=legend_patches, loc='lower center',
           bbox_to_anchor=(0.5, -0.05), ncol=2, fontsize=9)

plt.tight_layout()
path = os.path.join(OUTPUT_FOLDER, 'fig_perm_jt_table.png')
# [figure removed] plt.savefig(path, bbox_inches='tight', dpi=150)
# [figure removed] plt.show()
print(f'Saved: {path}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Permutation-Based Maximum JT Test — All 4 Configs, All Orderings
# Symmetric trend test: tests whether ANY monotonic ordering of groups is significant.
# Applied to all 4 configurations (race and quartile axes, Models A and B).
# Saves: fig_perm_jt_table.png = Supplementary Table 3
# n_perm=1000, race groups excluding Other (primary), all orderings tested.
# NOTE: This is the permutation max JT only. The directional JT (Q1→Q4) is in cell 31.
# ═══════════════════════════════════════════════════════════════════════════════
from itertools import permutations as _iperms
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from scipy import stats

def jt_stat(groups_obs):
    k = len(groups_obs); jt = 0
    for i in range(k-1):
        for j in range(i+1, k):
            u, _ = stats.mannwhitneyu(groups_obs[i], groups_obs[j], alternative='greater')
            jt += u
    return jt

def perm_jt_all_orderings(groups_obs, n_perm=1000, seed=42):
    """
    Max JT across ALL possible group orderings vs permutation null.
    Requires no pre-specified ordering — valid for both quartile and race axes.
    Non-sig = no monotonic trend exists in any direction under any ordering.
    Reference: Hollander, Wolfe & Chicken (2013) Nonparametric Statistical Methods, 3rd ed.
    """
    rng = np.random.default_rng(seed)
    k = len(groups_obs)
    all_ords = list(_iperms(range(k)))
    obs_max = max(jt_stat([groups_obs[i] for i in perm]) for perm in all_ords)
    all_obs = np.concatenate(groups_obs)
    ns = [len(g) for g in groups_obs]
    null_maxes = []
    for _ in range(n_perm):
        sh = rng.permutation(all_obs)
        pg, idx = [], 0
        for n in ns:
            pg.append(sh[idx:idx+n]); idx += n
        null_maxes.append(max(jt_stat([pg[i] for i in perm]) for perm in all_ords))
    return obs_max, np.mean(np.array(null_maxes) >= obs_max)

# ── Run for Low and Medium only (primary strata) ──────────────────────────────
print('Permutation JT — all orderings, n_perm=1000, primary strata only')
print('='*70)

configs = [
    ('Model A', 'A – All Quartiles', 'A – By Race'),
    ('Model B', 'B – Q1 Only',       'B – Q1 By Race'),
]

perm_jt_results = {}

for care in ['Mouthcare', 'Turning']:
    for sev in ['Low', 'Medium']:
        print(f'\n--- {care} | {sev} ---')
        for model_label, q_key, r_key in configs:

            # Quartile axis
            q_obs = [np.array(fold_aurocs_store.get((care, sev, q_key, q), []))
                     for q in QUARTILES]
            q_obs = [a for a in q_obs if len(a) >= 2]
            if len(q_obs) >= 3:
                jt, p = perm_jt_all_orderings(q_obs, n_perm=1000)
                sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
                print(f'  {model_label} → Quartile:  p={p:.3f} {sig}')
                perm_jt_results[(care, sev, model_label, 'Quartile')] = p

            # Race axis (excl Other)
            avail_r = [r for r in RACE_LABELS
                       if r != 'Other'
                       and len(fold_aurocs_store.get((care, sev, r_key, r), [])) >= 2]
            r_obs = [np.array(fold_aurocs_store.get((care, sev, r_key, r), []))
                     for r in avail_r]
            r_obs = [a for a in r_obs if len(a) >= 2]
            if len(r_obs) >= 3:
                jt, p = perm_jt_all_orderings(r_obs, n_perm=1000)
                sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
                print(f'  {model_label} → Race:      p={p:.3f} {sig}')
                perm_jt_results[(care, sev, model_label, 'Race')] = p

print('\nAll done.')

# ── Summary Table ──────────────────────────────────────────────────────────────
def fmt_p(p):
    if p is None: return '—'
    if p < 0.001: return '<0.001***'
    if p < 0.01:  return f'{p:.3f}**'
    if p < 0.05:  return f'{p:.3f}*'
    return f'{p:.3f}'

def cell_col(p):
    if p is None: return '#F5F5F5'
    if p < 0.05:  return '#FDEDEC'
    return '#EAFAF1'

strata_order = [
    ('Mouthcare', 'Low'),
    ('Mouthcare', 'Medium'),
    ('Turning',   'Low'),
    ('Turning',   'Medium'),
]

col_labels = ['Care', 'Severity',
              'Model A\nQuartile', 'Model A\nRace',
              'Model B\nQuartile', 'Model B\nRace']

table_data = []
cell_ps = []
for care, sev in strata_order:
    aq = perm_jt_results.get((care, sev, 'Model A', 'Quartile'))
    ar = perm_jt_results.get((care, sev, 'Model A', 'Race'))
    bq = perm_jt_results.get((care, sev, 'Model B', 'Quartile'))
    br = perm_jt_results.get((care, sev, 'Model B', 'Race'))
    table_data.append([care, sev, fmt_p(aq), fmt_p(ar), fmt_p(bq), fmt_p(br)])
    cell_ps.append((aq, ar, bq, br))

fig, ax = plt.subplots(figsize=(13, 4))
ax.axis('off')

tbl = ax.table(cellText=table_data, colLabels=col_labels,
               loc='center', cellLoc='center')
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.auto_set_column_width(col=list(range(len(col_labels))))
tbl.scale(1, 2.5)

# Header styling
for j in range(len(col_labels)):
    tbl[(0, j)].set_facecolor('#1F3864')
    tbl[(0, j)].set_text_props(color='white', fontweight='bold')
for j in [2, 4]:
    tbl[(0, j)].set_facecolor('#922B21')
for j in [3, 5]:
    tbl[(0, j)].set_facecolor('#1A5276')

# Cell colouring
for i, (aq, ar, bq, br) in enumerate(cell_ps, start=1):
    tbl[(i, 0)].set_facecolor('#F8F9FA')
    tbl[(i, 1)].set_facecolor('#F8F9FA')
    for col_idx, p in [(2,aq),(3,ar),(4,bq),(5,br)]:
        tbl[(i, col_idx)].set_facecolor(cell_col(p))
        if p is not None and p < 0.05:
            tbl[(i, col_idx)].set_text_props(fontweight='bold')

fig.suptitle(
    'Permutation-Based Maximum Jonckheere-Terpstra Test — Primary Strata (Low and Medium Severity)\n'
    'All possible group orderings, n_perm=1000, race groups excluding Other\n'
    '* p<0.05  ** p<0.01  *** p<0.001',
    fontsize=10, fontweight='bold', y=1.08
)

legend_patches = [
    mpatches.Patch(color='#FDEDEC', label='Significant (p<0.05)'),
    mpatches.Patch(color='#EAFAF1', label='Non-significant'),
]
fig.legend(handles=legend_patches, loc='lower center',
           bbox_to_anchor=(0.5, -0.05), ncol=2, fontsize=9)

plt.tight_layout()
path = os.path.join(OUTPUT_FOLDER, 'fig_perm_jt_table.png')
plt.savefig(path, bbox_inches='tight', dpi=150)
plt.show()
print(f'Saved: {path}')

## 10c. ΔAUROC — Deviation from Full-Cohort Baseline

ΔAUROC = group AUROC − full-cohort AUROC for the corresponding model and severity stratum. Positive ΔAUROC indicates above-average performance (Q1 patients whose mortality patterns are well captured); negative ΔAUROC indicates below-average performance (Q4 patients whose mortality is not well predicted). Race panels are expected to cluster near zero. Bootstrap 95% CIs computed on group predictions with full-cohort AUROC treated as a fixed reference.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  REDUNDANT WITH CELL 41 — plot_delta_auroc_panel is DEFINED     ║
# ║  here (needed) and also called here AND in cell 41.             ║
# ║  Cell 41's plot_combined_severity() calls plot_delta_auroc_panel ║
# ║  again for all (care × severity) combinations, overwriting the  ║
# ║  same fig_delta_auroc_*.png files.                              ║
# ║  → Run cell 37 to define the function (required by cell 41).   ║
# ║  → Skip the execution block at the bottom if running cell 41.  ║
# ║  → Or run both — the second run simply overwrites the files.   ║
# ╚══════════════════════════════════════════════════════════════════╝
# ══════════════════════════════════════════════════════════════════════════
# ΔAUROC PLOTS — deviation from full-cohort baseline
# ══════════════════════════════════════════════════════════════════════════

def compute_delta_auroc(group_yt, group_yp, full_yt, full_yp, n_boot=1000, seed=42):
    rng   = np.random.default_rng(seed)
    g_yt  = np.array(group_yt, dtype=float)
    g_yp  = np.array(group_yp, dtype=float)
    f_yt  = np.array(full_yt,  dtype=float)
    f_yp  = np.array(full_yp,  dtype=float)
    if len(np.unique(g_yt)) < 2 or len(np.unique(f_yt)) < 2:
        return np.nan, np.nan, np.nan, np.nan, np.nan
    auc_g = float(roc_auc_score(g_yt, g_yp))
    auc_f = float(roc_auc_score(f_yt, f_yp))
    delta = auc_g - auc_f
    idx_g = np.arange(len(g_yt))
    boot_aucs = []
    for _ in range(n_boot):
        bg = rng.choice(idx_g, size=len(idx_g), replace=True)
        if len(np.unique(g_yt[bg])) < 2:
            continue
        boot_aucs.append(roc_auc_score(g_yt[bg], g_yp[bg]))
    if not boot_aucs:
        return delta, np.nan, np.nan, auc_g, auc_f
    ci_lo = float(np.percentile(boot_aucs,  2.5)) - auc_f
    ci_hi = float(np.percentile(boot_aucs, 97.5)) - auc_f
    return delta, ci_lo, ci_hi, auc_g, auc_f


def _get_full_pool(care, sev, model_name):
    yt_all, yp_all = [], []
    for (cl, sl, mn, gt, g, yt, yp) in calibration_store:
        if cl == care and sl == sev and mn == model_name:
            yt_all.extend(np.array(yt).tolist())
            yp_all.extend(np.array(yp).tolist())
    if not yt_all: return None, None
    return np.array(yt_all, dtype=float), np.array(yp_all, dtype=float)


def plot_delta_auroc_panel(care, sev, save_path):
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    fig.patch.set_facecolor('#FAFAFA')

    SEV_COL  = {'Low': '#1E8449', 'Medium': '#B7770D', 'High': '#922B21'}
    SOFA_STR = {'Low': '0–6', 'Medium': '7–11', 'High': '≥12'}

    specs = [
        (0, 0, 'A – By Race',       'race',     RACE_LABELS, RACE_COLOURS,
         'Model A → Race  (ΔAUROC vs full cohort)',     'pool'),
        (0, 1, 'A – All Quartiles', 'quartile', QUARTILES,   Q_COLOURS,
         'Model A → Quartile  (ΔAUROC vs full cohort)', 'pool'),
        (1, 0, 'B – Q1 By Race',    'race',     RACE_LABELS, RACE_COLOURS,
         'Model B → Race  (ΔAUROC vs full cohort)',     'pool'),
        (1, 1, 'B – Q1 Only',       'quartile', QUARTILES,   Q_COLOURS,
         'Model B → Quartile  (ΔAUROC vs full cohort)', 'pool'),
    ]

    for row_i, col_i, model_name, axis_type, groups, colors, title, ref_type in specs:
        ax = axes[row_i, col_i]

        data_store = _pull_cal(care, sev, model_name, axis_type)

        base_model_name = 'A – All Quartiles' if 'A –' in model_name else 'B – Q1 Only'
        f_yt, f_yp = _get_full_pool(care, sev, base_model_name)
        ref_label  = 'Pooled full-cohort AUROC'

        if f_yt is None:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center',
                    transform=ax.transAxes, color='grey')
            ax.set_title(title, fontsize=9, fontweight='bold')
            continue

        deltas, ci_los, ci_his, labels, cols = [], [], [], [], []
        for g in groups:
            if g not in data_store: continue
            g_yt, g_yp = data_store[g]
            d, lo, hi, auc_g, auc_f = compute_delta_auroc(g_yt, g_yp, f_yt, f_yp)
            if np.isnan(d): continue
            deltas.append(d); ci_los.append(lo); ci_his.append(hi)
            labels.append(g); cols.append(colors.get(g, '#888888'))

        if not deltas:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center',
                    transform=ax.transAxes, color='grey')
            continue

        x      = np.arange(len(labels))
        deltas = np.array(deltas)
        ci_los = np.array(ci_los)
        ci_his = np.array(ci_his)

        # ── ±0.02 neutral band (no legend) ───────────────────────────────
        ax.fill_between([-0.5, len(labels) - 0.5],
                        [-0.02, -0.02], [0.02, 0.02],
                        alpha=0.08, color='grey', zorder=0)

        # ── Zero baseline (no legend) ─────────────────────────────────────
        ax.axhline(0, color='black', lw=1.5, zorder=5)

        # ── Muted bars ────────────────────────────────────────────────────
        bar_cols = [c if d < 0 else '#2874A6' for c, d in zip(cols, deltas)]
        ax.bar(x, deltas, color=bar_cols, alpha=0.55, width=0.5, zorder=2)

        # ── Error bars ────────────────────────────────────────────────────
        yerr = np.array([np.clip(deltas - ci_los, 0, 2),
                         np.clip(ci_his - deltas, 0, 2)])
        for xi_eb, (d_eb, ye_lo, ye_hi, col_eb) in enumerate(
                zip(deltas, yerr[0], yerr[1], cols)):
            ax.errorbar(xi_eb, d_eb, yerr=[[ye_lo], [ye_hi]], fmt='none',
                        ecolor=col_eb, capsize=5, capthick=2,
                        lw=1.8, zorder=3)

        # ── Scatter dots ──────────────────────────────────────────────────
        ax.scatter(x, deltas, color=cols, s=80, zorder=6,
                   edgecolors='white', linewidths=1.2)

        # ── Connecting line for quartile panels only ──────────────────────
        if axis_type == 'quartile' and len(x) >= 2:
            valid = [(xi, d) for xi, d in enumerate(deltas)
                     if not np.isnan(d)]
            if len(valid) >= 2:
                xv, yv = zip(*valid)
                line_col = ('#2874A6' if 'All Quartiles' in model_name
                            else '#C0392B')
                ax.plot(xv, yv, color=line_col, lw=1.8,
                        ls='-', alpha=0.75, zorder=4)

        # ── Value labels: Δ+0.xxx \n n=XXX ───────────────────────────────
        for xi, (d, lo, hi, lbl) in enumerate(
                zip(deltas, ci_los, ci_his, labels)):
            n_grp  = len(data_store[lbl][0]) if lbl in data_store else 0
            offset = 8 if d >= 0 else -18
            ax.annotate(f'Δ{d:+.3f}\nn={n_grp}',
                        (xi, hi if d >= 0 else lo),
                        textcoords='offset points', xytext=(0, offset),
                        ha='center', fontsize=7, fontweight='bold',
                        color=cols[xi], linespacing=1.2)

        # ── Range badge — bottom-right to avoid overlapping CI bars ───────
        if axis_type == 'race':
            mask    = np.array([l not in ('Other', 'Others', 'Other/Unknown')
                                for l in labels])
            rng_val = (deltas[mask].max() - deltas[mask].min()
                       if mask.sum() >= 2
                       else deltas.max() - deltas.min())
        else:
            rng_val = deltas.max() - deltas.min()
        ax.text(0.97, 0.03, f'Range = {rng_val:.3f}',
                transform=ax.transAxes, ha='right', va='bottom',
                fontsize=8.5, fontweight='bold', color='#333333',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                          edgecolor='#AAAAAA', alpha=0.9))

        # ── Axis formatting ───────────────────────────────────────────────
        ax.set_xticks(x)
        if axis_type == 'quartile':
            ax.set_xticklabels(
                ['Q1\n(most freq)', 'Q2', 'Q3', 'Q4\n(least freq)'],
                fontsize=9)
        else:
            ax.set_xticklabels(labels, fontsize=8, rotation=15, ha='right')
        ax.set_ylabel('ΔAUROC  (group AUROC − reference)', fontsize=9)
        ax.set_xlabel('Race Group' if axis_type == 'race'
                      else 'Care Quartile', fontsize=9)

        # Dynamic ylim — enough room for CI bars and value labels
        valid_ci = np.concatenate([
            ci_his[~np.isnan(ci_his)],
            ci_los[~np.isnan(ci_los)]
        ])
        y_abs = max(0.15, float(np.abs(valid_ci).max()) + 0.08) \
                if len(valid_ci) > 0 else 0.20
        ax.set_ylim(-y_abs, y_abs)

        ax.set_facecolor('#F8F9FA')
        ax.grid(axis='y', alpha=0.3, ls='--')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.set_title(title, fontweight='bold', fontsize=10)

    fig.suptitle(
        f'ΔAUROC — Deviation from Full-Cohort Baseline\n'
        f'{care}  |  {sev} Severity (SOFA {SOFA_STR[sev]})\n'
        f'All panels share the same zero line (pooled full-cohort AUROC) — '
        f'enabling direct comparison between race axis (flat) and quartile axis (spread)\n'
        f'Race panels flat near 0 = apparent fairness  |  '
        f'Quartile panels spread = real disparity hidden by race-based audit  |  '
        f'Q1 > 0 = best-care patients above average; Q4 < 0 = worst-care below average',
        fontsize=11, fontweight='bold', y=1.03,
        color=SEV_COL.get(sev, '#333333'))
    plt.tight_layout()
    # [figure removed — cell 41 generates same files] plt.savefig(save_path, bbox_inches='tight', dpi=150)
    # [figure removed — cell 41 generates same files] plt.show()
    print(f'Saved: {save_path}')


print('Generating ΔAUROC panels...')
for care in ['Mouthcare', 'Turning']:
    for sev in SEVERITY_LABELS:
        plot_delta_auroc_panel(
            care, sev,
            os.path.join(OUTPUT_FOLDER,
                         f'fig_delta_auroc_{care.lower()}_{sev.lower()}.png'))
print('Done.')

## 10e. Statistical Summary Table v1 (Superseded by Cell 71)

*Generates `fig_stat_summary_full.png` — organised by test type. Superseded by cell 71 which organises by paper claim (Sections A/B/C/D). Cell 71 is the version to use for submission.*

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  SUPERSEDED BY CELL 71 (publication statistical summary)        ║
# ║  This is stat summary v1 — one row per (care × severity),       ║
# ║  columns organised by test type.                                ║
# ║  Cell 71 reorganises the same data by paper CLAIM (Sections     ║
# ║  A/B/C/D) and is the version to submit. Saves to a different    ║
# ║  file (fig_stat_summary_full.png vs fig_pub_stat_summary.png).  ║
# ║  Keep this cell — it reads from stat_results which is built     ║
# ║  in cell 31 and also used by cell 71.                           ║
# ╚══════════════════════════════════════════════════════════════════╝

# ══════════════════════════════════════════════════════════════════════════
# PUBLICATION-READY STATISTICAL SUMMARY TABLE
# One row per (care, severity) — columns:
#   SOFA balance, Mortality trend, Model A/B quartile DeLong, JT,
#   Spearman, Race DeLong, Simpson's chi2, Variance decomposition
# ══════════════════════════════════════════════════════════════════════════

pub_rows = []
for care in ['Mouthcare', 'Turning']:
    for sev in SEVERITY_LABELS:
        res = stat_results.get((care, sev), {})
        qA = _pull_cal(care, sev, 'A – All Quartiles', 'quartile')
        qB = _pull_cal(care, sev, 'B – Q1 Only',       'quartile')
        rA = _pull_cal(care, sev, 'A – By Race',        'race')
        rB = _pull_cal(care, sev, 'B – Q1 By Race',     'race')

        def _auroc(d, g):
            return round(float(roc_auc_score(*d[g])), 3) if g in d else None

        row = {
            'Care': care, 'Severity': sev,
            # --- balance & mortality ---
            'SOFA balance KW p': _fmt(res.get('sofa_kw_p')),
            # --- Model A quartile ---
            'A: Q1 AUROC': _auroc(qA,'Q1'),
            'A: Q4 AUROC': _auroc(qA,'Q4'),
            'A: Q1–Q4 Δ': round(_auroc(qA,'Q1')-_auroc(qA,'Q4'),3)
                           if _auroc(qA,'Q1') and _auroc(qA,'Q4') else None,
            'A: Q-pairs sig (Bonf)': f"{res.get('modelA_q_n_sig_bonf','—')}/6",
            # --- Model B quartile ---
            'B: Q1 AUROC': res.get('modelB_q1_auroc'),
            'B: Q4 AUROC': res.get('modelB_q4_auroc'),
            'B: Q1–Q4 Δ': res.get('modelB_q1q4_delta'),
            'B: Q-pairs sig (Bonf)': f"{res.get('modelB_q_n_sig_bonf','—')}/6",
            'B: JT p': _fmt(res.get('modelB_q_jt_p')),
            'B: Spearman ρ': res.get('modelB_q_spearman_rho'),
            # --- Race stability ---
            'A→Race: sig pairs (Bonf)': f"{res.get('modelA_r_n_sig_bonf','—')}/{len(list(combinations([r for r in RACE_LABELS if r in rA],2)))}",
            'A→Race range': res.get('modelA_r_range'),
            'B→Race: sig pairs (Bonf)': f"{res.get('modelB_r_n_sig_bonf','—')}/{len(list(combinations([r for r in RACE_LABELS if r in rB],2)))}",
            'B→Race range': res.get('modelB_r_range'),
            # --- Variance decomposition (descriptive) ---
            'Q range (B)': res.get('var_decomp',{}).get('B_q_range'),
            'R range (B)': res.get('modelB_r_range'),
        }
        pub_rows.append(row)

pub_df = pd.DataFrame(pub_rows)
print('Statistical Summary Table')
print('='*120)
pd.set_option('display.max_columns', None); pd.set_option('display.width', 300)
print(pub_df.to_string(index=False))

# ── Matplotlib figure ─────────────────────────────────────────────────────
DISP_COLS = [c for c in pub_df.columns]
fig_h = max(5, 0.55*len(pub_df)+3)
fig, ax = plt.subplots(figsize=(28, fig_h))
ax.axis('off')
tbl = ax.table(
    cellText=[[str(v) if v is not None else '—' for v in pub_df.iloc[i]]
              for i in range(len(pub_df))],
    colLabels=DISP_COLS,
    loc='center', cellLoc='center')
tbl.auto_set_font_size(False); tbl.set_fontsize(8)
tbl.auto_set_column_width(col=list(range(len(DISP_COLS))))
for j in range(len(DISP_COLS)):
    tbl[(0,j)].set_facecolor('#1F3864')
    tbl[(0,j)].set_text_props(color='white',fontweight='bold')
for i in range(1, len(pub_df)+1):
    bg = '#EBF5FB' if pub_df.iloc[i-1]['Care']=='Mouthcare' else '#FDFEFE'
    for j in range(len(DISP_COLS)):
        tbl[(i,j)].set_facecolor(bg)
fig.suptitle(
    'Statistical Summary\n'
    'SOFA balance (non-sig=good) | '
    'Model B Q-axis gradient (sig=good) | Race-axis (non-sig=good) | '
    'Variance decomposition (B Q_range vs R_range — descriptive)',
    fontsize=10, fontweight='bold', y=1.04)
plt.tight_layout()
# [unused figure removed] path = os.path.join(OUTPUT_FOLDER, 'fig_stat_summary_full.png')
# [unused figure removed] plt.savefig(path, bbox_inches='tight', dpi=150)
# [unused figure removed] plt.show(); print(f'Saved: {path}')


## 10f. Enhanced 2×2 AUROC Panels — **Main Paper Figures 1–4**

For each care phenotype × severity combination, produces two figures:
- `fig_enhanced2x2_*.png` — 2×2 AUROC panel with DeLong uniformity badge (top-left), directional JT degradation test (bottom-right), and Spearman ρ
- `fig_delta_auroc_*.png` — ΔAUROC deviation panels for all four configurations

DeLong badge = uniformity test (stable/not stable). Directional JT = degradation test (Q1→Q4 trend). Bootstrap 95% CIs used for both axes.

In [ ]:

# ══════════════════════════════════════════════════════════════════════════
# ENHANCED 2×2 AUROC PANELS — with stat annotations per panel
#
# Each severity produces a 2×2 figure (fig_enhanced2x2_*.png):
#   (row 0, col 0): Model A → Race       — scatter dots, bootstrap CI, DeLong stability badge
#   (row 0, col 1): Model A → Quartile   — line, bootstrap CI, DeLong stability badge, Spearman ρ
#   (row 1, col 0): Model B → Race       — scatter dots, bootstrap CI, DeLong stability badge
#   (row 1, col 1): Model B → Quartile   — line, bootstrap CI, DeLong stability badge,
#                                             directional JT (bottom-right), Spearman ρ
#
# Stat annotations:
#   - DeLong stability badge (top-left): green = 0/N pairs sig (✔ no differences),
#                                        red = X/N pairs sig (⚠ differences detected)
#   - Directional JT p-value (bottom-right, Model B quartile only): tests Q1→Q4 degradation trend
#   - Spearman ρ (bottom-left, quartile panels): descriptive monotonicity summary
#   - Q1 vs Q4 bracket REMOVED — DeLong tests uniformity, not degradation direction
# ══════════════════════════════════════════════════════════════════════════

SOFA_STR_E = {'Low':'0–6','Medium':'7–11','High':'≥12'}
SEV_COL_E  = {'Low':'#1E8449','Medium':'#B7770D','High':'#922B21'}

def _bonf_pairs_all(data_dict, groups, group_type='quartile'):
    """Run all pairwise DeLong with Bonferroni. Returns list of (g1,g2,aA,aB,z,p_raw,p_adj,sig)."""
    avail=[g for g in groups if g in data_dict and len(np.unique(data_dict[g][0]))>1]
    if len(avail)<2: return []
    pairs=list(combinations(avail,2))
    raw_ps=[]
    presults=[]
    for g1,g2 in pairs:
        aA,aB,z,p=delong_independent(data_dict[g1][0],data_dict[g1][1],
                                      data_dict[g2][0],data_dict[g2][1])
        raw_ps.append(p if not np.isnan(p) else 1.0)
        presults.append((g1,g2,aA,aB,z,p))
    rej,padj,_,_=multipletests(raw_ps,method='bonferroni')
    return [(g1,g2,aA,aB,z,p,pa,bool(r))
            for (g1,g2,aA,aB,z,p),r,pa in zip(presults,rej,padj)]


def _draw_race_panel_v2(ax, data_dict, groups, colors, model_label, sev):
    """Race panel: scatter + CI bars + stat badge."""
    avail=[g for g in groups if g in data_dict]
    if not avail:
        ax.text(0.5,0.5,'No data',ha='center',va='center',transform=ax.transAxes); return

    pair_res = _bonf_pairs_all(data_dict, groups, 'race')
    n_sig = sum(1 for r in pair_res if r[7])
    n_pairs = len(pair_res)

    x=np.arange(len(avail))
    for xi,g in enumerate(avail):
        yt,yp=data_dict[g]
        auc,lo,hi=bootstrap_auroc_ci(yt,yp,n=1000)
        col=colors.get(g,'#666')
        ax.errorbar(xi,auc,yerr=[[auc-lo],[hi-auc]],fmt='none',
                    ecolor=col,capsize=5,capthick=2,lw=1.8,alpha=0.8)
        ax.scatter(xi,auc,color=col,s=110,zorder=5,edgecolors='white',lw=1.3)
        n_grp = len(yt)
        ax.annotate(f'{auc:.3f}\nn={n_grp}', (xi, hi),
                    textcoords='offset points', xytext=(0,6),
                    ha='center', fontsize=7.5, color=col, fontweight='bold',
                    linespacing=1.3)

    ax.set_xticks(x)
    ax.set_xticklabels([g.replace('/',' /\n') if '/' in g else g for g in avail],
                       fontsize=8, rotation=10, ha='right')
    ax.set_ylabel('AUROC (bootstrap 95% CI)',fontsize=9)
    ax.set_xlabel('Race / Ethnicity Group',fontsize=9)
    ax.set_ylim(0.38,1.02)
    ax.axhline(0.5,color='#BBBBBB',ls=':',lw=1,alpha=0.7)
    ax.set_facecolor('#F8F9FA')
    ax.grid(axis='y',alpha=0.25,ls='--')

    # n_sig badge
    badge_col = '#1E8449' if n_sig==0 else '#C0392B'
    badge_txt = (f'{n_sig}/{n_pairs} pairs sig\n(Bonferroni) — stable ✓'
                 if n_sig==0 else
                 f'{n_sig}/{n_pairs} pairs sig\n(Bonferroni) ⚠')
    ax.text(0.02,0.97,badge_txt,transform=ax.transAxes,
            ha='left',va='top',fontsize=8,fontweight='bold',color=badge_col,
            bbox=dict(boxstyle='round,pad=0.35',facecolor='white',
                      edgecolor=badge_col,alpha=0.9))
    ax.set_title(f'{model_label}\ntested by race',fontweight='bold',fontsize=10)

def _draw_quartile_panel_v2(ax, data_dict, groups, model_label, sev,
                             show_jt=False, care=None):
    """Quartile panel: line + CI + DeLong brackets + Spearman + JT."""
    avail = [g for g in groups if g in data_dict]
    if not avail:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center',
                transform=ax.transAxes)
        return

    pair_res = _bonf_pairs_all(data_dict, groups, 'quartile')
    n_sig   = sum(1 for r in pair_res if r[7])
    n_pairs = len(pair_res)

    x = np.arange(len(avail))
    aucs, los, his = [], [], []
    for g in avail:
        yt, yp = data_dict[g]
        auc_cv = float(roc_auc_score(yt, yp))
        _, lo, hi = bootstrap_auroc_ci(yt, yp, n=1000)
        aucs.append(auc_cv); los.append(lo); his.append(hi)
    aucs = np.array(aucs)
    los  = np.array(los)
    his  = np.array(his)

    model_col = '#2874A6' if 'All' in model_label else '#C0392B'
    ax.errorbar(x, aucs,
                yerr=[np.clip(aucs - los, 0, 1), np.clip(his - aucs, 0, 1)],
                color=model_col, marker='o', ls='-',
                lw=2.2, markersize=8, capsize=5, capthick=2,
                ecolor=model_col, alpha=0.9, zorder=3)
    for xi, (a, h) in enumerate(zip(aucs, his)):
        ax.annotate(f'{a:.3f}', (xi, h),
                    textcoords='offset points', xytext=(0, 7),
                    ha='center', fontsize=8.5,
                    color=model_col, fontweight='bold')

    # ── Q1 reference line REMOVED (was overlapping Spearman badge) ────────

    # ── Q1 vs Q4 bracket REMOVED ─────────────────────────────────────────
    # DeLong tests uniformity across all 6 pairs, not degradation.
    # Singling out Q1 vs Q4 as a 'primary test' was misleading —
    # degradation is tested by the directional JT (bottom-right).
    # The n_sig badge (top-right) summarises all-pairs DeLong correctly.

    # ── DeLong uniformity summary (top-left) ────────────────────────────
    # Shows whether DeLong detects any significant differences across pairs.
    # Not a test of degradation direction — that is the JT (bottom-right).
    if pair_res:
        if n_sig == 0:
            delong_txt = f'DeLong (Bonf): 0/{n_pairs} pairs sig\n✔ No significant differences'
            delong_col = '#1E8449'
            delong_edge = '#1E8449'
            delong_face = '#EAFAF1'
        else:
            delong_txt = f'DeLong (Bonf): {n_sig}/{n_pairs} pairs sig\n⚠ Significant differences detected'
            delong_col = '#C0392B'
            delong_edge = '#C0392B'
            delong_face = '#FDEDEC'
        ax.text(0.02, 0.97, delong_txt,
                transform=ax.transAxes, ha='left', va='top',
                fontsize=8, fontweight='bold', color=delong_col,
                bbox=dict(boxstyle='round,pad=0.35', facecolor=delong_face,
                          edgecolor=delong_edge, alpha=0.92))

    # ── Spearman (bottom-left) ────────────────────────────────────────────
    if len(aucs) >= 3:
        rho, p_sp = spearmanr(range(1, len(aucs) + 1), aucs)
        ax.text(0.02, 0.04, f'Spearman ρ={rho:.3f}  {_fmt(p_sp)}',
                transform=ax.transAxes, ha='left', va='bottom', fontsize=8,
                color='#333',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                          edgecolor='#AAAAAA', alpha=0.85))

    # ── Directional JT (bottom-right, Model B only) ─────────────────────
    # This is the degradation test — H1: AUROC decreases Q1→Q2→Q3→Q4.
    # Uses true_fold_aurocs_store (bootstrap AUROCs from pooled predictions).
    if show_jt and care:
        try:
            fs  = globals().get('true_fold_aurocs_store', {})
            mk  = 'B – Q1 Only' if 'Q1 Only' in model_label else 'A – All Quartiles'
            jt_obs = [fs.get((care, sev, mk, q), np.array([]))
                      for q in groups]
            jt_obs = [a for a in jt_obs if len(a) >= 2]
            if len(jt_obs) >= 3:
                jt, p_jt = jonckheere_trend_test(jt_obs)
                jt_col = '#C0392B' if p_jt < 0.05 else '#666'
                ax.text(0.98, 0.04,
                        f'Directional JT p={p_jt:.4f}' if p_jt >= 0.001 else 'Directional JT p<0.001',
                        transform=ax.transAxes, ha='right', va='bottom',
                        fontsize=8, color=jt_col,
                        bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                                  edgecolor=jt_col, alpha=0.85))
        except:
            pass

    # ── n_sig badge removed ──────────────────────────────────────────────
    # DeLong stability summary (top-left) already shows n_sig/n_pairs.

    xlabels = ['Q1\n(most freq)', 'Q2', 'Q3', 'Q4\n(least freq)']
    ax.set_xticks(x)
    ax.set_xticklabels(xlabels[:len(avail)], fontsize=9)
    ax.set_ylabel('AUROC (bootstrap 95% CI)', fontsize=9)
    ax.set_xlabel('Care Frequency Quartile', fontsize=9)
    # Extra top padding so bracket + value labels don't clip
    y_min = min(los) - 0.05
    y_max = max(his) + 0.18
    ax.set_ylim(max(0.3, y_min), min(1.02, y_max))
    ax.axhline(0.5, color='#BBBBBB', ls=':', lw=1, alpha=0.7)
    ax.set_facecolor('#F8F9FA')
    ax.grid(axis='y', alpha=0.25, ls='--')
    ax.set_title(f'{model_label}\ntested by quartile',
                 fontweight='bold', fontsize=10)

def plot_enhanced_2x2(care, sev, save_path):
    fig, axes = plt.subplots(2, 2, figsize=(15, 11))
    fig.patch.set_facecolor('#FAFAFA')

    qA = _pull_cal(care, sev, 'A – All Quartiles', 'quartile')
    qB = _pull_cal(care, sev, 'B – Q1 Only',       'quartile')
    rA = _pull_cal(care, sev, 'A – By Race',        'race')
    rB = _pull_cal(care, sev, 'B – Q1 By Race',     'race')

    _draw_race_panel_v2(axes[0,0],    rA, RACE_LABELS, RACE_COLOURS,
                        'Model A', sev)
    _draw_quartile_panel_v2(axes[0,1], qA, QUARTILES,
                             'Model A — All Quartiles', sev,
                             show_jt=False, care=care)
    _draw_race_panel_v2(axes[1,0],    rB, RACE_LABELS, RACE_COLOURS,
                        'Model B (Q1-trained)', sev)
    _draw_quartile_panel_v2(axes[1,1], qB, QUARTILES,
                             'Model B — Q1 Only', sev,
                             show_jt=True, care=care)

    # Row labels
    for ax_r, lbl in [(0,'Model A\n(trained: all Q)'),
                       (1,'Model B\n(trained: Q1 only)')]:
        axes[ax_r,0].annotate(lbl, xy=(-0.26,0.5), xycoords='axes fraction',
                               fontsize=10,fontweight='bold',va='center',
                               ha='center',rotation=90,color='#333',
                               annotation_clip=False)
    # Col headers
    for ax_c, lbl in [(0,'← Race axis\n(expect: stable, non-sig)'),
                       (1,'← Care Quartile axis\n(expect: variation, sig)')]:
        axes[0,ax_c].set_title(axes[0,ax_c].get_title()+'\n'+lbl,
                               fontsize=10,fontweight='bold')

    fig.suptitle(
        f'{care}  —  {sev} Severity (SOFA {SOFA_STR_E[sev]})\n'
        '2×2 AUROC Panel with Statistical Annotations\n'
        'CI = bootstrap 1000 resamples on pooled CV predictions (same method for race & quartile)\n'
        'DeLong badge (top-left) = uniformity test (are any groups different?)  |  Directional JT (bottom-right) = degradation test (Q1→Q4 trend)',
        fontsize=11,fontweight='bold',y=1.04,
        color=SEV_COL_E.get(sev,'#333'))
    plt.tight_layout()
    plt.savefig(save_path,bbox_inches='tight',dpi=150)
    plt.show()
    print(f'Saved: {save_path}')


def plot_combined_severity(care, sev, out_folder):
    """
    For one (care, severity): produce TWO figures side by side.
    Figure 1: Enhanced 2×2 raw AUROC (existing).
    Figure 2: 2×2 ΔAUROC deviation panels (new — point 1 & 7).
    """
    path1 = os.path.join(out_folder, f'fig_enhanced2x2_{care.lower()}_{sev.lower()}.png')
    path2 = os.path.join(out_folder, f'fig_delta_auroc_{care.lower()}_{sev.lower()}.png')
    plot_enhanced_2x2(care, sev, path1)
    plot_delta_auroc_panel(care, sev, path2)

print('Generating enhanced 2×2 + ΔAUROC panels per severity...')
for care in ['Mouthcare', 'Turning']:
    for sev in SEVERITY_LABELS:
        plot_combined_severity(care, sev, OUTPUT_FOLDER)
print('Done — each severity has 2 figures: raw AUROC 2×2 + ΔAUROC 2×2.')


<!-- superseded — see comprehensive statistical framework in cells 29–39 -->

In [ ]:
# Old detailed DeLong section — superseded by stat framework cells 30–39.
print("Old pairwise DeLong cell — superseded. See stat framework cells above.")


## 11. Calibration Analysis

Calibration curves and Expected Calibration Error (ECE) for Model B across quartiles. Raw XGBoost probabilities are corrected using per-fold Platt scaling (fitted on a 15% stratified hold-out from each training fold) before all calibration analyses. Isotonic regression is applied in parallel as a sensitivity check. Calibration curves are constructed using eight quantile-based bins; ECE is computed using ten equal-width bins.

For Model B, curves above the diagonal in Q3 and Q4 — where actual mortality exceeds predicted probability — indicate systematic under-prediction of mortality risk for lower-care patients, consistent with the hypothesis that these patients carry mortality risk not captured by clinical severity features alone.

In [ ]:
def plot_calibration_grid(care_label, model_name, save_path):
    sevs = SEVERITY_LABELS
    fig, axes = plt.subplots(1, len(sevs), figsize=(5*len(sevs), 5))

    for ax, sev in zip(axes, sevs):
        entries = [(g,yt,yp) for (cl,sl,mn,gt,g,yt,yp) in calibration_store
           if cl==care_label and sl==sev and mn==model_name and gt=='quartile']
        ax.plot([0,1],[0,1],'k--', lw=1.2, label='Perfect calibration', alpha=0.6)
        for q, yt, yp in sorted(entries, key=lambda x: x[0]):
            if len(yt) < 20:
                continue
            try:
                fp, mp = calibration_curve(np.array(yt), np.array(yp),
                                           n_bins=8, strategy='quantile')
                col = Q_COLOURS.get(q,'grey')
                ax.plot(mp, fp, marker='o', lw=2, color=col,
                        label=f'{q} (n={len(yt)})', markersize=6)
                ax.fill_between(mp, mp, fp, alpha=0.07, color=col)
            except:
                pass
        ax.set_xlabel('Mean Predicted Probability', fontsize=10)
        ax.set_ylabel('Actual Mortality Rate', fontsize=10)
        ax.set_title(f'SOFA {sev}\n(SOFA {"0–6" if sev=="Low" else "7–11" if sev=="Medium" else "≥12"})',
                     fontweight='bold')
        ax.legend(title='Test Quartile', fontsize=8)
        ax.set_xlim(0,1); ax.set_ylim(0,1)
        ax.text(0.52,0.04,'Above diagonal = model\nunder-predicts mortality',
                fontsize=7.5,color='#C0392B',alpha=0.8,
                bbox=dict(boxstyle='round,pad=0.3',facecolor='#FDEDEC',alpha=0.5))

    fig.suptitle(f'{care_label} — Calibration Curves | {model_name}\n'
                 f'Q3/Q4 above diagonal → model under-predicts their mortality → supports hypothesis',
                 fontsize=11, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(save_path, bbox_inches='tight', dpi=150)
    plt.show()
    print(f'  Saved: {save_path}')

for care in ['Mouthcare','Turning']:
    for model_tag, model_name in [
        ('modelA_quartile', 'A – All Quartiles'),
        ('modelB_quartile', 'B – Q1 Only'),
    ]:
        plot_calibration_grid(
            care, model_name,
            os.path.join(OUTPUT_FOLDER, f'fig_calibration_{care.lower()}_{model_tag}.png')
        )

## 11b. Brier Score Analysis

The Brier score (mean squared error between calibrated predicted probabilities and observed outcomes) is reported for all four configurations alongside AUROC. Unlike AUROC which measures discrimination only, the Brier score jointly captures discrimination and calibration accuracy. A widening Brier score gap from Q1 to Q4 for Model B indicates that the model's probability estimates become increasingly unreliable as care quality decreases — not merely its ranking of patients.

In [ ]:

# ── Brier Score Analysis — Supplementary Figure 4
# Prints summary table and saves fig_brier_combo_*.png
# Shows Brier score across all 4 configs and both care types.
# Brier score captures both discrimination and calibration (unlike AUROC alone).
brier_cols = ['Care Type','Severity','Model','Test Set','N','Mortality%','Brier','Brier_CI']
brier_summary = results_df[brier_cols].dropna(subset=['Brier']).copy()
brier_summary = brier_summary.sort_values(['Care Type','Severity','Model','Test Set'])

print('Brier Score Summary')
print('='*100)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 220)
pd.set_option('display.max_rows', 200)
print(brier_summary.to_string(index=False))

print('\n\nBrier Score Range (max - min)')
print('='*80)
for care in ['Mouthcare', 'Turning']:
    print(f'\n{care}:')
    for model in ['A – All Quartiles', 'B – Q1 Only', 'A – By Race', 'B – Q1 By Race']:
        sub = results_df[
            (results_df['Care Type'] == care) &
            (results_df['Model'] == model)
        ].dropna(subset=['Brier'])
        if len(sub) < 2:
            continue
        mn, mx = sub['Brier'].min(), sub['Brier'].max()
        print(f'  {model:<25}  range={mx-mn:.4f}  (min={mn:.4f}, max={mx:.4f})')


# ── Combined Brier master figure — same logic as combo4 AUROC ─────────────────
# All 4 model lines on one figure per care type.
# LEFT of divider = quartile axis, RIGHT = race axis.
# Y-axis zoomed to data range (not from 0) so differences are visible.

BRIER_SPECS = [
    ('A – All Quartiles', 'q', '#2874A6', 'o', '-',  'A->Quartile (all Q trained)'),
    ('B – Q1 Only',       'q', '#C0392B', 's', '--', 'B->Quartile (Q1 trained)'),
    ('A – By Race',       'r', '#E67E22', 'D', '-',  'A->Race (all Q trained)'),
    ('B – Q1 By Race',    'r', '#1E8449', '^', '--', 'B->Race (Q1 trained)'),
]

Q_XLABELS_B = ['Q1\n(most freq)', 'Q2', 'Q3', 'Q4\n(least freq)']
SOFA_STR_B  = {'Low':'0-6', 'Medium':'7-11', 'High':'>=12'}

def _get_brier_q(care, sev, model_name):
    mdf = results_df[
        (results_df['Care Type'] == care) &
        (results_df['Severity']  == sev) &
        (results_df['Model']     == model_name)
    ].copy()
    mdf['label'] = mdf['Test Set'].str.extract(r'(Q\d)')
    mdf = mdf.dropna(subset=['label','Brier']).sort_values('label').reset_index(drop=True)
    return mdf

def _get_brier_r(care, sev, model_name):
    mdf = results_df[
        (results_df['Care Type'] == care) &
        (results_df['Severity']  == sev) &
        (results_df['Model']     == model_name)
    ].copy()
    mdf['label'] = mdf['Test Set'].str.replace('Race: ', '', regex=False)
    mdf['_ord']  = mdf['label'].map({r: i for i, r in enumerate(RACE_LABELS)})
    mdf = mdf.dropna(subset=['label','Brier','_ord']).sort_values('_ord').reset_index(drop=True)
    return mdf

for care in ['Mouthcare', 'Turning']:
    fig, axes = plt.subplots(1, 3, figsize=(19, 6))
    fig.suptitle(
        care + ' - Brier Score: All 4 Models, Care Axis vs Race Axis\n'
        'LEFT of divider = care quartile test  |  RIGHT = race group test\n'
        'Lower Brier = better probabilistic calibration  |  Error bars = 95% CI',
        fontsize=12, fontweight='bold', y=1.04
    )

    for col, sev in enumerate(SEVERITY_LABELS):
        ax = axes[col]

        # Collect all brier values and CI bounds for smart y-axis range
        all_brier = []
        _all_bhi_vals = []
        _all_blo_vals = []

        rdf_sample = _get_brier_r(care, sev, 'A – By Race')
        n_q   = 4
        n_r   = len(rdf_sample)
        gap   = 1.2
        x_q   = np.arange(n_q)
        x_r   = np.arange(n_q + gap, n_q + gap + n_r)

        for mname, gtype, color, marker, ls, label in BRIER_SPECS:
            if gtype == 'q':
                mdf   = _get_brier_q(care, sev, mname)
                x_use = x_q[:len(mdf)]
            else:
                mdf   = _get_brier_r(care, sev, mname)
                x_use = x_r[:len(mdf)]

            if len(mdf) == 0:
                continue

            bm   = mdf['Brier'].values.astype(float)
            blo  = mdf['Brier_lo'].fillna(mdf['Brier']).values.astype(float)
            bhi  = mdf['Brier_hi'].fillna(mdf['Brier']).values.astype(float)
            yerr = np.array([np.clip(bm - blo, 0, 1), np.clip(bhi - bm, 0, 1)])
            all_brier.extend(bm.tolist())
            _all_bhi_vals.extend(bhi.tolist())
            _all_blo_vals.extend(blo.tolist())

            ax.errorbar(x_use, bm, yerr=yerr,
                        color=color, marker=marker, linestyle=ls,
                        linewidth=2, markersize=7.5, capsize=4, capthick=1.8,
                        ecolor=color, alpha=0.88,
                        label=label + '  [range=' + f'{bm.max()-bm.min():.3f}]',
                        zorder=3)

            # Alternate annotation offsets to avoid overlap
            spec_idx = [s[0] for s in BRIER_SPECS].index(mname)
            ann_y = [10, -16, 10, -16][spec_idx]
            ann_x = [-6,  6,  -6,   6][spec_idx]
            for xi, bi in enumerate(bm):
                ax.annotate(f'{bi:.3f}', (x_use[xi], bi),
                            textcoords='offset points',
                            xytext=(ann_x, ann_y),
                            ha='center', fontsize=7.5,
                            color=color, fontweight='bold', zorder=4)

        # Divider between quartile and race sections
        if n_r > 0:
            div_x = n_q + gap / 2 - 0.1
            ax.axvline(div_x, color='#999999', lw=1.5, ls='--', alpha=0.6, zorder=1)
            # Labels below x-axis using text at bottom of axes
            ax.text(n_q / 2 - 0.5, -0.018,
                    'Care axis',
                    fontsize=8.5, color='#2874A6', style='italic', fontweight='bold',
                    ha='center', va='top', transform=ax.get_xaxis_transform())
            ax.text(n_q + gap + n_r / 2 - 0.5, -0.018,
                    'Race axis',
                    fontsize=8.5, color='#E67E22', style='italic', fontweight='bold',
                    ha='center', va='top', transform=ax.get_xaxis_transform())

        # Smart y-axis: zoom to CI extremes with padding, not from 0
        if all_brier:
            _hi_src = [v for v in _all_bhi_vals if not np.isnan(v)]
            _lo_src = [v for v in _all_blo_vals if not np.isnan(v)]
            y_min = max(0, (min(_lo_src) if _lo_src else min(all_brier)) - 0.015)
            y_max = ((max(_hi_src) if _hi_src else max(all_brier)) + 0.025)
            ax.set_ylim(y_min, y_max)

        # X ticks
        race_lbls  = rdf_sample['label'].tolist() if n_r > 0 else []
        all_x      = list(x_q) + list(x_r)
        all_labels = Q_XLABELS_B[:n_q] + race_lbls
        ax.set_xticks(all_x)
        ax.set_xticklabels(all_labels, fontsize=8, rotation=18, ha='right')

        sev_col = {'Low':'#1E8449','Medium':'#B7770D','High':'#922B21'}[sev]
        ax.set_title(sev + ' Severity  (SOFA ' + SOFA_STR_B[sev] + ')',
                     fontweight='bold', fontsize=11, color='white',
                     bbox=dict(boxstyle='round,pad=0.3', facecolor=sev_col))
        ax.set_ylabel('Brier Score (lower = better)', fontsize=9)
        ax.set_xlabel('Test Group', fontsize=9)
        ax.set_facecolor('#F8F9FA')
        ax.grid(axis='y', alpha=0.3, linestyle='--')
        ax.axhline(0.25, color='#BBBBBB', ls=':', lw=1, alpha=0.6)
        ax.legend(fontsize=7.5, loc='upper left', framealpha=0.88,
                  handlelength=2.0, ncol=1)

    plt.tight_layout()
    path = os.path.join(OUTPUT_FOLDER, 'fig_brier_combo_' + care.lower() + '.png')
    plt.savefig(path, bbox_inches='tight', dpi=150)
    plt.show()
    print('Saved: ' + path)


## 11c. Calibration Drift — ECE and Predicted vs Actual

ECE per quartile (expected to increase Q1→Q4 in the primary strata) and mean predicted probability versus actual mortality rate per quartile, showing the direction and magnitude of miscalibration across care-frequency groups.

In [ ]:

import warnings
warnings.filterwarnings('ignore')

SEV_COLORS_DARK = {'Low': '#1E8449', 'Medium': '#B7770D', 'High': '#922B21'}
COLOR_B   = '#C0392B'
COLOR_ACT = '#2E86C1'
COLOR_GAP = '#7D3C98'
SOFA_STR  = {'Low':'0-6', 'Medium':'7-11', 'High':'>=12'}

def compute_ece(y_true, y_prob, n_bins=10):
    y_true = np.array(y_true)
    y_prob = np.array(y_prob)
    bins   = np.linspace(0, 1, n_bins + 1)
    ece    = 0.0
    n      = len(y_true)
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (y_prob >= lo) & (y_prob < hi)
        if mask.sum() == 0:
            continue
        acc  = y_true[mask].mean()
        conf = y_prob[mask].mean()
        ece += (mask.sum() / n) * abs(acc - conf)
    return ece

def collect_quartile_stats(care, sev, model_name):
    rows = []
    for q in QUARTILES:
        matches = [(yt, yp) for (cl, sl, mn, gt, gl, yt, yp) in calibration_store
                   if cl == care and sl == sev and mn == model_name
                   and gt == 'quartile' and gl == q]
        if not matches:
            continue
        yt, yp = matches[0]
        yt_arr = np.array(yt, dtype=float)
        yp_arr = np.array(yp, dtype=float)
        if len(yt_arr) < 10 or len(np.unique(yt_arr)) < 2:
            continue
        rows.append({
            'Quartile':   q,
            'N':          len(yt_arr),
            'Mean_Pred':  float(yp_arr.mean()),
            'Actual_Mort':float(yt_arr.mean()),
            'Gap':        float(yt_arr.mean() - yp_arr.mean()),
            'ECE':        compute_ece(yt_arr, yp_arr),
        })
    return rows

Q_LABELS = ['Q1\n(most freq)', 'Q2', 'Q3', 'Q4\n(least freq)']

for care in ['Mouthcare', 'Turning']:
    fig, axes = plt.subplots(2, 3, figsize=(17, 10))
    fig.suptitle(
        care + ' - Calibration Drift Across Care Quartiles\n'
        'Model B (trained on Q1 only) evaluated on each quartile\n'
        'Top: ECE per quartile (higher = worse)  |  '
        'Bottom: Mean predicted risk vs actual mortality',
        fontsize=12, fontweight='bold', y=1.02
    )

    all_rows = {}
    for sev in SEVERITY_LABELS:
        all_rows[sev] = collect_quartile_stats(care, sev, 'B – Q1 Only')

    for col, sev in enumerate(SEVERITY_LABELS):
        rows = all_rows[sev]
        if not rows:
            axes[0, col].set_visible(False)
            axes[1, col].set_visible(False)
            continue

        x         = np.arange(len(rows))
        ece       = np.array([r['ECE']          for r in rows])
        mean_pred = np.array([r['Mean_Pred']    for r in rows])
        act_mort  = np.array([r['Actual_Mort']  for r in rows])
        gap       = np.array([r['Gap']          for r in rows])
        ns        = [r['N'] for r in rows]
        qs        = [r['Quartile'] for r in rows]
        sev_col   = SEV_COLORS_DARK[sev]

        # Row 0: ECE
        ax = axes[0, col]
        ax.bar(x, ece, color=sev_col, alpha=0.65, edgecolor='white',
               linewidth=1.5, width=0.5, zorder=2)
        ax.plot(x, ece, color=sev_col, marker='o', linewidth=2.2,
                markersize=8, zorder=4)
        for xi, (ei, ni) in enumerate(zip(ece, ns)):
            ax.annotate(f'{ei:.3f}\n(n={ni})', (xi, ei),
                        textcoords='offset points', xytext=(0, 7),
                        ha='center', fontsize=8.5, fontweight='bold',
                        color=sev_col)
        ax.set_facecolor('#F8F9FA')
        ax.grid(axis='y', alpha=0.3, linestyle='--')
        ax.set_xticks(x)
        ax.set_xticklabels(Q_LABELS[:len(rows)], fontsize=9)
        ax.set_ylabel('Expected Calibration Error (ECE)', fontsize=9)
        ax.set_xlabel('Care Frequency Quartile', fontsize=9)
        ax.set_ylim(0, max(ece) * 1.6)
        ax.set_title(sev + ' Severity  (SOFA ' + SOFA_STR[sev] + ')',
                     fontweight='bold', fontsize=11, color='white',
                     bbox=dict(boxstyle='round,pad=0.3', facecolor=sev_col))
        if len(ece) >= 2:
            trend     = 'worsening up' if ece[-1] > ece[0] else 'improving down'
            trend_col = '#C0392B'      if ece[-1] > ece[0] else '#1E8449'
            ax.text(0.97, 0.97, 'Q1 to Q4: ' + trend,
                    transform=ax.transAxes, ha='right', va='top',
                    fontsize=8.5, color=trend_col, fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                              edgecolor=trend_col, alpha=0.9))

        # Row 1: Predicted vs Actual
        ax = axes[1, col]
        ax.plot(x, mean_pred, color=COLOR_B, marker='s', linewidth=2.4,
                markersize=9, linestyle='--', label='Mean predicted risk', zorder=3)
        ax.plot(x, act_mort,  color=COLOR_ACT, marker='o', linewidth=2.4,
                markersize=9, linestyle='-',  label='Actual mortality rate', zorder=4)
        ax.fill_between(x, mean_pred, act_mort,
                        where=(act_mort >= mean_pred),
                        alpha=0.15, color=COLOR_GAP, label='Underestimation gap')
        ax.fill_between(x, mean_pred, act_mort,
                        where=(act_mort < mean_pred),
                        alpha=0.15, color='#E67E22', label='Overestimation')
        for xi, (mp, am, gp) in enumerate(zip(mean_pred, act_mort, gap)):
            ax.annotate(f'{mp:.1%}', (xi, mp),
                        textcoords='offset points', xytext=(-14, -14),
                        ha='center', fontsize=8, color=COLOR_B, fontweight='bold')
            ax.annotate(f'{am:.1%}', (xi, am),
                        textcoords='offset points', xytext=(14, 7),
                        ha='center', fontsize=8, color=COLOR_ACT, fontweight='bold')
            if abs(gp) > 0.005:
                ax.annotate('D=' + f'{gp:+.1%}', (xi, (mp + am) / 2),
                            textcoords='offset points', xytext=(22, 0),
                            ha='left', fontsize=7.5, color=COLOR_GAP, fontweight='bold')
        ax.set_facecolor('#F8F9FA')
        ax.grid(axis='y', alpha=0.3, linestyle='--')
        ax.set_xticks(x)
        ax.set_xticklabels(Q_LABELS[:len(rows)], fontsize=9)
        ax.set_ylabel('Mortality Rate / Predicted Risk', fontsize=9)
        ax.set_xlabel('Care Frequency Quartile', fontsize=9)
        ax.set_ylim(0, max(act_mort.max(), mean_pred.max()) * 1.5)
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))
        ax.legend(fontsize=8, loc='upper left', framealpha=0.9)
        ax.set_title(sev + ' Severity  (SOFA ' + SOFA_STR[sev] + ')',
                     fontweight='bold', fontsize=11, color='white',
                     bbox=dict(boxstyle='round,pad=0.3', facecolor=sev_col))
        max_idx = int(np.argmax(np.abs(gap)))
        direction = 'underestimates' if gap[max_idx] > 0 else 'overestimates'
        ax.text(0.97, 0.97,
                'Largest gap at ' + qs[max_idx] + ':\nModel ' + direction +
                ' mortality by ' + f'{abs(gap[max_idx]):.1%}',
                transform=ax.transAxes, ha='right', va='top',
                fontsize=8, color=COLOR_GAP, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                          edgecolor=COLOR_GAP, alpha=0.9))

    plt.tight_layout()
    path = os.path.join(OUTPUT_FOLDER, 'fig_calibration_drift_' + care.lower() + '.png')
    plt.savefig(path, bbox_inches='tight', dpi=150)
    plt.show()
    print('Saved: ' + path)

    # Print summary table
    print('=' * 70)
    print('  ' + care + ' - Calibration Drift Summary (Model B)')
    print('=' * 70)
    print(f'  {"Severity":<10} {"Q":<4} {"N":<6} {"Pred%":<8} {"Actual%":<9} {"Gap%":<8} {"ECE":<7}')
    print('  ' + '-' * 52)
    for sev in SEVERITY_LABELS:
        for r in all_rows[sev]:
            flag = ' <- UNDERESTIMATE' if r['Gap'] > 0.03 else ''
            print(f'  {sev:<10} {r["Quartile"]:<4} {r["N"]:<6} '
                  f'{r["Mean_Pred"]:<8.1%} {r["Actual_Mort"]:<9.1%} '
                  f'{r["Gap"]:+.1%}   {r["ECE"]:.3f}{flag}')
        print()


## 12. Feature Importance — SHAP Analysis

In [ ]:
RENAME = {
    'gcs_min':'GCS (min)','respiration':'SOFA Respiration',
    'cardiovascular':'SOFA Cardiovascular','cns':'SOFA CNS',
    'renal':'SOFA Renal','liver':'SOFA Liver','coagulation':'SOFA Coagulation',
    'mbp_min':'MBP (min)','mbp_mean':'MBP (mean)','mbp_max':'MBP (max)',
    'heart_rate_min':'HR (min)','heart_rate_mean':'HR (mean)',
    'resp_rate_mean':'Resp Rate (mean)','temperature_mean':'Temp (mean)',
    'creatinine_max':'Creatinine (max)','bun_max':'BUN (max)',
    'albumin_min':'Albumin (min)','bicarbonate_min':'Bicarbonate (min)',
    'bilirubin_total_max':'Bilirubin (max)','wbc_max':'WBC (max)',
}
SOFA_SET = {'SOFA Respiration','SOFA Cardiovascular','SOFA CNS',
            'SOFA Renal','SOFA Liver','SOFA Coagulation'}

def plot_shap_grid(care_label, model_name, save_path, top_n=12):
    sevs = [s for s in SEVERITY_LABELS
            if (care_label, s, model_name) in shap_store]
    if not sevs:
        print(f'  No SHAP data for {care_label} | {model_name}')
        return
    fig, axes = plt.subplots(1, len(sevs), figsize=(6*len(sevs), 5))
    if len(sevs) == 1:
        axes = [axes]
    for ax, sev in zip(axes, sevs):
        model, X_train, feats = shap_store[(care_label, sev, model_name)]
        sample = X_train.sample(min(400, len(X_train)), random_state=RANDOM_STATE)
        explainer   = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(sample)
        mean_abs    = pd.Series(np.abs(shap_values).mean(axis=0), index=feats)
        top         = mean_abs.nlargest(top_n).sort_values()
        top.index   = [RENAME.get(f,f) for f in top.index]
        colors      = ['#1A5276' if f in SOFA_SET else '#1E8449' for f in top.index]
        ax.barh(range(len(top)), top.values, color=colors, edgecolor='white', height=0.7)
        ax.set_yticks(range(len(top)))
        ax.set_yticklabels(top.index, fontsize=9)
        ax.set_xlabel('Mean |SHAP value|', fontsize=9)
        ax.set_title(f'SOFA {sev}', fontweight='bold', fontsize=10)
    patch1 = mpatches.Patch(color='#1A5276', label='LODS proxy (SOFA subscores)')
    patch2 = mpatches.Patch(color='#1E8449', label='APS III proxy (labs/vitals)')
    fig.legend(handles=[patch1,patch2], fontsize=9,
               loc='lower center', bbox_to_anchor=(0.5,-0.05), ncol=2)
    fig.suptitle(f'{care_label} — SHAP Feature Importance | {model_name}',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, bbox_inches='tight', dpi=150)
    plt.show()
    print(f'  Saved: {save_path}')

for care in ['Mouthcare','Turning']:
    for mname, mshort in [('A – All Quartiles','A'),('B – Q1 Only','B')]:
        plot_shap_grid(care, mname,
                       os.path.join(OUTPUT_FOLDER, f'fig4_shap_{care.lower()}_model{mshort}.png'))

## 13. Results Export

In [ ]:
HDR_BG='1F3864'; HDR_FG='FFFFFF'; DARK_BG='2E4057'; BORDER='BDC3C7'
SEV_BG  = {'Low':'EBF5FB','Medium':'FEF9E7','High':'FDEDEC'}
MOD_BG  = {'A – All Quartiles':'D6E4F0','B – Q1 Only':'D5F5E3'}
thin    = Border(left=Side('thin',color=BORDER),right=Side('thin',color=BORDER),
                 top=Side('thin',color=BORDER), bottom=Side('thin',color=BORDER))

DISP_COLS = ['Care Type','Severity','Model','Test Set','N','Mortality%',
             'AUROC_CI','Sensitivity','Specificity','PPV','NPV','F1']
DISP_COLS = [c for c in DISP_COLS if c in results_df.columns]
COL_W     = {'Care Type':13,'Severity':10,'Model':22,'Test Set':20,'N':7,
             'Mortality%':12,'AUROC_CI':28,'Sensitivity':28,'Specificity':28,
             'PPV':28,'NPV':28,'F1':28}

def write_sheet(ws, df, title):
    cols = list(df.columns); nc = len(cols)
    ws.merge_cells(start_row=1,start_column=1,end_row=1,end_column=nc)
    c=ws.cell(1,1,title)
    c.font=Font('Calibri',bold=True,size=13,color=HDR_FG)
    c.fill=PatternFill('solid',fgColor=DARK_BG)
    c.alignment=Alignment(horizontal='center',vertical='center')
    ws.row_dimensions[1].height=24
    for ci,cn in enumerate(cols,1):
        c=ws.cell(2,ci,cn)
        c.font=Font('Calibri',bold=True,size=10,color=HDR_FG)
        c.fill=PatternFill('solid',fgColor=HDR_BG)
        c.alignment=Alignment(horizontal='center',vertical='center',wrap_text=True)
        c.border=thin
    ws.row_dimensions[2].height=34
    for ri,(_,row) in enumerate(df.iterrows(),3):
        sev=str(row.get('Severity',''))
        mod=str(row.get('Model',''))
        bg=MOD_BG.get(mod, SEV_BG.get(sev,'F8F9FA'))
        for ci,cn in enumerate(cols,1):
            val=row[cn]; c=ws.cell(ri,ci,val)
            c.font=Font('Calibri',size=10,bold=(cn=='AUROC_CI'))
            c.fill=PatternFill('solid',fgColor=bg)
            c.border=thin
            c.alignment=Alignment(
                horizontal='left' if cn in ('Care Type','Severity','Model','Test Set') else 'center',
                vertical='center')
        ws.row_dimensions[ri].height=18
    for ci,cn in enumerate(cols,1):
        ws.column_dimensions[get_column_letter(ci)].width=COL_W.get(cn,14)
    ws.freeze_panes='A3'

display_df = results_df[DISP_COLS]
wb = Workbook()
wb.remove(wb.active)

sheets = [
    ('All Results',     display_df,  'All Results — Mouthcare & Turning'),
    ('Mortality Table', mort_tbl,    'Mortality by Severity × Quartile (within-severity Q assignment)'),
]
for care in ['Mouthcare','Turning']:
    for sev in SEVERITY_LABELS:
        sub = display_df[
            (display_df['Care Type']==care) & (display_df['Severity']==sev)
        ]
        if len(sub)==0: continue
        sofa_range = {'Low':'0–6','Medium':'7–11','High':'≥12'}[sev]
        sheets.append((
            f'{care[:4]} – {sev}',
            sub,
            f'{care} | {sev} Severity (SOFA {sofa_range})'
        ))

for sname, df_s, title in sheets:
    ws = wb.create_sheet(title=sname)
    write_sheet(ws, df_s.reset_index(drop=True), title)
    print(f'   Sheet "{sname}" ({len(df_s)} rows)')

wb.save(OUTPUT_EXCEL)
print(f'\n Excel saved : {os.path.abspath(OUTPUT_EXCEL)}')
print(f' Figures in  : {os.path.abspath(OUTPUT_FOLDER)}/')

## 13b. Seed Sensitivity Analysis (Supplementary — Not Required)

With five-fold cross-validation as the primary evaluation framework, formal seed sensitivity analysis is not required: every patient is evaluated exactly once by a model that never trained on them, regardless of random seed. The only residual seed effect is XGBoost's internal subsampling, which is minor. This cell is retained for reference only and does not need to be run.

In [ ]:
# Seed sensitivity analysis — not required with 5-fold CV.
# Retained for reference only. Do not run in main pipeline.

## 14. Interpretation Framework

### Evidence hierarchy

| Evidence | Source | Criterion |
|---|---|---|
| SOFA balance | Section 5 output | KW p ≥ 0.05 across Q1–Q4 within each stratum |
| Mortality gradient | Section 8 — Mortality Table | Mortality rate increases Q1→Q4 within strata |
| Model B discrimination | Sections 10–10f | AUROC decreases Q1→Q4; DeLong Q1 vs Q4 significant after Bonferroni |
| Monotonic trend | Section 10b | JT p < 0.05 (secondary — pairwise DeLong is primary) |
| Race stability | Sections 10, 10b | 0/10 pairs significant after Bonferroni for both models |
| Calibration breakdown | Section 11 | Q3/Q4 above diagonal for Model B; ECE increases Q1→Q4 |
| Variance decomposition | Section 10d | Ratio Q/R_filt > 1 in primary strata |
| Sample size ruled out | Section 15 (sensitivity) | Model A_down outperforms Model B at Q3/Q4 in strongest strata |

### Severity strata definitions
- **Low**: SOFA 0–6 — absent or minimal organ dysfunction
- **Medium**: SOFA 7–11 — moderate multi-organ failure
- **High**: SOFA ≥12 — severe dysfunction; care-proxy signal expected to attenuate

## 15. Additional and Supplementary Analyses

The following cells provide supplementary analyses and publication-ready figures referenced in the paper. Each cell is self-contained and draws on outputs from the main pipeline above.

### Supplementary — AUROC with 95% CI: Model A vs Model B by Quartile

Line plots with annotated point estimates and 95% CI for both models across quartiles, per severity stratum and care phenotype. Provides a clear visual of the Q1→Q4 AUROC gradient.

In [ ]:

# ── AUROC with 95% CI — Model A vs B by Quartile
# Saves: fig_auroc_ci_*.png
# Simpler line-plot alternative to the enhanced 2×2 panel.
# NOT currently referenced in the paper — potential supplementary figure.
SEV_COLORS = {'Low':'#2ECC71', 'Medium':'#F39C12', 'High':'#E74C3C'}

def plot_auroc_ci_table(results_df, care_label):
    fig, axes = plt.subplots(1, len(SEVERITY_LABELS),
                             figsize=(5.5*len(SEVERITY_LABELS), 5.5))
    for ax, sev in zip(axes, SEVERITY_LABELS):
        sub = results_df[
            (results_df['Care Type'] == care_label) &
            (results_df['Severity']  == sev) &
            (results_df['Model'].isin(['A – All Quartiles','B – Q1 Only'])) &
            (results_df['Test Set'].str.contains('Q'))
        ].copy()
        if len(sub)==0: ax.set_visible(False); continue
        sub['q_label'] = sub['Test Set'].str.extract(r'(Q\d)')
        sub = sub.dropna(subset=['q_label','AUROC'])

        model_styles = {
            'A – All Quartiles': {'color':'#2E86C1','marker':'o','ls':'-',
                                  'label':'Model A (all Q trained)'},
            'B – Q1 Only':       {'color':'#C0392B','marker':'s','ls':'--',
                                  'label':'Model B (Q1 only trained)'},
        }
        for model, style in model_styles.items():
            mdf = sub[sub['Model']==model].sort_values('q_label')
            if len(mdf)==0: continue
            x     = np.arange(len(mdf))
            auroc = mdf['AUROC'].values.astype(float)
            lo    = mdf['AUROC_lo'].fillna(mdf['AUROC']).values.astype(float)
            hi    = mdf['AUROC_hi'].fillna(mdf['AUROC']).values.astype(float)
            ax.errorbar(x, auroc, yerr=[auroc-lo, hi-auroc],
                        color=style['color'], marker=style['marker'],
                        linestyle=style['ls'], linewidth=2.2,
                        markersize=8, capsize=5, capthick=2,
                        ecolor=style['color'], alpha=0.9,
                        label=style['label'])
            for xi,(a,l,h) in enumerate(zip(auroc,lo,hi)):
                ax.annotate(f'{a:.3f}\n({l:.3f}–{h:.3f})',
                            (xi,a), textcoords='offset points',
                            xytext=(0,13), ha='center', fontsize=7.5,
                            color=style['color'], fontweight='bold')

        ax.set_facecolor('#F8F9FA')
        ax.set_xticks(range(len(mdf)))
        ax.set_xticklabels(mdf['q_label'].values, fontsize=10)
        ax.set_ylim(0.4, 1.08)
        ax.axhline(0.5, color='grey', ls=':', lw=1, alpha=0.5)
        ax.set_xlabel('Test Quartile  (Q1 = most frequent care)', fontsize=10)
        ax.set_ylabel('AUROC (95% CI: mean ± 1.96×SE, 5-fold CV)', fontsize=10)
        sev_col = SEV_COLORS.get(sev, '#333333')
        ax.set_title(f'{sev} Severity', fontweight='bold', fontsize=12,
                     color='white',
                     bbox=dict(boxstyle='round,pad=0.3', facecolor=sev_col))
        ax.legend(fontsize=9, loc='lower left',
                  framealpha=0.9, edgecolor='#CCCCCC')
        ax.grid(axis='y', alpha=0.3, linestyle='--')

    fig.suptitle(
        f'{care_label} — AUROC with 95% Bootstrap CI: Model A vs Model B\n'
        f'Blue = Model A (all quartiles trained).  Red = Model B (Q1 only trained).\n'
        f'CI = mean ± 1.96×SE across 5 folds (not bootstrap). Divergence Q1→Q4 = training-environment effect.',
        fontsize=11, fontweight='bold', y=1.06)
    plt.tight_layout()
    path = os.path.join(OUTPUT_FOLDER, f'fig_auroc_ci_{care_label.lower()}.png')
    plt.savefig(path, bbox_inches='tight', dpi=150)
    plt.show(); print(f'Saved: {path}')

    print(f'\n{care_label} — AUROC Summary Table:')
    print('='*100)
    for sev in SEVERITY_LABELS:
        print(f'\n  {sev}:')
        sub2 = results_df[
            (results_df['Care Type']==care_label)&(results_df['Severity']==sev)&
            (results_df['Model'].isin(['A – All Quartiles','B – Q1 Only']))&
            (results_df['Test Set'].str.contains('Q'))
        ].copy()
        sub2['q'] = sub2['Test Set'].str.extract(r'(Q\d)')
        sub2 = sub2.dropna(subset=['q','AUROC']).sort_values(['q','Model'])
        for _,row in sub2.iterrows():
            print(f'    {row["Model"]:22s}  {row["q"]}:  '
                  f'AUROC={row["AUROC"]:.3f}  95%CI={row["AUROC_CI"]}  '
                  f'N={int(row["N"]):4d}  Mortality={row["Mortality%"]}%')

# [unused figure removed] for care in ['Mouthcare','Turning']:
#     plot_auroc_ci_table(results_df, care)
# Superseded by cell 41 enhanced 2x2 which shows the same information with stat annotations.


### Supplementary — Cohort Size and Mortality Heatmap (Table 1 equivalent)

Patient count and in-hospital mortality rate in each severity × care-frequency quartile cell, for both mouthcare and turning cohorts.

In [ ]:

# ── Cohort Size and Mortality Heatmap — Supplementary Figure 1 candidate
# Saves: fig_cohort_heatmap_*.png
# Two panels per care type: (1) patient count N, (2) mortality rate
# across severity × quartile. Confirms quartile design and cohort characteristics.
def plot_cohort_heatmap(data, care_label):
    counts = (data.groupby(['severity','quartile']).size()
              .unstack(fill_value=0).reindex(SEVERITY_LABELS)[QUARTILES])
    mort   = (data.groupby(['severity','quartile'])[MORTALITY].mean()
              .unstack(fill_value=np.nan).reindex(SEVERITY_LABELS)[QUARTILES])
    xlabels = ['Q1\n(most freq)','Q2','Q3','Q4\n(least freq)']

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Left: N — blue colormap
    ax = axes[0]
    im = ax.imshow(counts.values, cmap='Blues', aspect='auto', vmin=0)
    cbar = plt.colorbar(im, ax=ax); cbar.set_label('N patients', fontsize=9)
    ax.set_xticks(range(4)); ax.set_xticklabels(xlabels, fontsize=10)
    ax.set_yticks(range(3)); ax.set_yticklabels(SEVERITY_LABELS, fontsize=11)
    ax.set_title('Patient Count (N)', fontweight='bold', fontsize=12)
    ax.set_xlabel('Care Frequency Quartile', fontsize=10)
    for i in range(len(SEVERITY_LABELS)):
        for j in range(len(QUARTILES)):
            v = counts.values[i,j]
            dark = v > counts.values.max()*0.55
            ax.text(j, i, f'{int(v):,}', ha='center', va='center',
                    fontsize=12, fontweight='bold',
                    color='white' if dark else '#1A1A1A')

    # Right: mortality — red colormap
    ax = axes[1]
    im2 = ax.imshow(mort.values, cmap='Reds', aspect='auto', vmin=0, vmax=0.65)
    cbar2 = plt.colorbar(im2, ax=ax); cbar2.set_label('Mortality rate', fontsize=9)
    ax.set_xticks(range(4)); ax.set_xticklabels(xlabels, fontsize=10)
    ax.set_yticks(range(3)); ax.set_yticklabels(SEVERITY_LABELS, fontsize=11)
    ax.set_title('In-Hospital Mortality Rate', fontweight='bold', fontsize=12)
    ax.set_xlabel('Care Frequency Quartile', fontsize=10)
    for i in range(len(SEVERITY_LABELS)):
        for j in range(len(QUARTILES)):
            v = mort.values[i,j]
            if not np.isnan(v):
                dark = v > 0.38
                ax.text(j, i, f'{v:.1%}', ha='center', va='center',
                        fontsize=12, fontweight='bold',
                        color='white' if dark else '#1A1A1A')

    fig.suptitle(
        f'{care_label} — Cohort Size and Mortality by Severity × Quartile\n'
        f'Quartile assignment is within-severity — SOFA is balanced across Q1–Q4 by design.',
        fontsize=12, fontweight='bold', y=1.04)
    plt.tight_layout()
    path = os.path.join(OUTPUT_FOLDER, f'fig_cohort_heatmap_{care_label.lower()}.png')
    plt.savefig(path, bbox_inches='tight', dpi=150)
    plt.show(); print(f'Saved: {path}')

plot_cohort_heatmap(mc_data,   'Mouthcare')
plot_cohort_heatmap(turn_data, 'Turning')


### Supplementary — Table 1: Patient Characteristics by Quartile Within Severity

Baseline clinical characteristics per quartile within each severity stratum. Balanced characteristics across Q1–Q4 rule out systematic patient-level differences as an explanation for the observed performance gradient.

In [ ]:

# ── Table 1: Patient Characteristics by Quartile Within Severity
# Saves: fig_table1_*.png
# Baseline clinical characteristics (SOFA, age, LOS, vitals) per quartile.
# Balanced characteristics across Q1–Q4 confirm within-severity design worked.
def table1_by_quartile(data, care_label):
    age_col = 'admission_age' if 'admission_age' in data.columns else None
    los_col = 'los_icu'       if 'los_icu'       in data.columns else None
    rows = []
    for sev in SEVERITY_LABELS:
        sub = data[data['severity']==sev]
        for q in QUARTILES:
            qsub = sub[sub['quartile']==q]
            row = {
                'Severity':sev,'Quartile':q,'N':len(qsub),
                'Mortality %':round(qsub[MORTALITY].mean()*100,1),
                'SOFA (mean)':round(qsub[SOFA_COL].mean(),2),
                'Interval (h)':round(qsub[FREQ_COL].mean(),2),
            }
            if age_col: row['Age'] = round(qsub[age_col].mean(),1)
            if los_col: row['ICU LOS (d)'] = round(qsub[los_col].mean(),1)
            for feat in ['gcs_min','mbp_mean','creatinine_max','respiration',
                         'heart_rate_mean','temperature_mean']:
                if feat in qsub.columns:
                    row[feat] = round(qsub[feat].mean(),2)
            rows.append(row)

    t1 = pd.DataFrame(rows)
    print(f'\n{"="*80}\n  Table 1 — {care_label}\n{"="*80}')
    pd.set_option('display.max_columns',None); pd.set_option('display.width',220)
    print(t1.to_string(index=False))

    num_cols = [c for c in t1.columns if c not in ['Severity','Quartile','N']]

    # One subplot per severity, use a diverging colourmap per row
    fig, axes = plt.subplots(1, len(SEVERITY_LABELS),
                             figsize=(5.5*len(SEVERITY_LABELS), 3.5+len(num_cols)*0.42))
    sev_cmaps = {'Low':'Greens','Medium':'Oranges','High':'Reds'}

    for ax, sev in zip(axes, SEVERITY_LABELS):
        sdf    = t1[t1['Severity']==sev].set_index('Quartile')[num_cols]
        # Normalise with a minimum range floor so near-identical rows
        # stay visually pale rather than spanning the full colour range.
        # Floor = 10% of the row mean (or 0.5 absolute, whichever is larger).
        raw_range = sdf.max() - sdf.min()
        floor     = np.maximum(raw_range * 0, sdf.abs().mean() * 0.10).clip(lower=0.5)
        normed    = (sdf - sdf.min()) / (raw_range + floor)
        normed    = normed.clip(0, 1)
        cmap   = sev_cmaps.get(sev,'Blues')
        ax.imshow(normed.T, cmap=cmap, aspect='auto', vmin=0, vmax=1)
        ax.set_xticks(range(len(QUARTILES)))
        ax.set_xticklabels(QUARTILES, fontsize=11)
        ax.set_yticks(range(len(num_cols)))
        ax.set_yticklabels(num_cols, fontsize=9)
        sev_col = {'Low':'#1E8449','Medium':'#B7770D','High':'#922B21'}.get(sev,'#333')
        ax.set_title(f'{sev} Severity', fontweight='bold', fontsize=12,
                     color='white',
                     bbox=dict(boxstyle='round,pad=0.3', facecolor=sev_col))
        ax.set_xlabel('Quartile  (Q1 = most frequent care)', fontsize=9)
        for xi,q in enumerate(QUARTILES):
            for yi,col in enumerate(num_cols):
                if q in sdf.index:
                    val = sdf.loc[q,col]
                    bright = normed.loc[q,col] > 0.6
                    ax.text(xi,yi,f'{val}',ha='center',va='center',
                            fontsize=8.5,
                            color='white' if bright else '#1A1A1A',
                            fontweight='bold')

    fig.suptitle(
        f'{care_label} — Table 1: Patient Characteristics by Care Frequency Quartile\n'
        f'Colour intensity normalised per row. Uniform colour = balanced groups (no distribution shift).',
        fontsize=11, fontweight='bold', y=1.04)
    plt.tight_layout()
    path = os.path.join(OUTPUT_FOLDER, f'fig_table1_{care_label.lower()}.png')
    plt.savefig(path, bbox_inches='tight', dpi=150)
    plt.show(); print(f'Saved: {path}')
    return t1

# [unused figure removed] t1_mc   = table1_by_quartile(mc_data,   'Mouthcare')
# [unused figure removed] t1_turn = table1_by_quartile(turn_data, 'Turning')
# fig_table1_*.png not used in paper.


### Supplementary — Care Frequency Distribution

Distribution of `average_item_interval` per severity stratum with quartile boundary annotations. Confirms right-skewed distribution typical of ICU care event documentation and shows the inter-quartile contrasts underpinning the analysis.

In [ ]:

# ── Care Frequency Distribution Histograms
# Saves: fig_freq_dist_*.png
# Distribution of average_item_interval per severity stratum with quartile boundaries.
# Confirms right-skewed distribution and shows the spread underlying each quartile.
def plot_freq_distribution(data, care_label):
    from scipy.stats import normaltest
    SEV_HIST_COLORS = {'Low':'#2ECC71','Medium':'#F39C12','High':'#E74C3C'}
    Q_BOUNDARY_COLORS = ['#3498DB','#9B59B6','#E67E22']

    fig, axes = plt.subplots(1, len(SEVERITY_LABELS),
                             figsize=(5.5*len(SEVERITY_LABELS), 5.5), sharey=False)
    for ax, sev in zip(axes, SEVERITY_LABELS):
        sub  = data[data['severity']==sev][FREQ_COL].dropna()
        col  = SEV_HIST_COLORS.get(sev,'#5D6D7E')
        qs   = sub.quantile([0.25, 0.5, 0.75])
        q_range = qs[0.75] - qs[0.25]
        use_log = q_range < 2.0

        ax.hist(sub, bins=80, color=col, alpha=0.70, edgecolor='white', linewidth=0.3)
        ax.set_facecolor('#F8F9FA')
        if use_log:
            ax.set_xscale('log')
            ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'{x:.1f}'))

        for i, (boundary, bcol) in enumerate(zip(qs, Q_BOUNDARY_COLORS)):
            ax.axvline(boundary, color=bcol, ls='--', lw=2.0, alpha=0.95, zorder=5)

        ylim = ax.get_ylim()
        xlim = ax.get_xlim()
        x_span = xlim[1] - xlim[0]
        stagger_y = [0.93, 0.80, 0.67]   # vertical stagger to avoid overlap
        for i, (boundary, bcol, ypos) in enumerate(zip(qs, Q_BOUNDARY_COLORS, stagger_y)):
            if care_label == 'Turning':
                # Place label to the right of the line, avoiding overlap on narrow dist
                x_offset = x_span * 0.03
                ax.text(boundary + x_offset, ylim[1] * ypos,
                        f'Q{i+1}|Q{i+2} {boundary:.2f}h',
                        ha='left', fontsize=8, color=bcol, fontweight='bold', zorder=6,
                        bbox=dict(boxstyle='round,pad=0.2', facecolor='white',
                                  edgecolor=bcol, alpha=0.92))
            else:
                ax.text(boundary, ylim[1] * ypos,
                        f'Q{i+1}|Q{i+2}\n{boundary:.2f}h',
                        ha='center', fontsize=8.5, color=bcol, fontweight='bold', zorder=6,
                        bbox=dict(boxstyle='round,pad=0.2', facecolor='white',
                                  edgecolor=bcol, alpha=0.92))

        _, p_norm = normaltest(sub)
        skew_str = 'Right-skewed\n(normality p<0.001)' if p_norm<0.001 else f'Skewed (p={p_norm:.3f})'
        narrow_note = f'\n\u26a0 Narrow IQR ({q_range:.2f}h)\nLog x-axis used' if use_log else ''

        ax.set_title(f'{sev} Severity\n(n={len(sub):,})', fontweight='bold', fontsize=12,
                     color='white', bbox=dict(boxstyle='round,pad=0.3', facecolor=col))
        ax.set_xlabel('Mean inter-event interval (hours)', fontsize=10)
        ax.set_ylabel('Patient count', fontsize=10)
        ax.text(0.98, 0.97, skew_str+narrow_note, transform=ax.transAxes,
                ha='right', va='top', fontsize=8,
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='#CCCCCC', alpha=0.95))
        stats_str = (f'Mean = {sub.mean():.2f}h\nMedian = {sub.median():.2f}h\n'
                     f'IQR = {qs[0.25]:.2f}\u2013{qs[0.75]:.2f}h\n'
                     f'Q1 boundary: <{qs[0.25]:.2f}h\nQ4 boundary: >{qs[0.75]:.2f}h')
        ax.text(0.98, 0.68, stats_str, transform=ax.transAxes,
                ha='right', va='top', fontsize=7.5,
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='#CCCCCC', alpha=0.95))
        ax.grid(axis='y', alpha=0.3, linestyle='--')

    fig.suptitle(
        f'{care_label} \u2014 Distribution of Care Frequency (average_item_interval)\n'
        f'Coloured dashed lines = within-severity quartile boundaries. Log x-axis used when IQR < 2h.',
        fontsize=11, fontweight='bold', y=1.04)
    plt.tight_layout()
    path = os.path.join(OUTPUT_FOLDER, f'fig_freq_dist_{care_label.lower()}.png')
    plt.savefig(path, bbox_inches='tight', dpi=150)
    plt.show(); print(f'Saved: {path}')

    print(f'\n{care_label} \u2014 Care Frequency Summary:')
    print(f'{"Severity":10s}  {"Mean":>8s}  {"Median":>8s}  {"Q1 bdy":>10s}  {"Q4 bdy":>10s}  {"IQR":>8s}')
    print('-'*65)
    for sev in SEVERITY_LABELS:
        s = data[data['severity']==sev][FREQ_COL].dropna()
        qs2 = s.quantile([0.25,0.75])
        print(f'{sev:10s}  {s.mean():>7.2f}h  {s.median():>7.2f}h  '
              f'<{qs2[0.25]:>7.2f}h  >{qs2[0.75]:>7.2f}h  {qs2[0.75]-qs2[0.25]:>7.2f}h')

# [unused figure removed] plot_freq_distribution(mc_data,   'Mouthcare')
# [unused figure removed] plot_freq_distribution(turn_data, 'Turning')
# fig_freq_dist_*.png not used in paper.


### Supplementary — Monotonic Trend Analysis v2 (Superseded by Cell 71)

Alternative statistical summary table with Q1 vs Q4 as headline comparison and Kruskal-Wallis on race axis. **Superseded by cell 71** (publication stat summary organised by paper claim). Kept as independent cross-check of cell 31 DeLong results.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  SUPERSEDED BY CELL 71 (publication statistical summary)        ║
# ║  This is stat summary v2 — organised differently from cell 39,  ║
# ║  with Q1vsQ4 as headline and Kruskal-Wallis on race axis.       ║
# ║  Cell 71 is the final paper-organised version (by claim).       ║
# ║  This cell saves fig_stat_summary_table.png; cell 71 saves      ║
# ║  fig_pub_stat_summary.png. Both can coexist.                    ║
# ║  NOTE: This cell re-runs DeLong pairwise tests independently    ║
# ║  from cell 31 — results should match but computation is         ║
# ║  duplicated. Kept for cross-validation of cell 31 results.      ║
# ╚══════════════════════════════════════════════════════════════════╝

# ══════════════════════════════════════════════════════════════════════════════
# Statistical Summary Table
#
# QUARTILE axis (ordered — direction matters):
#   Headline : Q1 vs Q4 DeLong — the extreme "best vs worst care" comparison
#   Gradient : Q1 vs Q2, Q1 vs Q3 — how early does degradation appear?
#   Omnibus  : N of 6 pairwise DeLong significant (count, not list)
#   Trend    : Spearman ρ + JT p — supporting monotonicity evidence
#
# RACE axis (nominal — no direction, omnibus question):
#   Omnibus  : Kruskal-Wallis across all available race group AUROCs
#   Effect   : AUROC range (max − min across race groups)
#   N sig    : N of race pairs significant after Bonferroni correction
#              (with 10 pairs at α=0.05, expect 0.5 false positives uncorrected)
# ══════════════════════════════════════════════════════════════════════════════
from scipy.stats import spearmanr, kruskal
from itertools import combinations
from statsmodels.stats.multitest import multipletests

summary_rows = []

for care in ['Mouthcare', 'Turning']:
    for sev in SEVERITY_LABELS:
        for model_label, q_key, r_key, fold_key in [
            ('Model A', 'A – All Quartiles', 'A – By Race',    'A – All Quartiles'),
            ('Model B', 'B – Q1 Only',       'B – Q1 By Race', 'B – Q1 Only'),
        ]:
            row = {'Care': care, 'Severity': sev, 'Model': model_label}

            # ── Pull data ─────────────────────────────────────────────────────
            q_data = {g: (np.array(yt, dtype=float), np.array(yp, dtype=float))
                      for (cl,sl,mn,gt,g,yt,yp) in calibration_store
                      if cl==care and sl==sev and mn==q_key and gt=='quartile'
                      and len(np.unique(yt)) > 1}

            r_data = {g: (np.array(yt, dtype=float), np.array(yp, dtype=float))
                      for (cl,sl,mn,gt,g,yt,yp) in calibration_store
                      if cl==care and sl==sev and mn==r_key and gt=='race'
                      and len(np.unique(yt)) > 1}

            # ─────────────────────────────────────────────────────────────────
            # QUARTILE SECTION
            # ─────────────────────────────────────────────────────────────────
            avail_q = [q for q in QUARTILES if q in q_data]

            # Q1 vs Q4 — headline
            if 'Q1' in q_data and 'Q4' in q_data:
                auc1 = float(roc_auc_score(*q_data['Q1']))
                auc4 = float(roc_auc_score(*q_data['Q4']))
                _, _, z14, p14 = delong_independent(
                    q_data['Q1'][0], q_data['Q1'][1],
                    q_data['Q4'][0], q_data['Q4'][1])
                row['Q1 AUROC']   = round(auc1, 3)
                row['Q4 AUROC']   = round(auc4, 3)
                row['Q1−Q4 Δ']    = round(auc1 - auc4, 3)
                row['Q1vsQ4 z']   = round(z14, 2) if not np.isnan(z14) else None
                row['Q1vsQ4 p']   = round(p14, 4) if not np.isnan(p14) else None
            else:
                row.update({'Q1 AUROC': None, 'Q4 AUROC': None,
                            'Q1−Q4 Δ': None, 'Q1vsQ4 z': None, 'Q1vsQ4 p': None})

            # Q1 vs Q2 and Q1 vs Q3 — gradient
            for q_other, col_p in [('Q2','Q1vsQ2 p'), ('Q3','Q1vsQ3 p')]:
                if 'Q1' in q_data and q_other in q_data:
                    _, _, _, p = delong_independent(
                        q_data['Q1'][0], q_data['Q1'][1],
                        q_data[q_other][0], q_data[q_other][1])
                    row[col_p] = round(p, 4) if not np.isnan(p) else None
                else:
                    row[col_p] = None

            # N of 6 pairs significant (uncorrected) — omnibus count
            all_q_pairs = list(combinations(avail_q, 2))
            n_sig_q = 0
            for q1, q2 in all_q_pairs:
                _, _, _, p = delong_independent(
                    q_data[q1][0], q_data[q1][1],
                    q_data[q2][0], q_data[q2][1])
                if not np.isnan(p) and p < 0.05:
                    n_sig_q += 1
            row['Q sig pairs'] = f'{n_sig_q}/{len(all_q_pairs)}'

            # Spearman ρ
            q_aucs = [(i+1, float(roc_auc_score(*q_data[q])))
                      for i, q in enumerate(QUARTILES) if q in q_data]
            if len(q_aucs) >= 3:
                rho, p_sp = spearmanr([x[0] for x in q_aucs],
                                       [x[1] for x in q_aucs])
                row['Spearman ρ'] = round(rho, 3)
                row['Spearman p'] = round(p_sp, 4)
            else:
                row['Spearman ρ'] = None
                row['Spearman p'] = None

            # JT trend — uses fold_aurocs_store built in cell 30
            _fs = globals().get('fold_aurocs_store', {})
            obs = [_fs.get((care, sev, fold_key, q), np.array([]))
                   for q in QUARTILES]
            obs = [a for a in obs if len(a) >= 2]
            if len(obs) >= 3:
                jt, p_jt = jonckheere_trend_test(obs)
                row['JT p'] = round(p_jt, 4) if not np.isnan(p_jt) else None
            else:
                row['JT p'] = None

            # ─────────────────────────────────────────────────────────────────
            # RACE SECTION
            # ─────────────────────────────────────────────────────────────────
            avail_r = [r for r in RACE_LABELS if r in r_data]
            race_aucs = [float(roc_auc_score(*r_data[r])) for r in avail_r]

            # AUROC range across race groups — effect size
            if len(race_aucs) >= 2:
                row['Race AUROC range'] = round(max(race_aucs) - min(race_aucs), 3)
            else:
                row['Race AUROC range'] = None

            # Kruskal-Wallis omnibus across race group bootstrap AUROC samples
            # Use bootstrap samples as observations (1000 per group)
            # This gives the omnibus "is there any race variation?" answer
            if len(avail_r) >= 2:
                boot_samples = []
                for r in avail_r:
                    yt_r, yp_r = r_data[r]
                    rng = np.random.default_rng(42)
                    idx = np.arange(len(yt_r))
                    s = []
                    for _ in range(200):   # 200 resamples per group (fast)
                        b = rng.choice(idx, size=len(idx), replace=True)
                        if len(np.unique(yt_r[b])) > 1:
                            s.append(roc_auc_score(yt_r[b], yp_r[b]))
                    boot_samples.append(s)
                valid = [s for s in boot_samples if len(s) > 0]
                if len(valid) >= 2:
                    kw_stat, kw_p = kruskal(*valid)
                    row['Race KW p'] = round(kw_p, 4)
                else:
                    row['Race KW p'] = None
            else:
                row['Race KW p'] = None

            # N pairs significant after Bonferroni correction
            if len(avail_r) >= 2:
                race_pairs = list(combinations(avail_r, 2))
                raw_ps = []
                for r1, r2 in race_pairs:
                    _, _, _, p = delong_independent(
                        r_data[r1][0], r_data[r1][1],
                        r_data[r2][0], r_data[r2][1])
                    raw_ps.append(p if not np.isnan(p) else 1.0)
                if raw_ps:
                    reject, _, _, _ = multipletests(raw_ps, method='bonferroni')
                    n_sig_r = int(reject.sum())
                    row['Race_sig_pairs_Bonf'] = f'{n_sig_r}/{len(race_pairs)}'
                else:
                    row['Race_sig_pairs_Bonf'] = None
            else:
                row['Race_sig_pairs_Bonf'] = None

            summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)

# ── Print to console ──────────────────────────────────────────────────────────
def _fmt_p(v):
    if v is None or (isinstance(v, float) and np.isnan(v)): return '—'
    if v < 0.001: return '<0.001***'
    if v < 0.01:  return f'{v:.3f}**'
    if v < 0.05:  return f'{v:.3f}*'
    return f'{v:.3f}'

print(f"\n{'='*90}")
print("  STATISTICAL SUMMARY (Low & Medium severity are primary strata)")
print(f"{'='*90}")
for _, r in summary_df.iterrows():
    print(f"\n  {r['Care']} | {r['Severity']} | {r['Model']}")
    print(f"    QUARTILE: Q1={r['Q1 AUROC']}  Q4={r['Q4 AUROC']}  "
          f"Δ={r['Q1−Q4 Δ']}  Q1vsQ4 {_fmt_p(r['Q1vsQ4 p'])}  "
          f"({r['Q sig pairs']} pairs sig)")
    print(f"             Q1vsQ2 {_fmt_p(r['Q1vsQ2 p'])}  Q1vsQ3 {_fmt_p(r['Q1vsQ3 p'])}  "
          f"Spearman ρ={r['Spearman ρ']} {_fmt_p(r['Spearman p'])}  "
          f"JT {_fmt_p(r['JT p'])}")
    print(f"    RACE:     range={r['Race AUROC range']}  KW {_fmt_p(r['Race KW p'])}  "
          f"{r['Race_sig_pairs_Bonf']} sig pairs (Bonferroni)")

# ── Matplotlib table ──────────────────────────────────────────────────────────
DISP_COLS = [
    'Care','Severity','Model',
    # Quartile
    'Q1 AUROC','Q4 AUROC','Q1−Q4 Δ','Q1vsQ4 p',
    'Q1vsQ2 p','Q1vsQ3 p','Q sig pairs',
    'Spearman ρ','Spearman p','JT p',
    # Race
    'Race AUROC range','Race KW p','Race_sig_pairs_Bonf',
]
DISP_COLS = [c for c in DISP_COLS if c in summary_df.columns]

# Format p-value columns
p_cols = ['Q1vsQ4 p','Q1vsQ2 p','Q1vsQ3 p','Spearman p','JT p','Race KW p']
disp_df = summary_df.copy()
for col in p_cols:
    if col in disp_df.columns:
        disp_df[col] = disp_df[col].apply(_fmt_p)

fig_h = max(5, 0.5 * len(disp_df) + 2.5)
fig, ax = plt.subplots(figsize=(24, fig_h))
ax.axis('off')

tbl_data = [[str(disp_df.iloc[i][c]) if disp_df.iloc[i][c] is not None else '—'
             for c in DISP_COLS]
            for i in range(len(disp_df))]

tbl = ax.table(cellText=tbl_data, colLabels=DISP_COLS,
               loc='center', cellLoc='center')
tbl.auto_set_font_size(False)
tbl.set_fontsize(8.5)
tbl.auto_set_column_width(col=list(range(len(DISP_COLS))))

# Header row
for j in range(len(DISP_COLS)):
    tbl[(0, j)].set_facecolor('#1F3864')
    tbl[(0, j)].set_text_props(color='white', fontweight='bold')

# Highlight Q1vsQ4 p column header (primary)
if 'Q1vsQ4 p' in DISP_COLS:
    tbl[(0, DISP_COLS.index('Q1vsQ4 p'))].set_facecolor('#922B21')

# Section divider: shade quartile cols vs race cols differently in header
race_start = DISP_COLS.index('Race AUROC range') if 'Race AUROC range' in DISP_COLS else None
if race_start:
    for j in range(race_start, len(DISP_COLS)):
        tbl[(0, j)].set_facecolor('#1A5276')

# Row colouring
for i in range(1, len(disp_df) + 1):
    r = summary_df.iloc[i - 1]
    is_mb  = r['Model'] == 'Model B'
    q14_p  = r.get('Q1vsQ4 p')
    q14_sig = q14_p is not None and not (isinstance(q14_p, float) and np.isnan(q14_p)) and q14_p < 0.05

    if is_mb and q14_sig:
        bg = '#FDEDEC'      # red — Model B significant Q1vsQ4 (hypothesis confirmed)
    elif not is_mb and not q14_sig:
        bg = '#EAFAF1'      # green — Model A not significant (expected)
    elif r['Care'] == 'Mouthcare':
        bg = '#EBF5FB'
    else:
        bg = '#FDFEFE'

    for j in range(len(DISP_COLS)):
        tbl[(i, j)].set_facecolor(bg)
    # Bold primary test cell
    if 'Q1vsQ4 p' in DISP_COLS:
        tbl[(i, DISP_COLS.index('Q1vsQ4 p'))].set_text_props(fontweight='bold')

fig.suptitle(
    'Statistical Summary — Quartile Degradation (left) vs Race Stability (right)\n'
    'Quartile: Q1vsQ4 DeLong = primary  |  Spearman ρ + JT = secondary trend evidence\n'
    'Race: Kruskal-Wallis omnibus + N pairs sig after Bonferroni correction\n'
    'Green = Model A non-sig Q1vsQ4 (expected)  |  Red = Model B sig Q1vsQ4 (hypothesis confirmed)\n'
    '* p<0.05  ** p<0.01  *** p<0.001',
    fontsize=10.5, fontweight='bold', y=1.06
)
plt.tight_layout()
# [unused figure removed] path = fig_stat_summary_table.png
# [unused figure removed] plt.savefig(...)
# [unused figure removed] plt.show()
# Superseded by cell 71 publication stat summary.


### Sensitivity Analysis — Sample Size Control (Downsampled Model A)

To isolate whether observed performance differences between Model A and Model B reflect the care quality of the training cohort rather than the difference in training sample size, a size-matched variant (Model A_down) is trained on a random subsample of all-quartile data equal in size to Model B's Q1 training set, using identical five-fold CV splits. If Model A_down still outperforms Model B at Q2/Q3/Q4 in the strata where degradation is strongest, the performance gap is attributable to training environment — the mortality patterns learned from a high-care population — rather than to data volume.

In [ ]:

# ── Downsampled Model A sensitivity — 5-fold CV, consistent with main pipeline
# Trains a size-matched Model A (downsampled to Q1 n) and compares to Model B.
# Uses same StratifiedKFold splits so results are directly comparable.
print('='*70)
print('  Sensitivity Analysis — Downsampled Model A (matched n to Model B Q1)')
print('  Using 5-fold CV for consistency with main pipeline')
print('='*70)

SEV_COLORS = {'Low':'#2ECC71','Medium':'#F39C12','High':'#E74C3C'}
SEV_DARK   = {'Low':'#1E8449','Medium':'#B7770D','High':'#922B21'}
COLOR_ADOWN = '#17A589'
COLOR_B     = '#C0392B'
COLOR_A     = '#2E86C1'

ds_rows = []

for data_in, feats_in, care_label in [(mc_data, mc_feats, 'Mouthcare'),
                                       (turn_data, turn_feats, 'Turning')]:
    for sev in SEVERITY_LABELS:
        sub = data_in[data_in['severity'] == sev].copy()
        sub[MORTALITY] = sub[MORTALITY].fillna(0).astype(int)
        for _fc in feats_in:
            if _fc in sub.columns:
                sub[_fc] = sub[_fc].astype(float)
        if len(sub) < 80:
            continue

        # ── Same CV splits as main pipeline ──────────────────────────────────
        strat_key = sub['quartile'].astype(str) + '_' + sub[MORTALITY].astype(str)
        if strat_key.value_counts().min() < N_FOLDS:
            strat_key = sub[MORTALITY].astype(str)
        skf_c1  = StratifiedKFold(n_splits=N_FOLDS, shuffle=True,
                                   random_state=RANDOM_STATE)
        sub_arr = sub.reset_index(drop=True)

        fold_preds_Ad = {q: [] for q in QUARTILES}

        for fold_i, (tr_idx, te_idx) in enumerate(
                skf_c1.split(sub_arr, strat_key.values)):

            train_fold = sub_arr.iloc[tr_idx]
            test_fold  = sub_arr.iloc[te_idx]

            q1_tr  = train_fold[train_fold['quartile'] == 'Q1']
            n_q1   = len(q1_tr)
            if n_q1 < 20:
                continue

            # Downsample all-Q train to match Q1 n
            rng        = np.random.default_rng(RANDOM_STATE + fold_i)
            train_down = train_fold.iloc[
                rng.choice(len(train_fold), size=n_q1, replace=False)]
            med_d = train_down[feats_in].fillna(
                train_down[feats_in].median()).median()

            try:
                model_Ad = train_xgb(
                    train_down[feats_in].fillna(med_d), train_down[MORTALITY])
            except Exception as e:
                print(f'  {care_label}|{sev} fold {fold_i}: A_down failed: {e}')
                continue

            for q in QUARTILES:
                te_q = test_fold[test_fold['quartile'] == q]
                if len(te_q) < 5 or te_q[MORTALITY].nunique() < 2:
                    continue
                yp = model_Ad.predict_proba(te_q[feats_in].fillna(med_d))[:, 1]
                fold_preds_Ad[q].append((te_q[MORTALITY].values, yp))

        # ── Aggregate A_down across folds ────────────────────────────────────
        print(f'\n  {care_label} | {sev}')
        for q in QUARTILES:
            if not fold_preds_Ad[q]:
                continue
            row_d, _, _ = _aggregate_cv_predictions(fold_preds_Ad[q])
            if row_d is None:
                continue

            b_match = results_df[
                (results_df['Care Type'] == care_label) &
                (results_df['Severity']  == sev) &
                (results_df['Model']     == 'B – Q1 Only') &
                (results_df['Test Set'].str.contains(q))
            ]
            auc_b   = b_match['AUROC'].values[0]    if len(b_match) > 0 else np.nan
            ci_b_lo = b_match['AUROC_lo'].values[0] if len(b_match) > 0 else np.nan
            ci_b_hi = b_match['AUROC_hi'].values[0] if len(b_match) > 0 else np.nan
            diff    = round(row_d['AUROC'] - auc_b, 3) if (
                row_d['AUROC'] and not np.isnan(auc_b)) else np.nan

            print(f'    {q}: A_down={row_d["AUROC"]:.3f} '
                  f'({row_d["AUROC_lo"]:.3f}–{row_d["AUROC_hi"]:.3f})  '
                  f'B={auc_b:.3f}  diff={diff:+.3f}')

            n_q1_avg = int(np.mean([len(f[0]) for f in fold_preds_Ad[q]]) * N_FOLDS)
            ds_rows.append({
                'Care': care_label, 'Severity': sev, 'Quartile': q,
                'AUC_Adown': row_d['AUROC'],
                'Ad_lo':     row_d['AUROC_lo'],
                'Ad_hi':     row_d['AUROC_hi'],
                'AUC_B':     auc_b, 'B_lo': ci_b_lo, 'B_hi': ci_b_hi,
                'Diff': diff, 'n': n_q1_avg,
            })

ds_df = pd.DataFrame(ds_rows)

# ── Line graph 2×3 ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(17, 11))

for row_idx, care in enumerate(['Mouthcare', 'Turning']):
    for col_idx, sev in enumerate(SEVERITY_LABELS):
        ax  = axes[row_idx, col_idx]
        sub = ds_df[(ds_df['Care']==care) & (ds_df['Severity']==sev)].copy()
        if len(sub) == 0:
            ax.set_visible(False)
            continue

        sub = sub.sort_values('Quartile')
        x   = np.arange(len(sub))

        # Model A full reference
        a_full = results_df[
            (results_df['Care Type'] == care) &
            (results_df['Severity']  == sev) &
            (results_df['Model']     == 'A – All Quartiles') &
            (results_df['Test Set'].str.contains('Q'))
        ].copy()
        a_full['q'] = a_full['Test Set'].str.extract(r'(Q\d)')
        a_full = a_full.dropna(subset=['q','AUROC']).sort_values('q')
        if len(a_full) == len(sub):
            af  = a_full['AUROC'].values.astype(float)
            alo = a_full['AUROC_lo'].fillna(a_full['AUROC']).values.astype(float)
            ahi = a_full['AUROC_hi'].fillna(a_full['AUROC']).values.astype(float)
            ax.errorbar(x, af, yerr=[af-alo, ahi-af],
                        color=COLOR_A, marker='D', linestyle=':', linewidth=1.8,
                        markersize=7, capsize=4, capthick=1.8, ecolor=COLOR_A,
                        alpha=0.65, label='Model A (full training)', zorder=2)

        # A_down
        ad  = sub['AUC_Adown'].values.astype(float)
        ado = sub['Ad_lo'].values.astype(float)
        adh = sub['Ad_hi'].values.astype(float)
        ax.errorbar(x, ad, yerr=[ad-ado, adh-ad],
                    color=COLOR_ADOWN, marker='o', linestyle='-', linewidth=2.4,
                    markersize=9, capsize=5, capthick=2, ecolor=COLOR_ADOWN,
                    alpha=0.92, label=f'A_down (n≈{sub["n"].iloc[0]})', zorder=4)

        # Model B
        b   = sub['AUC_B'].values.astype(float)
        blo = sub['B_lo'].values.astype(float)
        bhi = sub['B_hi'].values.astype(float)
        ax.errorbar(x, b, yerr=[b-blo, bhi-b],
                    color=COLOR_B, marker='s', linestyle='--', linewidth=2.4,
                    markersize=9, capsize=5, capthick=2, ecolor=COLOR_B,
                    alpha=0.92, label='Model B (Q1-only training)', zorder=3)

        for xi in range(len(sub)):
            ax.annotate(f'{ad[xi]:.3f}', (xi, adh[xi]),
                        textcoords='offset points', xytext=(0, 7),
                        ha='center', fontsize=8, color=COLOR_ADOWN, fontweight='bold')
            ax.annotate(f'{b[xi]:.3f}',  (xi, blo[xi]),
                        textcoords='offset points', xytext=(0, -13),
                        ha='center', fontsize=8, color=COLOR_B, fontweight='bold')

        ax.axhline(0.5, color='grey', ls=':', lw=1, alpha=0.4)
        ax.set_facecolor('#F8F9FA')
        ax.set_xticks(x)
        ax.set_xticklabels(['Q1\n(most freq)','Q2','Q3','Q4\n(least freq)'], fontsize=9)
        ax.set_ylim(0.40, 1.02)
        ax.set_ylabel('AUROC (95% CI)', fontsize=9)
        ax.set_xlabel('Test Quartile', fontsize=9)
        ax.grid(axis='y', alpha=0.3, linestyle='--')
        ax.legend(fontsize=8, loc='lower left', framealpha=0.92)
        sev_col = SEV_DARK.get(sev, '#333333')
        ax.set_title(f'{care} — {sev} Severity',
                     fontweight='bold', fontsize=11, color='white',
                     bbox=dict(boxstyle='round,pad=0.3', facecolor=sev_col))

fig.suptitle(
    'Sensitivity Analysis — Downsampled Model A (teal) vs Model B (red) vs Model A full (blue, dashed)\n'
    'All models use 5-fold CV, identical splits. Error bars = 95% CI (mean ± 1.96×SE across folds).\n'
    'Teal above red at Q2–Q4 → care environment, not sample size, drives degradation.',
    fontsize=11, fontweight='bold', y=1.03)
plt.tight_layout()
path = os.path.join(OUTPUT_FOLDER, 'fig_sensitivity_downsample.png')
plt.savefig(path, bbox_inches='tight', dpi=150)
plt.show()
print(f'Saved: {path}')


## Publication Statistical Summary Table — **Supplementary Table 2**

Full statistical summary organised by the four paper claims:
- **Section A** — Design Validation: SOFA balance (KW test)
- **Section B** — Care-Frequency Degradation: pairwise DeLong (uniformity across all 6 pairs, Bonferroni), directional JT (degradation trend Q1→Q4, bootstrap inputs), Spearman ρ
- **Section C** — Race Stability: DeLong race pairs (Bonferroni)
- **Section D** — Variance Decomposition: Q-range vs R-range ratio

Note: DeLong tests whether groups are statistically different (uniformity). The directional JT tests whether AUROC decreases monotonically Q1→Q4 (degradation).

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PUBLICATION STATISTICAL SUMMARY TABLE
# Organised by the 4 paper claims, not by test execution order.
#
# Column grouping:
#   Section A — Design Validation (Claim 1)
#   Section B — Care-Frequency Degradation: Model B (Claim 2)
#   Section C — Demographic Audit Stability: Model B (Claim 3)
#   Section D — Variance Decomposition (Claim 4)
#
# One row per (Care × Severity).
# Model A values shown where they strengthen the argument (Claim 4 contrast).
# ══════════════════════════════════════════════════════════════════════════════

def _p(v, one_sided=False):
    """Format p-value with significance stars."""
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return '—'
    if v < 0.001: return '<0.001***'
    if v < 0.01:  return f'{v:.3f}**'
    if v < 0.05:  return f'{v:.3f}*'
    return f'{v:.3f}'

def _d(v, decimals=3):
    """Format a numeric value."""
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return '—'
    return f'{v:.{decimals}f}'

pub_rows = []

for care in ['Mouthcare', 'Turning']:
    for sev in SEVERITY_LABELS:
        res = stat_results.get((care, sev), {})
        vd  = res.get('var_decomp', {})

        # Pull calibration data for AUROC values
        qA = _pull_cal(care, sev, 'A – All Quartiles', 'quartile')
        qB = _pull_cal(care, sev, 'B – Q1 Only',       'quartile')
        rA = _pull_cal(care, sev, 'A – By Race',        'race')
        rB = _pull_cal(care, sev, 'B – Q1 By Race',     'race')

        def _auc(d, g):
            try: return round(float(roc_auc_score(*d[g])), 3)
            except: return None

        RACE_EX = [r for r in RACE_LABELS if r not in ('Other','Others','Other/Unknown')]

        row = {
            # ── Identifiers ──────────────────────────────────────────────────
            'Care':     care,
            'Severity': sev,

            # ── SECTION A: Design Validation ─────────────────────────────────
            'A1. SOFA balance (KW p)':        _p(res.get('sofa_kw_p')),

            # ── SECTION B: Claim 2 — Model B care-frequency degradation ──────
            'B1. Model B Q1 AUROC':  _d(res.get('modelB_q1_auroc')),
            'B1. Model B Q4 AUROC':  _d(res.get('modelB_q4_auroc')),
            'B2. Q1−Q4 Δ (PRIMARY)': _d(res.get('modelB_q1q4_delta')),
            'B2. Q1vsQ4 (DeLong adj p)':
                _p(res.get('modelB_q1q4_p_adj')),
            'B3. Sig pairs (B, Bonf)':
                f"{res.get('modelB_q_n_sig_bonf','—')}/6",
            'B4. Spearman ρ (effect size)':
                _d(res.get('modelB_q_spearman_rho')),
            'B5. JT trend (p)': _p(res.get('modelB_q_jt_p')),

            # ── SECTION C: Claim 3 — Model B race stability ───────────────────
            'C1. Model B race AUROC range':   _d(res.get('modelB_r_range')),
            'C2. Sig pairs (B race, Bonf)':
                f"{res.get('modelB_r_n_sig_bonf','—')}/"
                f"{len(list(__import__('itertools').combinations([r for r in RACE_LABELS if r in rB], 2)))}",

            # ── SECTION D: Claim 4 — Care explains more variance than race ────
            'D1. Model B Q_range':
                _d(vd.get('B_q_range')),
            'D1. Model B R_range (excl Other)':
                _d(res.get('modelB_r_range')),
            'D2. Q/R ratio (Model B)': (
                _d(round(vd['B_q_range'] / res['modelB_r_range'], 2), 2)
                if vd.get('B_q_range') and res.get('modelB_r_range')
                   and res['modelB_r_range'] > 0 else '—'
            ),
            'D3. Model A race range':  _d(res.get('modelA_r_range')),
        }
        pub_rows.append(row)

pub_df = pd.DataFrame(pub_rows)

# ── Console print ─────────────────────────────────────────────────────────────
print('\nPUBLICATION STATISTICAL SUMMARY')
print('Organised by paper claim — see column sections A / B / C / D')
print('='*130)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 400)
pd.set_option('display.max_colwidth', 20)
pub_df_print = pub_df.rename(columns={c: c.replace('\n',' ') for c in pub_df.columns})
print(pub_df_print.to_string(index=False))

# ── Matplotlib publication table ─────────────────────────────────────────────
DISP = list(pub_df.columns)
clean_labels = [c.replace('\n', ' ') for c in DISP]

# Section boundary column indices — now match actual column names
SEC_A_START = DISP.index('A1. SOFA balance (KW p)')
SEC_B_START = DISP.index('B1. Model B Q1 AUROC')
SEC_C_START = DISP.index('C1. Model B race AUROC range')
SEC_D_START = DISP.index('D1. Model B Q_range')

SEC_COLORS = {
    'ID':  '#2C3E50',
    'A':   '#1A5276',
    'B':   '#922B21',
    'C':   '#1E8449',
    'D':   '#6C3483',
}

def _sec(col_idx):
    if col_idx < 2:            return 'ID'
    if col_idx < SEC_B_START:  return 'A'
    if col_idx < SEC_C_START:  return 'B'
    if col_idx < SEC_D_START:  return 'C'
    return 'D'

SEC_LABELS = {
    'ID': '',
    'A':  'SECTION A — Design Validation (Claim 1)',
    'B':  'SECTION B — Model B Care Degradation (Claim 2)',
    'C':  'SECTION C — Model B Race Stability (Claim 3)',
    'D':  'SECTION D — Variance Decomposition (Claim 4)',
}

n_cols = len(DISP)
n_rows = len(pub_df)
fig_w  = max(28, n_cols * 1.6)
fig_h  = max(6,  n_rows * 0.55 + 3.5)

fig, ax = plt.subplots(figsize=(fig_w, fig_h))
ax.axis('off')

cell_text = [
    [str(pub_df.iloc[i][c]) if pub_df.iloc[i][c] is not None else '—'
     for c in DISP]
    for i in range(n_rows)
]

tbl = ax.table(
    cellText=cell_text,
    colLabels=clean_labels,
    loc='center', cellLoc='center'
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(8)
tbl.auto_set_column_width(col=list(range(n_cols)))

# ── Header row colouring by section ──────────────────────────────────────────
for j, col in enumerate(DISP):
    sec = _sec(j)
    tbl[(0, j)].set_facecolor(SEC_COLORS[sec])
    tbl[(0, j)].set_text_props(color='white', fontweight='bold', fontsize=7.5)

# ── Data row colouring ────────────────────────────────────────────────────────
pri_col = 'B2. Q1vsQ4 (DeLong adj p)'
for i in range(1, n_rows + 1):
    r = pub_df.iloc[i - 1]
    for j in range(n_cols):
        sec = _sec(j)
        tints = {'ID': '#ECF0F1', 'A': '#EAF2FF', 'B': '#FDEDEC',
                 'C': '#EAFAF1', 'D': '#F5EEF8'}
        tbl[(i, j)].set_facecolor(tints[sec])

    if pri_col in DISP:
        tbl[(i, DISP.index(pri_col))].set_text_props(fontweight='bold')

    pri_val = r.get(pri_col, '')
    if isinstance(pri_val, str) and '*' in str(pri_val):
        if pri_col in DISP:
            tbl[(i, DISP.index(pri_col))].set_facecolor('#FADBD8')

# ── Section header bar above the table ───────────────────────────────────────
ax_bbox = ax.get_position()
prev_sec = None
sec_start_j = 0
for j in range(n_cols + 1):
    cur_sec = _sec(j) if j < n_cols else 'END'
    if cur_sec != prev_sec and prev_sec is not None and prev_sec != 'ID':
        mid = (sec_start_j + j - 1) / 2 / n_cols
        ax.annotate(
            SEC_LABELS.get(prev_sec, ''),
            xy=(mid, 1.08), xycoords='axes fraction',
            ha='center', va='bottom', fontsize=9,
            fontweight='bold', color=SEC_COLORS[prev_sec],
            annotation_clip=False
        )
    if cur_sec != prev_sec:
        sec_start_j = j
    prev_sec = cur_sec

fig.suptitle(
    'Statistical Summary — Organised by Paper Claim\n'
    'Section A: Design Validation  |  Section B: Model B Care Degradation  |  '
    'Section C: Model B Race Stability  |  Section D: Variance Decomposition\n'
    'Primary evidence: Model B pairwise DeLong gradient (B2-B3) + JT trend (B5)  |  '
    'Spearman ρ = descriptive effect size (n=4)  |  '
    '* p<0.05  ** p<0.01  *** p<0.001',
    fontsize=10, fontweight='bold', y=1.12
)
plt.tight_layout()
path = os.path.join(OUTPUT_FOLDER, 'fig_pub_stat_summary.png')
plt.savefig(path, bbox_inches='tight', dpi=150)
plt.show()
print(f'Saved: {path}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Balanced AUROC + DeltaAUROC Panels — all 4 configs, Low, Medium, and High
# Subsampled to n_min per stratum, N_ITER=500, descriptive only (no stats)
# Delta style matches plot_delta_auroc_panel: bars + error bars + value labels
# ═══════════════════════════════════════════════════════════════════════════════
from sklearn.metrics import roc_auc_score
import numpy as np
import matplotlib.pyplot as plt

N_ITER  = 500
SEED    = 42
rng_pan = np.random.default_rng(SEED)

RACE_COLS = {
    'White': '#2874A6', 'Black': '#E74C3C',
    'Hispanic/Latino': '#27AE60', 'Asian': '#F39C12'
}
RACE_LIST = [r for r in RACE_LABELS if r != 'Other']
Q_COLS    = {'Q1': '#1A5276', 'Q2': '#2874A6',
             'Q3': '#E67E22', 'Q4': '#C0392B'}

SEV_COL = {'Low': '#1E8449', 'Medium': '#B7770D', 'High': '#922B21'}

for care_label in ['Mouthcare', 'Turning']:
    for sev in ['Low', 'Medium', 'High']:

        # ── Find n_min ────────────────────────────────────────────────────
        ns = []
        for r_key in ['A – By Race', 'B – Q1 By Race']:
            for r in RACE_LIST:
                entry = next(((yt, yp) for (cl, sl, mn, gt, g, yt, yp)
                              in calibration_store
                              if cl == care_label and sl == sev and mn == r_key
                              and gt == 'race' and g == r
                              and len(np.unique(yt)) > 1), None)
                if entry:
                    ns.append(len(entry[0]))
        if not ns:
            print(f'Skipping {care_label} {sev}: no race data found')
            continue
        n_min = min(ns)
        print(f'\n{care_label} | {sev} | n_min = {n_min}')

        # ── Compute balanced race AUROCs + deltas ─────────────────────────
        bal_aucs   = {}   # (model_name, race) → (mean, ci_lo, ci_hi)
        delta_aucs = {}   # (model_name, race) → (mean_delta, ci_lo, ci_hi)
        cohort_ref = {}   # model_name → mean full-cohort AUROC at balanced n

        for model_name, r_key, q_key in [
            ('Model A', 'A – By Race',    'A – All Quartiles'),
            ('Model B', 'B – Q1 By Race', 'B – Q1 Only')
        ]:
            # Full cohort pool (all quartile predictions combined)
            all_yt, all_yp = [], []
            for q in QUARTILES:
                entry = next(((yt, yp) for (cl, sl, mn, gt, g, yt, yp)
                              in calibration_store
                              if cl == care_label and sl == sev and mn == q_key
                              and gt == 'quartile' and g == q
                              and len(np.unique(yt)) > 1), None)
                if entry:
                    all_yt.extend(entry[0])
                    all_yp.extend(entry[1])
            all_yt = np.array(all_yt, dtype=float)
            all_yp = np.array(all_yp, dtype=float)

            # Bootstrap full cohort AUROC at balanced n (n_min * 4 to match
            # roughly the same total as 4 race groups each at n_min)
            boot_ref = []
            for _ in range(N_ITER):
                if len(all_yt) < n_min * 4:
                    idx = rng_pan.choice(len(all_yt), size=len(all_yt),
                                         replace=True)
                else:
                    idx = rng_pan.choice(len(all_yt), size=n_min * 4,
                                         replace=False)
                yt_s, yp_s = all_yt[idx], all_yp[idx]
                if len(np.unique(yt_s)) < 2:
                    continue
                boot_ref.append(roc_auc_score(yt_s, yp_s))
            ref_mean = np.mean(boot_ref) if boot_ref else np.nan
            cohort_ref[model_name] = ref_mean

            # Race group balanced AUROCs and deltas
            for r in RACE_LIST:
                entry = next(((yt, yp) for (cl, sl, mn, gt, g, yt, yp)
                              in calibration_store
                              if cl == care_label and sl == sev and mn == r_key
                              and gt == 'race' and g == r
                              and len(np.unique(yt)) > 1), None)
                if not entry:
                    continue
                yt_a = np.array(entry[0], dtype=float)
                yp_a = np.array(entry[1], dtype=float)
                n    = len(yt_a)

                if n < n_min:
                    print(f'  Skipping {r}: n={n} < n_min={n_min}')
                    continue

                boot, boot_delta = [], []
                for _ in range(N_ITER):
                    idx = rng_pan.choice(n, size=n_min, replace=False)
                    yt_s, yp_s = yt_a[idx], yp_a[idx]
                    if len(np.unique(yt_s)) < 2:
                        continue
                    a = roc_auc_score(yt_s, yp_s)
                    boot.append(a)
                    # Reference from same iteration (matched n)
                    ref_size = min(n_min * 4, len(all_yt))
                    idx_r = rng_pan.choice(len(all_yt), size=ref_size,
                                           replace=False)
                    yt_r, yp_r = all_yt[idx_r], all_yp[idx_r]
                    if len(np.unique(yt_r)) >= 2:
                        boot_delta.append(a - roc_auc_score(yt_r, yp_r))

                if boot:
                    bal_aucs[(model_name, r)] = (
                        np.mean(boot),
                        np.percentile(boot,  2.5),
                        np.percentile(boot, 97.5))
                if boot_delta:
                    delta_aucs[(model_name, r)] = (
                        np.mean(boot_delta),
                        np.percentile(boot_delta,  2.5),
                        np.percentile(boot_delta, 97.5))

        # ── Get quartile AUROCs and deltas (full n) ───────────────────────
        q_aucs   = {}   # (model_name, q) → auc
        q_deltas = {}   # (model_name, q) → delta

        for model_name, q_key in [('Model A', 'A – All Quartiles'),
                                   ('Model B', 'B – Q1 Only')]:
            ref = cohort_ref.get(model_name, np.nan)
            for q in QUARTILES:
                entry = next(((yt, yp) for (cl, sl, mn, gt, g, yt, yp)
                              in calibration_store
                              if cl == care_label and sl == sev and mn == q_key
                              and gt == 'quartile' and g == q
                              and len(np.unique(yt)) > 1), None)
                if entry:
                    a = float(roc_auc_score(*entry))
                    q_aucs[(model_name, q)]   = a
                    q_deltas[(model_name, q)] = a - ref

        # ══════════════════════════════════════════════════════════════════
        # FIGURE 1 — Balanced AUROC (raw values)
        # ══════════════════════════════════════════════════════════════════
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))

        panel_specs = [
            (0, 0, 'Model A', 'race',
             f'Config 1: Model A → Race\n(balanced n={n_min})'),
            (0, 1, 'Model A', 'quartile',
             'Config 2: Model A → Quartile'),
            (1, 0, 'Model B', 'race',
             f'Config 4: Model B → Race\n(balanced n={n_min})'),
            (1, 1, 'Model B', 'quartile',
             'Config 3: Model B → Quartile'),
        ]

        for row, col, model_name, axis, title in panel_specs:
            ax = axes[row, col]
            MODEL_COL = '#2874A6' if model_name == 'Model A' else '#C0392B'

            if axis == 'race':
                races = [r for r in RACE_LIST
                         if (model_name, r) in bal_aucs]
                if not races:
                    ax.text(0.5, 0.5, 'Insufficient n for\nbalanced evaluation',
                            ha='center', va='center',
                            transform=ax.transAxes, fontsize=10, color='grey')
                    ax.set_title(title, fontsize=10, fontweight='bold')
                    continue
                vals   = [bal_aucs[(model_name, r)][0] for r in races]
                ci_los = [bal_aucs[(model_name, r)][1] for r in races]
                ci_his = [bal_aucs[(model_name, r)][2] for r in races]
                cols   = [RACE_COLS.get(r, '#888') for r in races]
                x      = np.arange(len(races))
                ax.bar(x, vals, color=cols, alpha=0.72,
                       edgecolor='white', width=0.5)
                for xi, (v, lo, hi, c) in enumerate(
                        zip(vals, ci_los, ci_his, cols)):
                    ax.errorbar(xi, v, yerr=[[v - lo], [hi - v]],
                                fmt='none', color=c,
                                capsize=5, capthick=2, linewidth=2)
                    ax.text(xi, hi + 0.004, f'{v:.3f}',
                            ha='center', va='bottom',
                            fontsize=9, fontweight='bold', color=c)
                ax.set_xticks(x)
                ax.set_xticklabels(
                    [r.replace('Hispanic/Latino', 'Hisp/Latino')
                     for r in races],
                    fontsize=9, rotation=15, ha='right')

            else:  # quartile
                qs   = [q for q in QUARTILES if (model_name, q) in q_aucs]
                vals = [q_aucs[(model_name, q)] for q in qs]
                x    = np.arange(len(qs))
                ax.plot(x, vals, color=MODEL_COL, marker='o',
                        linewidth=2.5, markersize=10, zorder=4)
                for xi, v in enumerate(vals):
                    ax.text(xi, v + 0.006, f'{v:.3f}',
                            ha='center', va='bottom',
                            fontsize=9, fontweight='bold', color=MODEL_COL)
                ax.set_xticks(x)
                ax.set_xticklabels(
                    [f'{q}\n({"most" if q == "Q1" else "least" if q == "Q4" else ""})'
                     if q in ['Q1', 'Q4'] else q for q in qs],
                    fontsize=9)

            ax.set_ylim(0.40, 0.85)
            ax.set_ylabel('AUROC', fontsize=9)
            ax.axhline(0.5, color='grey', ls=':', lw=1, alpha=0.4)
            ax.set_facecolor('#F8F9FA')
            ax.grid(axis='y', alpha=0.3, ls='--')
            ax.set_title(title, fontsize=10, fontweight='bold')

        fig.suptitle(
            f'{care_label} — {sev} Severity — Balanced AUROC\n'
            f'Left: Race axis balanced to n={n_min} per group '
            f'({N_ITER} subsamples, mean ± 95% CI)  |  '
            f'Right: Care-quartile axis (full n, equal by design)\n'
            f'Even at equal sample sizes, the quartile axis shows a '
            f'consistent gradient while the race axis shows no directional pattern',
            fontsize=10, fontweight='bold', y=1.03,
            bbox=dict(boxstyle='round,pad=0.4',
                      facecolor=SEV_COL.get(sev, '#333'), alpha=0.15))
        plt.tight_layout()
        path = os.path.join(
            OUTPUT_FOLDER,
            f'fig_balanced_auroc_{care_label.lower()}_{sev.lower()}.png')
        plt.savefig(path, bbox_inches='tight', dpi=150)
        plt.show()
        print(f'Saved: {path}')

# ══════════════════════════════════════════════════════════════════
        # FIGURE 2 — Balanced ΔAUROC styled to match plot_delta_auroc_panel
        # Muted bars + coloured scatter dots on top + neutral band + range badge
        # ══════════════════════════════════════════════════════════════════
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        fig.patch.set_facecolor('#FAFAFA')

        for row, col, model_name, axis, title in panel_specs:
            ax = axes[row, col]
            MODEL_COL = '#2874A6' if model_name == 'Model A' else '#C0392B'

            if axis == 'race':
                groups_avail = [r for r in RACE_LIST
                                if (model_name, r) in delta_aucs]
                if not groups_avail:
                    ax.text(0.5, 0.5, 'Insufficient n for\nbalanced evaluation',
                            ha='center', va='center',
                            transform=ax.transAxes, fontsize=10, color='grey')
                    ax.set_title(title, fontsize=10, fontweight='bold')
                    continue
                vals   = np.array([delta_aucs[(model_name, r)][0]
                                   for r in groups_avail])
                ci_los = np.array([delta_aucs[(model_name, r)][1]
                                   for r in groups_avail])
                ci_his = np.array([delta_aucs[(model_name, r)][2]
                                   for r in groups_avail])
                dot_cols = [RACE_COLS.get(r, '#888') for r in groups_avail]
                x = np.arange(len(groups_avail))
                xlabels = [r.replace('Hispanic/Latino', 'Hispanic/Latino')
                           for r in groups_avail]
                ns_grp = [len(next((yt for (cl,sl,mn,gt,g,yt,yp)
                                    in calibration_store
                                    if cl==care_label and sl==sev
                                    and mn==({'Model A':'A – By Race',
                                              'Model B':'B – Q1 By Race'}[model_name])
                                    and gt=='race' and g==r), []))
                          for r in groups_avail]

            else:  # quartile
                groups_avail = [q for q in QUARTILES
                                if (model_name, q) in q_deltas]
                if not groups_avail:
                    ax.text(0.5, 0.5, 'No data', ha='center', va='center',
                            transform=ax.transAxes, fontsize=10, color='grey')
                    ax.set_title(title, fontsize=10, fontweight='bold')
                    continue
                vals   = np.array([q_deltas[(model_name, q)]
                                   for q in groups_avail])
                # Quartile panels have no CI from balanced analysis —
                # use full-n pooled bootstrap CI to match original style
                ci_los_list, ci_his_list = [], []
                q_key = ('A – All Quartiles' if model_name == 'Model A'
                         else 'B – Q1 Only')
                for q in groups_avail:
                    entry = next(((yt, yp) for (cl, sl, mn, gt, g, yt, yp)
                                  in calibration_store
                                  if cl == care_label and sl == sev
                                  and mn == q_key and gt == 'quartile'
                                  and g == q
                                  and len(np.unique(yt)) > 1), None)
                    if entry:
                        ref = cohort_ref.get(model_name, np.nan)
                        _, lo, hi = bootstrap_auroc_ci(*entry, n=500)
                        ci_los_list.append(lo - ref)
                        ci_his_list.append(hi - ref)
                    else:
                        ci_los_list.append(np.nan)
                        ci_his_list.append(np.nan)
                ci_los = np.array(ci_los_list)
                ci_his = np.array(ci_his_list)
                dot_cols = [Q_COLS.get(q, '#888') for q in groups_avail]
                x = np.arange(len(groups_avail))
                xlabels = [f'{q}\n({"most freq" if q=="Q1" else "least freq" if q=="Q4" else ""})'
                           if q in ['Q1','Q4'] else q for q in groups_avail]
                ns_grp = [len(next((yt for (cl,sl,mn,gt,g,yt,yp)
                                    in calibration_store
                                    if cl==care_label and sl==sev
                                    and mn==q_key and gt=='quartile' and g==q),
                                   []))
                          for q in groups_avail]

            # ── Neutral band ±0.02 ────────────────────────────────────────
            ax.axhspan(-0.02, 0.02, color='#CCCCCC', alpha=0.25, zorder=0)

            # ── Zero baseline ─────────────────────────────────────────────
            ax.axhline(0, color='black', lw=1.2, zorder=1,
                       label='Baseline (Pooled full-cohort AUROC)')

            # ── Muted bars ────────────────────────────────────────────────
            ax.bar(x, vals, color='#5B9BD5', alpha=0.35,
                   width=0.45, zorder=2, edgecolor='none')

            # ── Error bars ────────────────────────────────────────────────
            yerr = np.array([np.clip(vals - ci_los, 0, 2),
                             np.clip(ci_his - vals, 0, 2)])
            for xi, (v, lo, hi, dc) in enumerate(
                    zip(vals, ci_los, ci_his, dot_cols)):
                if np.isnan(v):
                    continue
                ax.errorbar(xi, v,
                            yerr=[[max(0, v - lo)], [max(0, hi - v)]],
                            fmt='none', color=dc,
                            capsize=4, capthick=1.8, linewidth=1.5,
                            zorder=3)

            # ── Coloured scatter dots on top of bars ──────────────────────
            for xi, (v, dc) in enumerate(zip(vals, dot_cols)):
                if np.isnan(v):
                    continue
                ax.scatter(xi, v, color=dc, s=60, zorder=5,
                           edgecolors='white', linewidths=0.8)

            # ── Line connecting dots (quartile panels only) ───────────────
            if axis == 'quartile':
                valid = [(xi, v) for xi, v in enumerate(vals)
                         if not np.isnan(v)]
                if len(valid) >= 2:
                    xs_v, ys_v = zip(*valid)
                    ax.plot(xs_v, ys_v, color=MODEL_COL, lw=1.8,
                            ls='-', alpha=0.7, zorder=4)

            # ── Value labels: Δ+0.xxx \n n=XXX ───────────────────────────
            for xi, (v, hi, n_g, dc) in enumerate(
                    zip(vals, ci_his, ns_grp, dot_cols)):
                if np.isnan(v):
                    continue
                label_y = hi + 0.005 if v >= 0 else ci_los[xi] - 0.005
                va_dir  = 'bottom' if v >= 0 else 'top'
                ax.annotate(
                    f'Δ{v:+.3f}\nn={n_g}',
                    (xi, label_y),
                    ha='center', va=va_dir,
                    fontsize=7.5, color=dc, fontweight='bold',
                    linespacing=1.3)

            # ── Range badge ───────────────────────────────────────────────
            valid_vals = vals[~np.isnan(vals)]
            if len(valid_vals) >= 2:
                rng = float(valid_vals.max() - valid_vals.min())
                ax.text(0.98, 0.97, f'Range = {rng:.3f}',
                        transform=ax.transAxes, ha='right', va='top',
                        fontsize=9, fontweight='bold',
                        color='#333',
                        bbox=dict(boxstyle='square,pad=0.3',
                                  facecolor='white',
                                  edgecolor='#555', linewidth=1.2))

            # ── Legend (first panel only to avoid repetition) ─────────────
            if row == 0 and col == 0:
                ax.legend(fontsize=7.5, loc='upper left', framealpha=0.9)

            # ── Axis formatting ───────────────────────────────────────────
            y_abs = max(0.15, np.nanmax(np.abs(
                np.concatenate([ci_his, ci_los])
            )) + 0.04)
            ax.set_ylim(-y_abs, y_abs)
            ax.set_ylabel('ΔAUROC (group AUROC − reference)', fontsize=9)
            ax.set_xticks(x)
            ax.set_xticklabels(xlabels, fontsize=8.5,
                               rotation=15 if axis == 'race' else 0,
                               ha='right' if axis == 'race' else 'center')
            ax.set_xlabel('Race Group' if axis == 'race'
                          else 'Care Quartile', fontsize=9)
            ax.set_facecolor('#F8F9FA')
            ax.grid(axis='y', alpha=0.25, ls='--', zorder=0)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)

            # ── Expected note at bottom (matches original style) ──────────
            note = ('Performance similar across race groups — '
                    'balanced evaluation' if axis == 'race'
                    else 'Performance gradient across care quartiles')
            ax.text(0.02, 0.02, note,
                    transform=ax.transAxes, ha='left', va='bottom',
                    fontsize=7, style='italic', color='#555',
                    bbox=dict(boxstyle='round,pad=0.3',
                              facecolor='white', edgecolor='#CCC',
                              alpha=0.85))

            ax.set_title(title, fontsize=10, fontweight='bold')

        # ── Row and col headers matching original layout ──────────────────
        axes[0, 0].set_title(
            axes[0, 0].get_title() + '\n(ΔAUROC vs full cohort)',
            fontsize=10, fontweight='bold')
        axes[0, 1].set_title(
            axes[0, 1].get_title() + '\n(ΔAUROC vs full cohort)',
            fontsize=10, fontweight='bold')
        axes[1, 0].set_title(
            axes[1, 0].get_title() + '\n(ΔAUROC vs full cohort)',
            fontsize=10, fontweight='bold')
        axes[1, 1].set_title(
            axes[1, 1].get_title() + '\n(ΔAUROC vs full cohort)',
            fontsize=10, fontweight='bold')

        fig.suptitle(
            f'ΔAUROC — Deviation from Full-Cohort Baseline\n'
            f'{care_label}  |  {sev} Severity (SOFA {{"Low":"0–6","Medium":"7–11","High":"≥12"}}[sev])\n'
            f'All panels share the same zero line (pooled full-cohort AUROC) — '
            f'enabling direct comparison between race axis (flat) and quartile axis (spread)\n'
            f'Race panels flat near 0 = apparent fairness  |  '
            f'Quartile panels spread = real disparity hidden by race-based audit  |  '
            f'Q1 > 0 = best-care patients above average; Q4 < 0 = worst-care below average',
            fontsize=9, fontweight='bold', y=1.04,
            color=SEV_COL.get(sev, '#333'))

        plt.tight_layout()
        path = os.path.join(
            OUTPUT_FOLDER,
            f'fig_balanced_delta_{care_label.lower()}_{sev.lower()}.png')
        plt.savefig(path, bbox_inches='tight', dpi=150)
        plt.show()
        print(f'Saved: {path}')

print('\nAll balanced panels saved.')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Supplementary Table: XGBoost Hyperparameter Configuration
# ═══════════════════════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

rows = [
    # (Parameter, Value, Rationale)
    ('Model architecture',    'XGBoost (gradient-boosted decision trees)', ''),
    ('Max estimators',        '500 (upper bound)',                         'Early stopping finds true optimum'),
    ('Early stopping rounds', '20',                                        'Patience before stopping on val log-loss'),
    ('Internal val split',    '15% stratified hold-out',                   'Used in phase 1 only; phase 2 refits on full training data'),
    ('Max tree depth',        '3',                                         'Reduced from 4; depth 4 caused train AUROC ~1.0 / test ~0.65'),
    ('Min child weight',      '10',                                        'Requires ≥10 patients per leaf; prevents splits on tiny subgroups'),
    ('Gamma',                 '1.0',                                       'Minimum loss reduction required to make a split'),
    ('L1 regularisation',     'alpha = 0.5',                               'Shrinks small feature weights toward zero'),
    ('L2 regularisation',     'lambda = 2.0',                              'Smooths feature weights'),
    ('Row subsampling',       '0.8',                                       'Fraction of training rows sampled per tree'),
    ('Column subsampling',    '0.8',                                       'Fraction of features sampled per tree'),
    ('Learning rate',         '0.05',                                      ''),
    ('Class imbalance',       'scale_pos_weight = negatives / positives',  'Corrects for mortality class imbalance'),
    ('Evaluation metric',     'Log-loss',                                  'Used for early stopping'),
    ('Random seed',           '42',                                        'Fixed for reproducibility'),
]

col_labels = ['Parameter', 'Value', 'Rationale']
col_widths  = [0.22, 0.28, 0.50]

HDR_COL  = '#1F3864'
HDR_TXT  = '#FFFFFF'
ALT_COL  = '#EBF5FB'
BASE_COL = '#FFFFFF'
LINE_COL = '#BDC3C7'

row_h   = 0.38
hdr_h   = 0.45
fig_w   = 16
fig_h   = hdr_h + len(rows) * row_h + 0.6

fig, ax = plt.subplots(figsize=(fig_w, fig_h))
ax.set_xlim(0, 1)
ax.set_ylim(0, fig_h)
ax.axis('off')

# Title
ax.text(0.5, fig_h - 0.15,
        'Supplementary Table: XGBoost Hyperparameter Configuration',
        ha='center', va='top', fontsize=13, fontweight='bold', color='#1F3864')

# Header row
y_top = fig_h - 0.45
x = 0
for ci, (lbl, cw) in enumerate(zip(col_labels, col_widths)):
    rect = mpatches.FancyBboxPatch(
        (x, y_top - hdr_h), cw, hdr_h,
        boxstyle='square,pad=0', linewidth=0.8,
        edgecolor=LINE_COL, facecolor=HDR_COL)
    ax.add_patch(rect)
    ax.text(x + cw/2, y_top - hdr_h/2, lbl,
            ha='center', va='center', fontsize=10,
            fontweight='bold', color=HDR_TXT)
    x += cw

# Data rows
for ri, (param, value, rationale) in enumerate(rows):
    y = y_top - hdr_h - (ri + 1) * row_h
    bg = ALT_COL if ri % 2 == 0 else BASE_COL
    x = 0
    for ci, (text, cw) in enumerate(zip([param, value, rationale], col_widths)):
        rect = mpatches.FancyBboxPatch(
            (x, y), cw, row_h,
            boxstyle='square,pad=0', linewidth=0.5,
            edgecolor=LINE_COL, facecolor=bg)
        ax.add_patch(rect)
        ax.text(x + 0.012, y + row_h/2, text,
                ha='left', va='center', fontsize=9, color='#1A1A1A',
                wrap=True)
        x += cw

plt.tight_layout(pad=0)
path = os.path.join(OUTPUT_FOLDER, 'supplementary_table_hyperparameters.png')
plt.savefig(path, bbox_inches='tight', dpi=150)
plt.show()
print(f'Saved: {path}')

## Supplementary Analysis — Continuous SOFA Gradient

Tests whether the three-stratum severity result reflects a genuine continuous gradient rather than a discrete threshold effect.

**Design (two complementary analyses):**
- **Coarser bins (width = 4, min 200 patients):** Both Model A and Model B are run, showing the A-vs-B asymmetry across SOFA (cell 75).
- **Fine-grained bins (width = 2, min 80 patients):** Model B only, showing Q1 and Q4 AUROC lines for the MPOR gradient figure (cell 76).

**Hypothesis:** As SOFA increases, the Q1−Q4 AUROC gap narrows continuously — showing that Low severity is not an exception but the region where care variation has the highest marginal influence on outcomes. This also demonstrates that the framework's sensitivity is appropriately calibrated: where illness severity dominates, the care signal attenuates; where care variation matters most, the signal is strongest.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# MPOR GRADIENT — Figure 5 (Coarser bins: width=4, both Model A and Model B)
# Saves: fig_sofa_gradient_{care}_A_vs_B.png, fig_sofa_jt_pvalue_{care}.png,
#        fig_sofa_gradient_gap_comparison.png
# All 4 quartiles shown across SOFA spectrum. Asymmetry between Model A (flat)
# and Model B (gradient) provides continuous-severity validation of 3-stratum result.
# Model B: trained on Q1 only — expected to show degradation gradient
# The asymmetry between A and B is the key evidence
# ══════════════════════════════════════════════════════════════════════════════
import numpy as np, pandas as pd, matplotlib.pyplot as plt, os, warnings
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
warnings.filterwarnings('ignore')


def sofa_gradient_analysis(
    care_data, care_label,
    bin_width=4, min_patients=200, min_events=20,
    random_state=42
):
    """
    Run Model A (all-quartile training) and Model B (Q1-only training)
    across fine-grained SOFA bins. Returns per-bin AUROC for Q1-Q4 for both models.
    """
    avail = [c for c in FEATURE_COLS if c in care_data.columns]
    N_FOLDS = 5

    sofa_max = int(care_data[SOFA_COL].max()) + 1
    bin_edges = list(range(0, sofa_max + bin_width, bin_width))

    results = []

    for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
        bin_data = care_data[
            (care_data[SOFA_COL] >= lo) & (care_data[SOFA_COL] < hi)
        ].copy()

        n_total  = len(bin_data)
        n_events = int(bin_data[MORTALITY].sum())
        if n_total < min_patients or n_events < min_events:
            continue

        sofa_mid  = lo + bin_width / 2
        bin_label = f'{lo}-{hi-1}'

        # Re-assign quartiles within SOFA bin
        ranks = bin_data[FREQ_COL].rank(method='first')
        try:
            bin_data = bin_data.copy()
            bin_data['quartile'] = pd.qcut(ranks, q=4, labels=QUARTILES)
        except Exception:
            continue
        bin_data = bin_data.dropna(subset=['quartile']).copy()

        # Check all quartiles have both outcomes
        skip = False
        for q in QUARTILES:
            qsub = bin_data[bin_data['quartile'] == q]
            if len(qsub) < 15 or qsub[MORTALITY].nunique() < 2:
                skip = True
                break
        if skip:
            continue

        strat_key = bin_data['quartile'].astype(str) + '_' + bin_data[MORTALITY].astype(str)
        if strat_key.value_counts().min() < N_FOLDS:
            continue

        bin_arr = bin_data.reset_index(drop=True)
        skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=random_state)

        fold_preds_A = {q: [] for q in QUARTILES}
        fold_preds_B = {q: [] for q in QUARTILES}

        for tr_idx, te_idx in skf.split(bin_arr, strat_key.values):
            train_fold = bin_arr.iloc[tr_idx]
            test_fold  = bin_arr.iloc[te_idx]
            train_med  = train_fold[avail].median()

            # ── Model A: train on all quartiles ──────────────────────────────
            Xtr_A = train_fold[avail].fillna(train_med)
            ytr_A = train_fold[MORTALITY]
            if ytr_A.nunique() < 2:
                continue
            try:
                model_A = train_xgb(Xtr_A, ytr_A, Xtr_A, ytr_A)
                raw_cal_A = model_A.predict_proba(
                    train_fold[avail].fillna(train_med))[:, 1]
                platt_A = fit_platt_scaler(ytr_A.values, raw_cal_A)
                for q in QUARTILES:
                    te_q = test_fold[test_fold['quartile'] == q]
                    if len(te_q) < 5:
                        continue
                    raw = model_A.predict_proba(te_q[avail].fillna(train_med))[:, 1]
                    yp  = apply_calibrator(platt_A, raw)
                    fold_preds_A[q].append((te_q[MORTALITY].values, yp))
            except Exception:
                pass

            # ── Model B: train on Q1 only ─────────────────────────────────
            q1_tr = train_fold[train_fold['quartile'] == 'Q1']
            if len(q1_tr) < 15 or q1_tr[MORTALITY].nunique() < 2:
                continue
            try:
                model_B = train_xgb(
                    q1_tr[avail].fillna(train_med), q1_tr[MORTALITY],
                    q1_tr[avail].fillna(train_med), q1_tr[MORTALITY]
                )
                raw_q1 = model_B.predict_proba(q1_tr[avail].fillna(train_med))[:, 1]
                platt_B = fit_platt_scaler(q1_tr[MORTALITY].values, raw_q1)
                for q in QUARTILES:
                    te_q = test_fold[test_fold['quartile'] == q]
                    if len(te_q) < 5:
                        continue
                    raw = model_B.predict_proba(te_q[avail].fillna(train_med))[:, 1]
                    yp  = apply_calibrator(platt_B, raw)
                    fold_preds_B[q].append((te_q[MORTALITY].values, yp))
            except Exception:
                continue

        # ── Aggregate both models ─────────────────────────────────────────
        row = {
            'sofa_mid': sofa_mid, 'sofa_bin': bin_label,
            'n': n_total, 'mortality_pct': round(n_events / n_total * 100, 1),
        }

        for model_name, fold_preds in [('A', fold_preds_A), ('B', fold_preds_B)]:
            for q in QUARTILES:
                fas = [roc_auc_score(yt, yp)
                       for yt, yp in fold_preds[q]
                       if len(np.unique(yt)) >= 2 and len(yt) >= 5]
                if fas:
                    m  = np.mean(fas)
                    se = np.std(fas, ddof=1) / np.sqrt(len(fas)) if len(fas) > 1 else 0
                    row[f'M{model_name}_{q}']     = round(m, 3)
                    row[f'M{model_name}_{q}_lo']  = round(max(0.0, m - 1.96*se), 3)
                    row[f'M{model_name}_{q}_hi']  = round(min(1.0, m + 1.96*se), 3)
                else:
                    row[f'M{model_name}_{q}']    = np.nan
                    row[f'M{model_name}_{q}_lo'] = np.nan
                    row[f'M{model_name}_{q}_hi'] = np.nan

        # ── Permutation max JT for Model B quartiles ────────────────────
        # Tests whether Q1-Q4 ordering is significantly monotonic within this bin
        jt_arrays_B = []
        for q in QUARTILES:
            fas = [roc_auc_score(yt, yp)
                   for yt, yp in fold_preds_B[q]
                   if len(np.unique(yt)) >= 2 and len(yt) >= 5]
            jt_arrays_B.append(np.array(fas) if fas else np.array([]))

        valid_jt = [g for g in jt_arrays_B if len(g) >= 2]
        if len(valid_jt) >= 3:
            try:
                jt_stat_B, jt_p_B = permutation_max_jt(jt_arrays_B, n_perm=1000, seed=42)
            except:
                jt_p_B = np.nan
        else:
            jt_p_B = np.nan

        # Also directional JT for Model B
        if len(valid_jt) >= 3:
            try:
                _, dir_jt_p_B = jonckheere_trend_test(jt_arrays_B)
            except:
                dir_jt_p_B = np.nan
        else:
            dir_jt_p_B = np.nan

        row['perm_jt_p_B']  = round(jt_p_B, 3) if not np.isnan(jt_p_B) else np.nan
        row['dir_jt_p_B']   = round(dir_jt_p_B, 3) if not np.isnan(dir_jt_p_B) else np.nan

        results.append(row)
        gap_A = (row.get('MA_Q1', np.nan) or 0) - (row.get('MA_Q4', np.nan) or 0)
        gap_B = (row.get('MB_Q1', np.nan) or 0) - (row.get('MB_Q4', np.nan) or 0)
        sig   = '*' if (not np.isnan(jt_p_B) and jt_p_B < 0.05) else ' '
        print(f'  SOFA {bin_label:6s} | n={n_total:4d} | mort={n_events/n_total*100:.0f}% | ' +
              f'A gap={gap_A:+.3f} | B gap={gap_B:+.3f} | perm JT p={round(jt_p_B,3) if not np.isnan(jt_p_B) else "nan"}{sig}')

    return pd.DataFrame(results)


# ── Run ───────────────────────────────────────────────────────────────────────
print('SOFA Gradient — Mouthcare (bin width=4, Models A and B)')
print('=' * 60)
sofa_mc   = sofa_gradient_analysis(mc_data,   'Mouthcare')

print('\nSOFA Gradient — Turning (bin width=4, Models A and B)')
print('=' * 60)
sofa_turn = sofa_gradient_analysis(turn_data, 'Turning')


# ── Figure: Q1-Q4 lines for Model B (all quartiles shown) ────────────────────
Q_COLORS = {
    'Q1': '#1a6e9e',  # dark blue
    'Q2': '#5ba3cc',  # medium blue
    'Q3': '#e08070',  # medium red
    'Q4': '#c0392b',  # dark red
}

BOUND_LOW  = 6.5
BOUND_HIGH = 11.5

for care_label, df_p in [('Mouthcare', sofa_mc), ('Turning', sofa_turn)]:
    if df_p.empty:
        print(f'{care_label}: no data')
        continue

    fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=False)
    fig.suptitle(
        f'{care_label} — Model Performance Across SOFA Spectrum\n' +
        'Model A (all-quartile training) vs Model B (Q1-only training)',
        fontsize=12, fontweight='bold'
    )

    x = df_p['sofa_mid'].values

    for ax, (model_name, model_label) in zip(axes, [
        ('A', 'Model A (trained on all quartiles)'),
        ('B', 'Model B (trained on Q1 only)')
    ]):
        for q in QUARTILES:
            col     = f'M{model_name}_{q}'
            col_lo  = f'M{model_name}_{q}_lo'
            col_hi  = f'M{model_name}_{q}_hi'

            if col not in df_p.columns:
                continue

            y    = df_p[col].values
            y_lo = df_p[col_lo].values
            y_hi = df_p[col_hi].values

            valid = ~np.isnan(y)
            if valid.sum() == 0:
                continue

            ls = '-' if q in ['Q1', 'Q2'] else '--'
            ax.plot(x[valid], y[valid],
                    color=Q_COLORS[q], lw=2, marker='o',
                    markersize=6, linestyle=ls,
                    label=f'{q}', zorder=3)
            ax.fill_between(x[valid], y_lo[valid], y_hi[valid],
                            alpha=0.10, color=Q_COLORS[q])

        # Stratum boundaries
        ax.axvline(BOUND_LOW,  color='gray', ls='--', lw=1.5, alpha=0.6)
        ax.axvline(BOUND_HIGH, color='gray', ls='--', lw=1.5, alpha=0.6)

        ax.set_xlabel('SOFA Score (bin midpoint)', fontsize=11)
        ax.set_ylabel('AUROC', fontsize=11)
        ax.set_title(model_label, fontsize=11, fontweight='bold')
        ax.set_ylim(0.45, 0.85)
        ax.legend(fontsize=9, loc='upper right', title='Quartile')
        ax.grid(True, alpha=0.3)

        # Stratum labels
        for xpos, lbl in [(3, 'Low'), (9, 'Medium'), (14, 'High')]:
            ax.text(xpos, 0.47, lbl, ha='center', fontsize=9,
                    color='dimgray', style='italic')

    # ── Add JT p-value panel below ──────────────────────────────────────
    if 'perm_jt_p_B' in df_p.columns and df_p['perm_jt_p_B'].notna().any():
        fig2b, ax_jt = plt.subplots(1, 1, figsize=(7, 3))
        fig2b.suptitle(
            f'{care_label} — Permutation Max JT p-value (Model B) Across SOFA Spectrum',
            fontsize=11, fontweight='bold'
        )
        x_jt  = df_p['sofa_mid'].values
        jt_p  = df_p['perm_jt_p_B'].values

        # Bar chart: lower p = stronger monotonic ordering
        bar_colors_jt = ['#1a6e9e' if (not np.isnan(p) and p < 0.05) else '#aaaaaa'
                         for p in jt_p]
        ax_jt.bar(x_jt, jt_p, width=3.2, color=bar_colors_jt,
                  alpha=0.8, edgecolor='white', zorder=2)
        ax_jt.axhline(0.05, color='red', ls='--', lw=1.5,
                      label='p = 0.05', zorder=3)
        ax_jt.axvline(BOUND_LOW,  color='gray', ls='--', lw=1.5, alpha=0.6)
        ax_jt.axvline(BOUND_HIGH, color='gray', ls='--', lw=1.5, alpha=0.6)
        ax_jt.set_xlabel('SOFA Score (bin midpoint)', fontsize=11)
        ax_jt.set_ylabel('Permutation JT p-value', fontsize=11)
        ax_jt.set_ylim(0, 1.05)
        ax_jt.invert_yaxis()  # lower p-value at top = stronger signal
        ax_jt.legend(fontsize=9)
        ax_jt.grid(True, alpha=0.3, axis='y')
        for xpos, lbl in [(3, 'Low'), (9, 'Medium'), (14, 'High')]:
            ax_jt.text(xpos, 1.0, lbl, ha='center', fontsize=9,
                       color='dimgray', style='italic')
        ax_jt.set_title(
            'Significant (blue) = Q1-Q4 ordering is monotonically significant within SOFA bin',
            fontsize=9, style='italic'
        )
        plt.tight_layout()
        fname_jt = f'fig_sofa_jt_pvalue_{care_label.lower()}.png'
        path_jt  = os.path.join(OUTPUT_FOLDER, fname_jt)
        plt.savefig(path_jt, dpi=150, bbox_inches='tight')
        plt.show()
        plt.close()
        print(f'Saved: {path_jt}')

    plt.tight_layout()
    fname = f'fig_sofa_gradient_{care_label.lower()}_A_vs_B.png'
    path  = os.path.join(OUTPUT_FOLDER, fname)
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f'Saved: {path}')


# ── Gap comparison figure: Model A vs Model B ─────────────────────────────────
fig3, axes3 = plt.subplots(2, 2, figsize=(14, 8))
fig3.suptitle('Q1 − Q4 AUROC Gap: Model A vs Model B Across SOFA Spectrum',
              fontsize=13, fontweight='bold')

for col_i, (care_label, df_p) in enumerate(
    [('Mouthcare', sofa_mc), ('Turning', sofa_turn)]
):
    if df_p.empty:
        continue
    x = df_p['sofa_mid'].values

    for row_i, (model_name, model_label) in enumerate(
        [('A', 'Model A'), ('B', 'Model B')]
    ):
        ax = axes3[row_i, col_i]
        q1_col = f'M{model_name}_Q1'
        q4_col = f'M{model_name}_Q4'
        if q1_col not in df_p.columns:
            continue

        gap = df_p[q1_col].values - df_p[q4_col].values
        bar_colors = ['#1a6e9e' if g > 0 else '#c0392b' for g in gap]
        ax.bar(x, gap, width=3.2, color=bar_colors, alpha=0.75,
               edgecolor='white', zorder=2)
        ax.axhline(0, color='black', lw=1)
        ax.axvline(BOUND_LOW,  color='gray', ls='--', lw=1.5, alpha=0.6)
        ax.axvline(BOUND_HIGH, color='gray', ls='--', lw=1.5, alpha=0.6)
        ax.set_title(f'{care_label} — {model_label}', fontsize=11, fontweight='bold')
        ax.set_xlabel('SOFA Score (bin midpoint)', fontsize=10)
        ax.set_ylabel('Q1 − Q4 AUROC Gap', fontsize=10)
        ax.grid(True, alpha=0.3, axis='y')
        for xpos, lbl in [(3, 'Low'), (9, 'Med'), (14, 'High')]:
            ax.text(xpos, -0.01,
                    lbl, ha='center', fontsize=8, color='dimgray', style='italic',
                    clip_on=False)

plt.tight_layout()
path3 = os.path.join(OUTPUT_FOLDER, 'fig_sofa_gradient_gap_comparison.png')
plt.tight_layout()
plt.savefig(path3, dpi=150, bbox_inches='tight')
plt.show()
plt.close()
print(f'Saved: {path3}')

# ── Summary ───────────────────────────────────────────────────────────────────
for label, df_p in [('Mouthcare', sofa_mc), ('Turning', sofa_turn)]:
    print(f'\n{label} Summary:')
    if df_p.empty:
        print('  No results.')
        continue
    cols = ['sofa_bin', 'n', 'mortality_pct']
    for q in QUARTILES:
        for m in ['A', 'B']:
            c = f'M{m}_{q}'
            if c in df_p.columns:
                cols.append(c)
    for jt_col in ['perm_jt_p_B', 'dir_jt_p_B']:
        if jt_col in df_p.columns:
            cols.append(jt_col)
    print(df_p[cols].to_string(index=False))


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# MPOR GRADIENT — Simplified Q1 vs Q4 View (Fine bins: width=2, Model B only)
# Saves: fig_sofa_gradient_auroc.png (Q1/Q4 AUROC lines), fig_sofa_gradient_gap.png
# Simpler view showing only Q1 vs Q4 gap across SOFA. Redefines sofa_gradient_analysis
# with finer bins (width=2) and Model B only. Cell 75 (all quartiles, A+B) is primary.
# ══════════════════════════════════════════════════════════════════════════════
import numpy as np, pandas as pd, matplotlib.pyplot as plt, os, warnings
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
warnings.filterwarnings('ignore')


# NOTE: This cell redefines sofa_gradient_analysis with finer bins (width=2, Model B only).
# The version in cell 75 used width=4 and ran both Model A and Model B.
# This version produces the MPOR gradient figure (Q1 vs Q4 lines across SOFA).
def sofa_gradient_analysis(
    care_data, care_label,
    bin_width=2, min_patients=80, min_events=10,
    random_state=42
):
    """
    Run Model B Q1-only evaluation across fine-grained SOFA bins (MPOR gradient figure).
    Within each bin, quartiles are re-assigned by care frequency.
    Returns DataFrame with per-bin Q1/Q4 AUROC, gap, n, mortality rate.

    bin_width=2: SOFA bins of 2 score points — finer resolution than cell 75 (width=4)
    min_patients=80: minimum cohort size per bin for statistical reliability
    min_events=10: minimum mortality events per bin
    This version runs Model B only (Q1-trained) and is used for the MPOR figure.
    """
    avail = [c for c in FEATURE_COLS if c in care_data.columns]
    N_FOLDS = 5

    sofa_max = int(care_data[SOFA_COL].max()) + 1
    bin_edges = list(range(0, sofa_max + bin_width, bin_width))

    results = []
    for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
        bin_data = care_data[
            (care_data[SOFA_COL] >= lo) & (care_data[SOFA_COL] < hi)
        ].copy()

        n_total  = len(bin_data)
        n_events = int(bin_data[MORTALITY].sum())
        if n_total < min_patients or n_events < min_events:
            continue

        sofa_mid = lo + bin_width / 2
        bin_label = f'{lo}-{hi-1}'

        # Re-assign quartiles within this SOFA bin by care frequency
        ranks = bin_data[FREQ_COL].rank(method='first')
        try:
            bin_data['quartile'] = pd.qcut(ranks, q=4, labels=QUARTILES)
        except Exception:
            continue
        bin_data = bin_data.dropna(subset=['quartile']).copy()
        if bin_data['quartile'].value_counts().min() < 10:
            continue

        strat_key = bin_data['quartile'].astype(str) + '_' + bin_data[MORTALITY].astype(str)
        if strat_key.value_counts().min() < N_FOLDS:
            continue

        bin_arr = bin_data.reset_index(drop=True)
        skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=random_state)
        fold_preds_B = {q: [] for q in QUARTILES}

        for tr_idx, te_idx in skf.split(bin_arr, strat_key.values):
            train_fold = bin_arr.iloc[tr_idx]
            test_fold  = bin_arr.iloc[te_idx]
            train_med  = train_fold[avail].median()
            q1_tr = train_fold[train_fold['quartile'] == 'Q1']
            if len(q1_tr) < 15 or q1_tr[MORTALITY].nunique() < 2:
                continue
            try:
                model_B = train_xgb(
                    q1_tr[avail].fillna(train_med), q1_tr[MORTALITY],
                    q1_tr[avail].fillna(train_med), q1_tr[MORTALITY]
                )
                raw_q1 = model_B.predict_proba(q1_tr[avail].fillna(train_med))[:, 1]
                platt_B = fit_platt_scaler(q1_tr[MORTALITY].values, raw_q1)
                for q in QUARTILES:
                    te_q = test_fold[test_fold['quartile'] == q]
                    if len(te_q) < 5 or te_q[MORTALITY].nunique() < 2:
                        continue
                    raw = model_B.predict_proba(te_q[avail].fillna(train_med))[:, 1]
                    yp  = apply_calibrator(platt_B, raw)
                    fold_preds_B[q].append((te_q[MORTALITY].values, yp))
            except Exception:
                continue

        # Aggregate
        aurocs, ci_lo, ci_hi = {}, {}, {}
        for q in QUARTILES:
            fas = [roc_auc_score(yt, yp)
                   for yt, yp in fold_preds_B[q]
                   if len(np.unique(yt)) >= 2 and len(yt) >= 5]
            if fas:
                m  = np.mean(fas)
                se = np.std(fas, ddof=1) / np.sqrt(len(fas)) if len(fas) > 1 else 0
                aurocs[q] = m
                ci_lo[q]  = max(0.0, m - 1.96 * se)
                ci_hi[q]  = min(1.0, m + 1.96 * se)
            else:
                aurocs[q] = ci_lo[q] = ci_hi[q] = np.nan

        q1_a = aurocs.get('Q1', np.nan)
        q4_a = aurocs.get('Q4', np.nan)
        gap  = q1_a - q4_a if not (np.isnan(q1_a) or np.isnan(q4_a)) else np.nan

        results.append({
            'sofa_mid': sofa_mid, 'sofa_bin': bin_label,
            'n': n_total, 'mortality_pct': round(n_events / n_total * 100, 1),
            'Q1_AUROC': round(q1_a, 3), 'Q1_lo': round(ci_lo['Q1'], 3),
            'Q1_hi':    round(ci_hi['Q1'], 3),
            'Q4_AUROC': round(q4_a, 3), 'Q4_lo': round(ci_lo['Q4'], 3),
            'Q4_hi':    round(ci_hi['Q4'], 3),
            'gap': round(gap, 3) if not np.isnan(gap) else np.nan,
        })
        print(f'  SOFA {bin_label:6s} | n={n_total:4d} | mort={n_events/n_total*100:.0f}% | ' 
              f'Q1={q1_a:.3f} Q4={q4_a:.3f} gap={gap:+.3f}')

    return pd.DataFrame(results)


# ── Run ───────────────────────────────────────────────────────────────────────
print('SOFA Gradient — Mouthcare')
print('=' * 50)
sofa_mc   = sofa_gradient_analysis(mc_data,   'Mouthcare')

print('\nSOFA Gradient — Turning')
print('=' * 50)
sofa_turn = sofa_gradient_analysis(turn_data, 'Turning')


# ── Figure 1: Q1 and Q4 AUROC lines across SOFA ───────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle(
    'Model B (Q1-trained) Performance Across Continuous SOFA Spectrum',
    fontsize=13, fontweight='bold'
)

BOUND_LOW  = 6.5
BOUND_HIGH = 11.5

for ax, (df_p, title) in zip(axes, [(sofa_mc, 'Mouthcare'), (sofa_turn, 'Turning')]):
    if df_p.empty:
        ax.text(0.5, 0.5, 'Insufficient data', ha='center', transform=ax.transAxes)
        continue

    x = df_p['sofa_mid'].values

    # Q1 line
    ax.plot(x, df_p['Q1_AUROC'], color='#1a6e9e', lw=2.5, marker='o',
            markersize=6, label='Q1 (highest care)', zorder=3)
    ax.fill_between(x, df_p['Q1_lo'], df_p['Q1_hi'], alpha=0.15, color='#1a6e9e')

    # Q4 line
    ax.plot(x, df_p['Q4_AUROC'], color='#c0392b', lw=2.5, marker='s',
            markersize=6, label='Q4 (lowest care)', zorder=3)
    ax.fill_between(x, df_p['Q4_lo'], df_p['Q4_hi'], alpha=0.15, color='#c0392b')

    # Gap shading (where Q1 > Q4)
    q1_arr = df_p['Q1_AUROC'].values
    q4_arr = df_p['Q4_AUROC'].values
    ax.fill_between(x, q4_arr, q1_arr,
                    where=(q1_arr >= q4_arr), alpha=0.1,
                    color='green', label='Q1−Q4 gap')

    # Stratum boundaries
    for bnd in [BOUND_LOW, BOUND_HIGH]:
        ax.axvline(bnd, color='gray', ls='--', lw=1.5, alpha=0.6)

    # Stratum labels at bottom of plot
    ymin = 0.45
    for xpos, lbl in [(3, 'Low'), (9, 'Medium'), (14, 'High')]:
        ax.text(xpos, ymin + 0.01, lbl, ha='center', fontsize=9,
                color='dimgray', style='italic')

    ax.set_xlabel('SOFA Score (bin midpoint)', fontsize=11)
    ax.set_ylabel('AUROC', fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylim(0.45, 0.85)
    ax.legend(fontsize=9, loc='upper right')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
path1 = os.path.join(OUTPUT_FOLDER, 'fig_sofa_gradient_auroc.png')
plt.savefig(path1, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {path1}')


# ── Figure 2: Q1-Q4 gap bar chart across SOFA ────────────────────────────────
fig2, axes2 = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
fig2.suptitle('Q1 − Q4 AUROC Gap Across SOFA Spectrum (Model B)',
              fontsize=13, fontweight='bold')

for ax, (df_p, title) in zip(axes2, [(sofa_mc, 'Mouthcare'), (sofa_turn, 'Turning')]):
    if df_p.empty:
        continue
    x   = df_p['sofa_mid'].values
    gap = df_p['gap'].fillna(0).values
    bar_colors = ['#1a6e9e' if g > 0 else '#c0392b' for g in gap]
    ax.bar(x, gap, width=1.6, color=bar_colors, alpha=0.75, edgecolor='white', zorder=2)
    ax.axhline(0, color='black', lw=1)
    for bnd in [BOUND_LOW, BOUND_HIGH]:
        ax.axvline(bnd, color='gray', ls='--', lw=1.5, alpha=0.6)
    ax.set_xlabel('SOFA Score (bin midpoint)', fontsize=11)
    ax.set_ylabel('Q1 − Q4 AUROC Gap', fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_xlim(left=0)

plt.tight_layout()
# Add stratum labels after layout is finalized
for ax2 in axes2:
    try:
        ylo, yhi = ax2.get_ylim()
        label_y = ylo - (yhi - ylo) * 0.12
        for xpos, lbl in [(3, 'Low'), (9, 'Medium'), (14, 'High')]:
            ax2.text(xpos, label_y, lbl, ha='center', fontsize=9,
                     color='dimgray', style='italic', clip_on=False)
    except:
        pass
path2 = os.path.join(OUTPUT_FOLDER, 'fig_sofa_gradient_gap.png')
plt.savefig(path2, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {path2}')

# ── Summary tables ────────────────────────────────────────────────────────────
for label, df_p in [('Mouthcare', sofa_mc), ('Turning', sofa_turn)]:
    print(f'\n{label} SOFA Gradient Summary:')
    if df_p.empty:
        print('  No results.')
    else:
        print(df_p[['sofa_bin', 'n', 'mortality_pct',
                    'Q1_AUROC', 'Q4_AUROC', 'gap']].to_string(index=False))
